# AST Audio Spectrogram Transformer — DIMER Acoustic Ecology E2E Tutorial (Standalone)

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/ast-audio-classification-pipeline) [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/ast-audio-classification-pipeline/blob/main/tutorials/ast_audio_classification_colab.ipynb) [![Hugging Face](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-MIT%2Fast--finetuned--audioset-ffcc4d?style=flat)](https://huggingface.co/MIT/ast-finetuned-audioset-10-10-0.4593) [![Upstream](https://img.shields.io/badge/Upstream-YuanGongND%2Fast-181717?style=flat&logo=github&logoColor=white)](https://github.com/YuanGongND/ast) [![arXiv](https://img.shields.io/badge/arXiv-2104.01778-b31b1b.svg)](https://arxiv.org/abs/2104.01778)

**Profile:** `E2E`  
**Mode:** `GUIDED`  
**Notebook specification:** DIMER Notebook Specification 2.2 — **standalone** (§4)  
**Capability:** pretrained AudioSet classification & supervised acoustic ecology adaptation with `MIT/ast-finetuned-audioset-10-10-0.4593`

**This notebook is standalone.** Section 2 carries the repository's package (4 modules under `src/ast_audio_classification_pipeline/`, at revision `d20a47c1d65e`), the stage runner, the hash-locked requirements (47 packages), the pinned snapshot manifests and the licence, and verifies every carried file against its SHA-256 before use, so the notebook keeps working after export even if the repository changes or disappears. Nothing is installed into the notebook kernel: a pinned `uv` (checked by size and SHA-256) builds an isolated Python environment from the lock with `--require-hashes`, and every stage runs there in its own process, so the hosted runtime's own packages are never replaced and no restart is needed. Its only external dependencies are PyPI, the managed CPython build that `uv` downloads, and the Hugging Face Hub at the immutable revision `f826b80d28226b62986cc218e5cec390b1096902` (~346 MB, digest-verified before loading). It was generated by `tools/build_notebook.py` (build_notebook.py/3.0); edit the repository and regenerate rather than editing cells.

**Run all:** Selecting **Run all** in a fresh supported runtime checks the runtime, writes and hash-verifies the carried files, builds an isolated hash-locked Python environment (nothing is installed into the notebook kernel, so no restart is needed), stages and digest-verifies the pinned AST snapshot, demonstrates the unchanged 527-label AudioSet head, builds and validates the 24-clip acoustic-ecology dataset, refuses duplicate clips, creates a seeded stratified split with a held-out evaluation set, re-heads the classifier onto three classes, freezes the transformer backbone, measures the untrained head and a training-majority baseline, runs the bounded classifier-head fine-tune and exports the adapter, evaluates the held-out clips and an unseen clip in a fresh process, reloads the adapter in another fresh process and checks numeric parity, and writes machine-readable outputs. The default path needs no repository clone, DIMER worker or service, credential, upload dialog, or configuration edit (NOTEBOOK_SPEC 2.2 §5, RUN1, RUN7, RUN10, FT2).

**Bring Your Own Data:** Two optional branches are disabled by default. `USE_BYOD = True` (Section 5) scores one PCM WAV with the unchanged AudioSet head. `USE_BYOD_DATASET = True` (Section 6) replaces the generated dataset with one ZIP of three folders, `geophony/`, `biophony/` and `anthrophony/` (at the top of the archive or inside one enclosing folder), with at least two 16 kHz mono or multichannel PCM WAV files per class, each 0.025–10.24 s; the ZIP then goes through the same validation, duplicate refusal, seeded split, adaptation, evaluation, export and reload stages. Each branch reads the file named in its path field (`BYOD_WAV_PATH`, `BYOD_ZIP_PATH`) on any runtime; with the field empty, Google Colab shows an upload dialog. Files stay in this runtime.

The Audio Spectrogram Transformer (AST) applies a Vision Transformer (ViT) architecture directly to audio spectrograms. At inference, input audio is resampled to 16 kHz, converted into a 128-bin Kaldi filterbank with a 10 ms hop, padded or cropped to 1024 frames (10.24 s), and processed by the transformer backbone.

**Supervised adaptation in this notebook:** besides pretrained 527-class AudioSet event inference, the notebook adapts the model to a three-class acoustic ecology task (`ADAPT_CLASSES = ('geophony', 'biophony', 'anthrophony')`). The backbone is frozen and the classifier head is re-headed (`Linear(768, 3)` behind the head's LayerNorm), so only 3,843 parameters are trained, with bounded AdamW on cached backbone features. The training itself is short: the 2026-09-16 local pre-flight measured 1.3 s for the five epochs in a warm CPU process (Windows, Python 3.12). On a fresh hosted runtime, building the isolated environment and downloading the model take longer — a few minutes is an estimate that depends on the network. Evaluation against a training-majority baseline and a portable adapter export (`ast-audio-adapter-v1.pt`) complete the end-to-end workflow.

**Learning objectives:** after this notebook you should be able to (1) explain how AST turns a waveform into a spectrogram and why the pretrained AudioSet head's sigmoid scores are not calibrated probabilities; (2) explain why a held-out evaluation needs distinct, independent clips, and check a dataset for duplicates before splitting; (3) adapt a pretrained audio model to new classes by re-heading it and training only the head on a frozen backbone; (4) read accuracy, macro-F1, per-class scores and a confusion matrix, and compare them with a baseline fitted on training labels; (5) export the adapted head, reload it in a new process and verify that it reproduces the trained model's scores; and (6) change one variable (evaluation noise) and explain what the change does to the metrics.

**This notebook does not demonstrate:** speech transcription, speaker identification, temporal localisation of events inside the window, source separation, calibrated confidence, or unconstrained multi-hour training. Adaptation is bounded to the classifier head.

## How to use this notebook

**Who this notebook is for.** Learners who can run cells in a hosted notebook and read short Python, and who want to see how a pretrained audio model is evaluated and adapted to new classes. No prior experience with audio models is assumed: each term is explained where it is first needed, and the glossary below collects them.

**Running it.** In Colab, choose *Runtime → Run all*. The default path needs no edit, no upload, no account, no token and no runtime restart. Sections 1–3 build an isolated environment from hash-locked packages and download about 346 MB of verified weights, so they take the longest before any model runs; read ahead while they finish, or run one cell at a time with *Shift + Enter*.

**Where the code runs.** The notebook kernel installs nothing and imports no model library. Each learner cell calls `run_stage('…')`, which runs one stage of the carried stage runner in its own process with the isolated environment's Python, streams what it prints, and stops the notebook with the stage's own error message if it fails. Stages hand results to each other only through files in the run directory — the verified snapshot, the dataset, the adapter and JSON records.

**Two kinds of cell.** *Learner cells* (Sections 4–14) are the machine-learning workflow; each runs one stage or reads one record and prints compact dictionaries for you to read. *Infrastructure cells* (Sections 1–3: the runtime check, the carried code, the isolated install and the pinned-weight staging) are collapsed and titled **Infrastructure**. You may run them without studying their implementation: they exist for reproducibility and provenance, not as prerequisite machine-learning knowledge.

**Form controls.** Some learner cells start with fields that Colab renders as a form: `USE_BYOD` and `BYOD_WAV_PATH` (Section 5); `USE_BYOD_DATASET` and `BYOD_ZIP_PATH` (Section 6); and `RUN_ACTIVITY` and `ACTIVITY_SNR_DB` in the optional activity (Section 14). Section 9 also sets `EPOCHS`, `BATCH_SIZE` and `LEARNING_RATE`. Leave them at their defaults for the first run: the notes and sample answers describe the default path.

**Section tags.** Each numbered heading carries one tag. **[Concept]** — what the model does and why. **[Evaluation practice]** — how the evidence is produced and how to read it. **[Engineering]** — reproducibility, provenance and packaging.

**Predict, then check.** Before each principal result a **Predict before running** prompt asks you to commit to an expectation; after it, **What to notice** describes normal output, and a collapsed **Check your reasoning** answer follows each checkpoint. Write your own answer first, then open it. Exact numbers can vary between CPU and GPU and between library builds, so the notes describe the shape of a normal result rather than fixed values.

## The task: Input → Model/System → Output

| Stage | Input | Model / system | Output |
|---|---|---|---|
| **Pretrained inference** | one 16 kHz waveform | filterbank spectrogram → AST backbone → the original 527-class AudioSet head, sigmoid per class | five AudioSet labels with independent scores |
| **Adaptation** | 18 labelled training clips | frozen backbone features → a new 3-class head trained with AdamW and cross-entropy | a small adapter file holding only the head |
| **Evaluation** | 6 held-out labelled clips and one unseen clip | the frozen backbone plus the reloaded head, softmax over three classes, argmax decision | accuracy, macro-F1, per-class scores, a confusion matrix, and a comparison with a baseline |

## Roadmap

| Section | Tag | What happens | What you read |
|---|---|---|---|
| 1. Check the runtime | [Engineering] | Linux and disk checked; a fresh run directory | the GPU (or CPU) |
| 2. Carry the code, install the runtime | [Engineering] | carried files verified; an isolated hash-locked environment | versions |
| 3. Pin, stage and verify | [Engineering] | the snapshot downloaded and digest-checked | file count |
| 4. Confirm the runtime | [Engineering] | versions checked against the lock | device, ceilings |
| 5. The pretrained head | [Concept] | AudioSet labels for a 440 Hz tone | top-5 labels |
| 6. The dataset | [Evaluation practice] | 24 clips built, validated, duplicates refused | distinct waveforms |
| 7. The split | [Evaluation practice] | 18 train / 6 held-out evaluation | counts, duplicates = 0 |
| 8. Re-head and baselines | [Concept] | new head, frozen backbone, two baselines | the score to beat |
| 9. Fine-tune and export | [Concept] | five epochs on the head; adapter saved | the epoch history |
| 10. Held-out evaluation | [Evaluation practice] | the adapter scored in a fresh process | the principal result |
| 11. An unseen clip | [Evaluation practice] | one new clip, three scores | the decision rule |
| 12. Reload parity | [Engineering] | another fresh process reproduces every score | `PASSED` |
| 13. Provenance bundle | [Engineering] | one JSON linking every record | the file list |
| 14. Optional activity | [Concept] | add noise to the evaluation clips (off by default) | your comparison |
| Troubleshooting | [Engineering] | common failures and what to do | when something fails |
| Interpretation and conclusion | [Evaluation practice] | limits and an evidence-based conclusion | your conclusion |

**Fast path.** Short on time? Run all, then read Sections 8, 10 and 12 and the conclusion: they carry the principal results. The canonical path ends with Section 13; Section 14 changes nothing unless you switch it on.

<details>
<summary><strong>Glossary</strong> — open when a term is unfamiliar</summary>

| Term | Meaning in this notebook |
|---|---|
| **Waveform** | The audio as a sequence of numbers between -1 and 1, 16,000 per second (16 kHz). |
| **Spectrogram / filterbank** | A time × frequency picture of the sound: 128 frequency bands every 10 ms. AST reads this picture. |
| **Backbone** | The transformer that turns the spectrogram into one 768-number summary (a *feature*). Frozen here. |
| **Head** | The small layer that turns the feature into one score per class. Replaced and trained here. |
| **Re-heading** | Swapping the 527-class AudioSet head for a new, randomly initialised 3-class head. |
| **Freezing** | Marking the backbone's parameters as not trainable, so only the head changes. |
| **Sigmoid score** | The pretrained head's per-label score in 0–1; labels are scored independently and the scores do not sum to one. |
| **Softmax score** | The adapted head's scores; they are positive and sum to one over the three classes, but they are not calibrated probabilities. |
| **Argmax** | The decision rule: the class with the highest score is the prediction. |
| **Calibration** | Whether a score of 0.9 is right about 90% of the time. Nothing here calibrates the scores. |
| **Epoch** | One pass over the 18 training clips. |
| **Held-out evaluation set** | Clips kept out of training and used only to measure the result. No separate validation set is used to choose anything. |
| **Accuracy** | The share of evaluation clips whose predicted class is correct. |
| **Macro-F1** | The F1 score (balance of precision and recall) computed per class, then averaged with equal weight per class. |
| **Confusion matrix** | A table of counts: rows are the true class, columns the predicted class; the diagonal holds the correct predictions. |
| **Majority baseline** | A trivial rule that always predicts the most common class in the training split. |
| **SNR (dB)** | Signal-to-noise ratio: how much louder the clip is than the added noise; 0 dB means equally loud. |
| **Adapter** | The exported file holding only the trained head, class names and the base model's identity. |
| **Digest (SHA-256)** | A fingerprint of a file's bytes or a waveform's samples; a single changed value changes it. |
| **Hash-locked environment** | A separate Python environment built from a requirements file that pins every package to one version and one set of SHA-256 digests; the installer refuses anything else. |
| **Stage** | One step of the workflow run as its own process by `run_stage`; it reads the files earlier stages wrote and writes its own. |
| **BYOD** | Bring Your Own Data: optional switches to run the same workflow on your own audio. |

</details>

## Prerequisites

- **Runtime:** a fresh Linux x86_64 runtime — Google Colab, Kaggle, or a Linux Jupyter server — with about 11 GB of free disk. A GPU is used automatically when present (the default runtime type is a T4) but is not required: every stage also runs on CPU, in float32 on both. Windows and macOS kernels are not supported, because the hash-locked environment is built for manylinux x86_64.
- **Knowledge:** basic Python; what a classifier and a train/test split are. Spectrograms, re-heading and the metrics are explained where they are first used; the glossary collects the terms.
- **Data:** the default path is self-contained and generates its audio in code. Optional single-clip BYOD accepts one PCM WAV. Optional dataset BYOD accepts one ZIP of `geophony/`, `biophony/` and `anthrophony/` folders (at the top level or inside one enclosing folder; `__MACOSX/` and dot-files are skipped) with at least two 16 kHz PCM WAV files per class; clips must be 0.025–10.24 s and no clip may appear twice. The ZIP is capped at 100 files, 64 MiB compressed, and 256 MiB expanded, and it is read in memory, never extracted. Do not upload confidential or restricted audio to a hosted runtime unless authorized.
- **External access:** the Hugging Face Hub, to fetch the pinned `MIT/ast-finetuned-audioset-10-10-0.4593` snapshot (~346 MB) at revision `f826b80d2822…`; PyPI (`files.pythonhosted.org`), for the pinned `uv` wheel and the 47 hash-locked packages; and the managed CPython build (python-build-standalone) that `uv` downloads for the isolated environment. No repository clone and no credentials are required; nothing is installed from this repository.

## 1. Check the runtime · [Engineering]

> **Infrastructure.** The code cells in Sections 1–3 are collapsed. You may run them without studying their implementation; they exist for reproducibility and provenance. The learning activities start in Section 4.

**Input:** a fresh hosted runtime. **System:** checks that it is Linux x86_64 with enough free disk, reports the GPU if there is one, and creates a new run directory. **Output:** the directories this run will use. Each run writes to a new directory under `outputs/ast_audio_classification/`, so an earlier export cannot be mistaken for a current result. The verified snapshot is kept in `weights/` and reused by a later run.

In [ ]:
# @title Infrastructure: check the runtime and disk; create a fresh run directory
import hashlib
import json
import os
import platform
import shutil
import subprocess
import tempfile
import time
import uuid
from pathlib import Path

SESSION_START = time.perf_counter()
if platform.system() != 'Linux' or platform.machine() != 'x86_64':
    raise RuntimeError('This notebook needs a Linux x86_64 runtime (Google Colab, Kaggle or Linux Jupyter): its locked environment is built for manylinux x86_64.')
try:
    gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True, text=True)
    GPU_NAME = gpu.stdout.strip() if gpu.returncode == 0 and gpu.stdout.strip() else None
except FileNotFoundError:
    GPU_NAME = None
STEM = 'ast_audio_classification'
ROOT = Path.cwd() / 'outputs' / STEM / uuid.uuid4().hex[:12]
ROOT.mkdir(parents=True)
WEIGHTS = Path.cwd() / 'weights'
WEIGHTS.mkdir(exist_ok=True)
ENV_ROOT = Path(tempfile.gettempdir()) / (STEM + '_env_' + ROOT.name)
staged_gib = sum(p.stat().st_size for p in WEIGHTS.rglob('*') if p.is_file()) / 1024**3
need = {'weights': max(0.0, 0.5 - staged_gib), 'environment': 10.0}
free = {'weights': shutil.disk_usage(WEIGHTS).free / 1024**3, 'environment': shutil.disk_usage(tempfile.gettempdir()).free / 1024**3}
if os.stat(WEIGHTS).st_dev == os.stat(tempfile.gettempdir()).st_dev:
    short = free['weights'] < need['weights'] + need['environment']
else:
    short = free['weights'] < need['weights'] or free['environment'] < need['environment']
if short:
    raise RuntimeError(f'Not enough free disk: need about {need} GiB, free {free} GiB. Start a fresh runtime (see Troubleshooting).')
print({'gpu': GPU_NAME or 'none (the stages run on CPU)', 'kernel_python': platform.python_version(), 'run_directory': str(ROOT), 'weights': str(WEIGHTS), 'environment': str(ENV_ROOT), 'free_gib': {k: round(v, 1) for k, v in free.items()}})

{'gpu': 'Tesla T4, 15360 MiB', 'kernel_python': '3.13.15', 'run_directory': '/content/outputs/ast_audio_classification/322b15d2d44d', 'weights': '/content/weights', 'environment': '/tmp/ast_audio_classification_env_322b15d2d44d', 'free_gib': {'weights': 193.2, 'environment': 193.2}}


**Expected result:** one dictionary naming the GPU (for example `Tesla T4, 15360 MiB`) or `none (the stages run on CPU)`, the kernel's Python version, the run directory, the weights directory, the isolated environment's directory and the free disk. If the cell stops with a platform or disk message, see **Troubleshooting**.

## 2. Carry the code and install the locked runtime · [Engineering]

> **Infrastructure.** The next two code cells are collapsed. The first **is** the repository's code, carried so that this notebook works on its own; the second builds the environment every stage runs in.

The first cell holds, as text, the files the workflow needs: the package's four modules under `src/ast_audio_classification_pipeline/` (identity constants, snapshot verification and staging, validation, the pipeline class, the generated sample data and the metrics), the stage runner `tutorial_stages.py`, the hash-locked `requirements.txt` (47 packages), the snapshot manifest and the licence. It writes each file into the run directory and checks its SHA-256 against `CARRIED_HASHES`, stopping on any mismatch. The text is the repository's files byte for byte; the repository's parity test (`tests/test_notebook_parity.py`) fails whenever the two diverge, so what runs here is what the repository tests. Nothing in this cell runs a model.

In [ ]:
# @title Infrastructure: write and verify the carried package, stage runner, lock and manifests
CARRIED_FILES = {'src/ast_audio_classification_pipeline/__init__.py': 'from .metrics import (\n    confusion_matrix,\n    evaluate_classification,\n    macro_f1_score,\n    majority_class_baseline,\n    multiclass_accuracy,\n    per_class_metrics,\n)\nfrom .pipeline import (\n    ARTIFACT_FORMAT,\n    ARTIFACT_FORMAT_VERSION,\n    DEFAULT_TOP_K,\n    DEFAULT_WEIGHTS_DIR,\n    INPUT_SCHEMA,\n    MAX_AUDIO_SECONDS,\n    MAX_INPUT_SECONDS,\n    MIN_AUDIO_SECONDS,\n    MODEL_ID,\n    MODEL_KEY,\n    MODEL_LICENSE,\n    MODEL_REVISION,\n    NUM_LABELS,\n    SAMPLE_RATE,\n    ASTAudioClassificationPipeline,\n    evaluation_report,\n    freeze_backbone,\n    rehead_model,\n    stage_missing_files,\n    validate_inputs,\n    verify_snapshot,\n)\nfrom .samples import (\n    ADAPT_CLASSES,\n    DATASET_REPRESENTATION,\n    MAX_DATASET_AUDIO_SECONDS,\n    MIN_DATASET_AUDIO_SECONDS,\n    generate_audio_clip,\n    split_dataset,\n    synthetic_audio_dataset,\n    tutorial_tone,\n    validate_dataset,\n)\n\n__all__ = [\n    "ADAPT_CLASSES",\n    "ARTIFACT_FORMAT",\n    "ARTIFACT_FORMAT_VERSION",\n    "DATASET_REPRESENTATION",\n    "DEFAULT_TOP_K",\n    "DEFAULT_WEIGHTS_DIR",\n    "INPUT_SCHEMA",\n    "MAX_AUDIO_SECONDS",\n    "MAX_DATASET_AUDIO_SECONDS",\n    "MAX_INPUT_SECONDS",\n    "MIN_AUDIO_SECONDS",\n    "MIN_DATASET_AUDIO_SECONDS",\n    "MODEL_ID",\n    "MODEL_KEY",\n    "MODEL_LICENSE",\n    "MODEL_REVISION",\n    "NUM_LABELS",\n    "SAMPLE_RATE",\n    "ASTAudioClassificationPipeline",\n    "confusion_matrix",\n    "evaluate_classification",\n    "evaluation_report",\n    "freeze_backbone",\n    "generate_audio_clip",\n    "macro_f1_score",\n    "majority_class_baseline",\n    "multiclass_accuracy",\n    "per_class_metrics",\n    "rehead_model",\n    "split_dataset",\n    "stage_missing_files",\n    "synthetic_audio_dataset",\n    "tutorial_tone",\n    "validate_dataset",\n    "validate_inputs",\n    "verify_snapshot",\n]\n', 'src/ast_audio_classification_pipeline/metrics.py': '"""Evaluation metrics for multiclass audio classification."""\n\nfrom __future__ import annotations\n\nfrom collections import Counter\nfrom collections.abc import Sequence\nfrom typing import Any\n\n\ndef _validate_num_classes(num_classes: int) -> None:\n    if isinstance(num_classes, bool) or not isinstance(num_classes, int) or num_classes < 1:\n        raise ValueError("num_classes must be a positive int")\n\n\ndef _validate_class_indices(values: Sequence[int], num_classes: int, *, name: str) -> None:\n    for value in values:\n        if isinstance(value, bool) or not isinstance(value, int):\n            raise TypeError(f"{name} must contain ints, got {type(value).__name__}")\n        if not 0 <= value < num_classes:\n            raise ValueError(f"{name} class index {value} outside [0, {num_classes - 1}]")\n\n\ndef multiclass_accuracy(predictions: Sequence[int], targets: Sequence[int]) -> float:\n    """Compute overall accuracy as the fraction of correctly classified clips."""\n    if len(predictions) != len(targets):\n        raise ValueError(f"length mismatch: predictions={len(predictions)}, targets={len(targets)}")\n    if len(targets) == 0:\n        raise ValueError("cannot compute accuracy over empty targets")\n    correct = sum(1 for p, t in zip(predictions, targets, strict=True) if p == t)\n    return correct / len(targets)\n\n\ndef confusion_matrix(\n    predictions: Sequence[int], targets: Sequence[int], num_classes: int\n) -> list[list[int]]:\n    """Compute confusion matrix where rows are ground truth classes and columns are predicted classes."""\n    _validate_num_classes(num_classes)\n    if len(predictions) != len(targets):\n        raise ValueError(f"length mismatch: predictions={len(predictions)}, targets={len(targets)}")\n    _validate_class_indices(predictions, num_classes, name="predictions")\n    _validate_class_indices(targets, num_classes, name="targets")\n    matrix = [[0] * num_classes for _ in range(num_classes)]\n    for p, t in zip(predictions, targets, strict=True):\n        matrix[t][p] += 1\n    return matrix\n\n\ndef per_class_metrics(\n    predictions: Sequence[int], targets: Sequence[int], num_classes: int\n) -> dict[int, dict[str, float]]:\n    """Compute per-class precision, recall, and F1-score."""\n    if len(predictions) != len(targets):\n        raise ValueError(f"length mismatch: predictions={len(predictions)}, targets={len(targets)}")\n    matrix = confusion_matrix(predictions, targets, num_classes)\n    results: dict[int, dict[str, float]] = {}\n    for c in range(num_classes):\n        tp = matrix[c][c]\n        fp = sum(matrix[other][c] for other in range(num_classes) if other != c)\n        fn = sum(matrix[c][other] for other in range(num_classes) if other != c)\n        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0\n        recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0\n        f1 = (2.0 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0\n        results[c] = {\n            "precision": round(float(precision), 4),\n            "recall": round(float(recall), 4),\n            "f1": round(float(f1), 4),\n            "support": sum(matrix[c]),\n        }\n    return results\n\n\ndef macro_f1_score(\n    predictions: Sequence[int], targets: Sequence[int], num_classes: int\n) -> float:\n    """Compute unweighted macro-averaged F1 score across all classes."""\n    pcm = per_class_metrics(predictions, targets, num_classes)\n    f1_sum = sum(metrics["f1"] for metrics in pcm.values())\n    return round(float(f1_sum / num_classes), 4) if num_classes > 0 else 0.0\n\n\ndef majority_class_baseline(targets: Sequence[int], num_classes: int) -> dict[str, Any]:\n    """Compute the accuracy of a constant trivial classifier predicting the majority class."""\n    _validate_num_classes(num_classes)\n    if len(targets) == 0:\n        raise ValueError("cannot compute baseline over empty targets")\n    _validate_class_indices(targets, num_classes, name="targets")\n    counts = Counter(targets)\n    majority_class, count = counts.most_common(1)[0]\n    majority_acc = count / len(targets)\n    return {\n        "majority_class_index": int(majority_class),\n        "majority_class_count": int(count),\n        "majority_class_accuracy": round(float(majority_acc), 4),\n        "num_classes": num_classes,\n        "n_samples": len(targets),\n    }\n\n\ndef evaluate_classification(\n    predictions: Sequence[int],\n    targets: Sequence[int],\n    class_names: Sequence[str],\n) -> dict[str, Any]:\n    """Comprehensive multi-class classification evaluation against baselines."""\n    num_classes = len(class_names)\n    if num_classes < 1:\n        raise ValueError("class_names must not be empty")\n    if any(not isinstance(name, str) or not name.strip() for name in class_names):\n        raise ValueError("class_names must contain non-empty strings")\n    if len(set(class_names)) != num_classes:\n        raise ValueError("class_names must be unique")\n    acc = multiclass_accuracy(predictions, targets)\n    pcm = per_class_metrics(predictions, targets, num_classes)\n    macro_f1 = macro_f1_score(predictions, targets, num_classes)\n    baseline = majority_class_baseline(targets, num_classes)\n    cm = confusion_matrix(predictions, targets, num_classes)\n\n    named_per_class = {class_names[c]: pcm[c] for c in range(num_classes)}\n\n    return {\n        "accuracy": round(float(acc), 4),\n        "macro_f1": macro_f1,\n        "per_class": named_per_class,\n        "confusion_matrix": cm,\n        "baseline": baseline,\n        "accuracy_delta_vs_baseline": round(float(acc - baseline["majority_class_accuracy"]), 4),\n        "num_samples": len(targets),\n        "num_classes": num_classes,\n    }\n', 'src/ast_audio_classification_pipeline/samples.py': '"""Deterministic in-code sample audio data: pretrained demonstration tone and acoustic ecology dataset.\n\nNothing here is downloaded and nothing requires external dependencies beyond numpy,\nensuring zero network dataset dependencies and deterministic execution across clean-room environments.\n\nTwo separate label contexts live here and must not be conflated:\n1. ``tutorial_tone``: 440 Hz sinusoidal waveform demonstrating pretrained 527-class AudioSet inference.\n2. ``synthetic_audio_dataset``: 24 audio clips labelled with ``ADAPT_CLASSES``, the canonical\n   Krause/Pijanowski tripartite acoustic ecology vocabulary:\n   - ``geophony``: non-biological natural sounds (rain, wind, flowing water).\n   - ``biophony``: non-human biological vocalizations (birdsong, insect chirps).\n   - ``anthrophony``: human-generated mechanical sounds (engines, drones, machinery).\n"""\n\nfrom __future__ import annotations\n\nimport random\nfrom collections.abc import Sequence\nfrom typing import Any\n\nimport numpy as np\n\nSAMPLE_RATE = 16_000\nSAMPLE_DURATION = 3.0  # seconds; comfortably inside AST\'s 10.24 s window\nMIN_DATASET_AUDIO_SECONDS = 0.025\nMAX_DATASET_AUDIO_SECONDS = 10.24\nTONE_HZ = 440.0\nTONE_AMPLITUDE = 0.5\n\n# Canonical Krause/Pijanowski acoustic ecology classification vocabulary\nADAPT_CLASSES: tuple[str, ...] = ("geophony", "biophony", "anthrophony")\nDATASET_REPRESENTATION = "io.github.kurtvalcorza.dataset.audio.waveform-classification.v1"\n\n\ndef tutorial_tone(\n    duration: float = SAMPLE_DURATION,\n    sample_rate: int = SAMPLE_RATE,\n    frequency: float = TONE_HZ,\n    amplitude: float = TONE_AMPLITUDE,\n) -> np.ndarray:\n    """Deterministic single-tone audio waveform demonstrating pretrained AudioSet inference."""\n    if not isinstance(sample_rate, int) or isinstance(sample_rate, bool) or sample_rate <= 0:\n        raise ValueError("sample_rate must be a positive int")\n    if not isinstance(duration, int | float) or isinstance(duration, bool) or duration <= 0:\n        raise ValueError("duration must be positive")\n    if not isinstance(frequency, int | float) or isinstance(frequency, bool) or frequency <= 0:\n        raise ValueError("frequency must be positive")\n    if not isinstance(amplitude, int | float) or isinstance(amplitude, bool) or not 0 <= amplitude <= 1:\n        raise ValueError("amplitude must be in [0, 1]")\n    t = np.linspace(0.0, duration, int(sample_rate * duration), endpoint=False, dtype=np.float32)\n    return (amplitude * np.sin(2.0 * np.pi * frequency * t)).astype(np.float32)\n\n\ndef generate_audio_clip(\n    index: int,\n    class_idx: int,\n    duration: float = SAMPLE_DURATION,\n    sample_rate: int = SAMPLE_RATE,\n    seed: int = 42,\n) -> np.ndarray:\n    """Generate a deterministic synthetic 1-D audio waveform characteristic of its acoustic class."""\n    if isinstance(index, bool) or not isinstance(index, int) or index < 0:\n        raise ValueError("index must be a non-negative int")\n    valid_class_idx = (\n        isinstance(class_idx, int)\n        and not isinstance(class_idx, bool)\n        and 0 <= class_idx < len(ADAPT_CLASSES)\n    )\n    if not valid_class_idx:\n        raise ValueError(f"class_idx must be an int in [0, {len(ADAPT_CLASSES) - 1}]")\n    if sample_rate != SAMPLE_RATE:\n        raise ValueError(f"sample_rate must be {SAMPLE_RATE}")\n    if not isinstance(duration, int | float) or isinstance(duration, bool):\n        raise ValueError("duration must be numeric")\n    if not MIN_DATASET_AUDIO_SECONDS <= duration <= MAX_DATASET_AUDIO_SECONDS:\n        raise ValueError(\n            f"duration must be in [{MIN_DATASET_AUDIO_SECONDS}, {MAX_DATASET_AUDIO_SECONDS}]"\n        )\n    if isinstance(seed, bool) or not isinstance(seed, int):\n        raise ValueError("seed must be an int")\n    n_samples = int(sample_rate * duration)\n    t = np.linspace(0.0, duration, n_samples, endpoint=False, dtype=np.float32)\n    rng = np.random.default_rng(seed=seed + index * 101 + class_idx * 17)\n\n    if class_idx == 0:\n        # Geophony: broadband turbulent noise with slow gentle envelope (simulating rain/wind)\n        raw_noise = rng.standard_normal(n_samples).astype(np.float32)\n        # 1-pole low-pass smooth filter\n        alpha = 0.15\n        filtered = np.zeros_like(raw_noise)\n        acc = 0.0\n        for i in range(n_samples):\n            acc = alpha * raw_noise[i] + (1.0 - alpha) * acc\n            filtered[i] = acc\n        # Gentle swell envelope\n        envelope = 0.7 + 0.3 * np.sin(2.0 * np.pi * 0.5 * t)\n        signal = filtered * envelope\n    elif class_idx == 1:\n        # Biophony: rapid frequency-modulated chirp whistles (simulating bird song / biophonic calls)\n        chirp_rate = 3.0 + (index % 3) * 1.5  # chirps per second\n        f_center = 2800.0 + (index % 4) * 400.0\n        f_dev = 1200.0\n        # Phase modulation\n        instantaneous_freq = f_center + f_dev * np.sin(2.0 * np.pi * chirp_rate * t)\n        phase = 2.0 * np.pi * np.cumsum(instantaneous_freq) / sample_rate\n        fundamental = np.sin(phase)\n        harmonic = 0.35 * np.sin(2.0 * phase)\n        # Pulsed amplitude gate\n        gate = np.clip(np.sin(2.0 * np.pi * chirp_rate * t), 0.0, 1.0) ** 2\n        signal = (fundamental + harmonic) * gate\n    elif class_idx == 2:\n        # Anthrophony: low-frequency harmonic drone with motor buzz (simulating engine/machinery)\n        f0 = 120.0 + (index % 3) * 20.0  # 120 Hz fundamental\n        harmonics = [\n            (1.0, 0.45),\n            (2.0, 0.30),\n            (3.0, 0.20),\n            (4.0, 0.15),\n            (5.0, 0.10),\n        ]\n        drone = np.zeros(n_samples, dtype=np.float32)\n        for mult, weight in harmonics:\n            drone += weight * np.sin(2.0 * np.pi * (f0 * mult) * t)\n        # Add motor stroke amplitude pulsation (15 Hz)\n        engine_pulse = 0.75 + 0.25 * np.sin(2.0 * np.pi * 15.0 * t)\n        signal = drone * engine_pulse\n    # Normalize peak amplitude safely to [-0.85, 0.85]\n    peak = np.max(np.abs(signal))\n    if peak > 1e-6:\n        signal = (signal / peak) * 0.85\n    return signal.astype(np.float32)\n\n\ndef synthetic_audio_dataset(\n    n_samples: int = 24,\n    seed: int = 42,\n    duration: float = SAMPLE_DURATION,\n    sample_rate: int = SAMPLE_RATE,\n) -> list[dict[str, Any]]:\n    """Generate a balanced deterministic dataset across ADAPT_CLASSES."""\n    if isinstance(n_samples, bool) or not isinstance(n_samples, int) or n_samples < len(ADAPT_CLASSES) * 2:\n        raise ValueError(f"n_samples must be an int >= {len(ADAPT_CLASSES) * 2}")\n    if n_samples % len(ADAPT_CLASSES):\n        raise ValueError(f"n_samples must be divisible by {len(ADAPT_CLASSES)} for class balance")\n    if isinstance(seed, bool) or not isinstance(seed, int):\n        raise ValueError("seed must be an int")\n    num_classes = len(ADAPT_CLASSES)\n    records: list[dict[str, Any]] = []\n    for idx in range(n_samples):\n        class_idx = idx % num_classes\n        waveform = generate_audio_clip(\n            index=idx,\n            class_idx=class_idx,\n            duration=duration,\n            sample_rate=sample_rate,\n            seed=seed,\n        )\n        records.append(\n            {\n                "id": f"clip-{idx:03d}",\n                "waveform": waveform,\n                "sample_rate": sample_rate,\n                "label": class_idx,\n                "class_name": ADAPT_CLASSES[class_idx],\n                "duration": round(float(duration), 4),\n            }\n        )\n    return records\n\n\ndef split_dataset(\n    records: Sequence[dict[str, Any]],\n    val_fraction: float = 0.25,\n    seed: int = 42,\n) -> tuple[list[dict[str, Any]], list[dict[str, Any]]]:\n    """Stratified train/validation split preserving balanced class distribution."""\n    if not 0.0 < val_fraction < 1.0:\n        raise ValueError("val_fraction must be strictly between 0 and 1")\n    by_class: dict[int, list[dict[str, Any]]] = {}\n    for r in records:\n        by_class.setdefault(r["label"], []).append(r)\n\n    rng = random.Random(seed)\n    train_records: list[dict[str, Any]] = []\n    val_records: list[dict[str, Any]] = []\n\n    for label in sorted(by_class.keys()):\n        class_list = list(by_class[label])\n        if len(class_list) < 2:\n            raise ValueError(f"class {label} requires at least 2 records for a disjoint split")\n        rng.shuffle(class_list)\n        n_val = min(len(class_list) - 1, max(1, round(len(class_list) * val_fraction)))\n        val_records.extend(class_list[:n_val])\n        train_records.extend(class_list[n_val:])\n\n    rng.shuffle(train_records)\n    rng.shuffle(val_records)\n    return train_records, val_records\n\n\ndef validate_dataset(\n    records: Sequence[dict[str, Any]],\n    class_names: Sequence[str] = ADAPT_CLASSES,\n) -> dict[str, Any]:\n    """Validate that audio records strictly conform to the classification contract."""\n    if not isinstance(records, Sequence) or isinstance(records, str | bytes):\n        raise TypeError(f"records must be a sequence, got {type(records).__name__}")\n    if len(records) < 2:\n        raise ValueError(f"dataset requires at least 2 records, got {len(records)}")\n\n    num_classes = len(class_names)\n    if num_classes < 2:\n        raise ValueError("class_names must contain at least 2 classes")\n    if any(not isinstance(name, str) or not name.strip() for name in class_names):\n        raise ValueError("class_names must contain non-empty strings")\n    if len(set(class_names)) != num_classes:\n        raise ValueError("class_names must be unique")\n    class_counts = [0] * num_classes\n    seen_ids: set[str] = set()\n\n    for i, r in enumerate(records):\n        if not isinstance(r, dict):\n            raise TypeError(f"record {i} must be a dict, got {type(r).__name__}")\n        if "waveform" not in r:\n            raise KeyError(f"record {i} missing required key \'waveform\'")\n        if "label" not in r:\n            raise KeyError(f"record {i} missing required key \'label\'")\n        if "id" not in r:\n            raise KeyError(f"record {i} missing required key \'id\'")\n        if "sample_rate" not in r:\n            raise KeyError(f"record {i} missing required key \'sample_rate\'")\n        if "class_name" not in r:\n            raise KeyError(f"record {i} missing required key \'class_name\'")\n\n        record_id = r["id"]\n        if not isinstance(record_id, str) or not record_id.strip():\n            raise ValueError(f"record {i} id must be a non-empty string")\n        if record_id in seen_ids:\n            raise ValueError(f"record {i} duplicates id {record_id!r}")\n        seen_ids.add(record_id)\n\n        sample_rate = r["sample_rate"]\n        if sample_rate != SAMPLE_RATE:\n            raise ValueError(f"record {i} sample_rate must be {SAMPLE_RATE}, got {sample_rate!r}")\n\n        waveform = r["waveform"]\n        if not isinstance(waveform, np.ndarray):\n            raise TypeError(f"record {i} waveform must be a np.ndarray, got {type(waveform).__name__}")\n        if waveform.ndim != 1:\n            raise ValueError(f"record {i} waveform must be 1-D mono, got shape {waveform.shape}")\n        if waveform.dtype != np.float32:\n            raise TypeError(f"record {i} waveform must be float32, got {waveform.dtype}")\n        if not np.all(np.isfinite(waveform)):\n            raise ValueError(f"record {i} waveform contains NaN or inf values")\n        if waveform.size == 0:\n            raise ValueError(f"record {i} waveform must not be empty")\n        if np.max(np.abs(waveform)) > 1.0:\n            raise ValueError(f"record {i} waveform samples must be in [-1, 1]")\n        duration = waveform.size / sample_rate\n        if not MIN_DATASET_AUDIO_SECONDS <= duration <= MAX_DATASET_AUDIO_SECONDS:\n            raise ValueError(\n                f"record {i} duration {duration:.4f} s outside "\n                f"[{MIN_DATASET_AUDIO_SECONDS}, {MAX_DATASET_AUDIO_SECONDS}]"\n            )\n\n        label = r["label"]\n        if isinstance(label, bool) or not isinstance(label, int):\n            raise TypeError(f"record {i} label must be an int, got {type(label).__name__}")\n        if not 0 <= label < num_classes:\n            raise ValueError(f"record {i} label {label} outside valid range [0, {num_classes - 1}]")\n        if r["class_name"] != class_names[label]:\n            raise ValueError(\n                f"record {i} class_name {r[\'class_name\']!r} does not match "\n                f"label {label} ({class_names[label]!r})"\n            )\n\n        class_counts[label] += 1\n\n    missing_classes = [class_names[i] for i, count in enumerate(class_counts) if count == 0]\n    if missing_classes:\n        raise ValueError(f"dataset has no records for classes: {missing_classes}")\n\n    return {\n        "representation": DATASET_REPRESENTATION,\n        "n_records": len(records),\n        "num_classes": num_classes,\n        "class_counts": {class_names[c]: class_counts[c] for c in range(num_classes)},\n        "verdict": "accepted",\n    }\n', 'src/ast_audio_classification_pipeline/pipeline.py': 'from __future__ import annotations\n\nimport hashlib\nimport json\nfrom collections.abc import Callable, Mapping, Sequence\nfrom dataclasses import dataclass, field\nfrom pathlib import Path\nfrom typing import Any\n\nimport numpy as np\n\nMODEL_ID = "MIT/ast-finetuned-audioset-10-10-0.4593"\nMODEL_REVISION = "f826b80d28226b62986cc218e5cec390b1096902"\nMODEL_LICENSE = "bsd-3-clause"\nMODEL_KEY = "ast-audioset"\nDEFAULT_WEIGHTS_DIR = Path(__file__).resolve().parents[2] / "weights" / MODEL_KEY\nMANIFEST_NAME = "dimer-base-manifest.json"\n\n# The pinned checkpoint expects 16 kHz mono float32. Its feature extractor computes a 128-bin Kaldi\n# fbank with a 10 ms hop and pads or crops to 1024 frames, so only the first 10.24 s of audio reach\n# the model; anything longer is cropped and reported as `truncated`.\nSAMPLE_RATE = 16_000\nWINDOW_FRAMES = 1024\nMAX_AUDIO_SECONDS = WINDOW_FRAMES * 0.010  # 10.24 s model window\nMAX_INPUT_SECONDS = 120.0  # hard ceiling: longer input is rejected, the caller must chunk\nMIN_AUDIO_SECONDS = 0.025  # one 25 ms fbank frame\nNUM_LABELS = 527  # AudioSet ontology classes in the pinned config.json\nDEFAULT_TOP_K = 5\nACTIVATION = "sigmoid"\nARTIFACT_FORMAT = "org.valcorza.ast-audio.adapter.v1"\nARTIFACT_FORMAT_VERSION = "1.0"\n\n\ndef verify_snapshot(path: str | Path | None = None) -> dict[str, Any]:\n    """Check the local snapshot against its DIMER manifest; raise naming the first mismatch."""\n    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR\n    manifest_path = root / MANIFEST_NAME\n    if not manifest_path.is_file():\n        raise FileNotFoundError(f"snapshot manifest not found: {manifest_path}")\n    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))\n    if manifest.get("modelId") != MODEL_ID:\n        raise ValueError(f"manifest modelId {manifest.get(\'modelId\')!r} != {MODEL_ID!r}")\n    if manifest.get("revision") != MODEL_REVISION:\n        raise ValueError(f"manifest revision {manifest.get(\'revision\')!r} != {MODEL_REVISION!r}")\n    for entry in manifest["files"]:\n        file_path = root / entry["path"]\n        if not file_path.is_file():\n            raise FileNotFoundError(f"snapshot file missing: {file_path}")\n        size = file_path.stat().st_size\n        if size != entry["bytes"]:\n            raise ValueError(f"{entry[\'path\']}: size {size} != manifest {entry[\'bytes\']}")\n        digest = hashlib.sha256()\n        with open(file_path, "rb") as fh:\n            for chunk in iter(lambda: fh.read(1 << 20), b""):\n                digest.update(chunk)\n        if digest.hexdigest() != entry["sha256"]:\n            raise ValueError(f"{entry[\'path\']}: sha256 {digest.hexdigest()} != manifest {entry[\'sha256\']}")\n    return manifest\n\n\ndef _hub_download(relative_path: str, root: Path) -> None:\n    """Fetch one manifest-listed file at MODEL_REVISION straight into the snapshot directory."""\n    from huggingface_hub import hf_hub_download\n\n    hf_hub_download(MODEL_ID, relative_path, revision=MODEL_REVISION, local_dir=str(root))\n\n\ndef stage_missing_files(\n    path: str | Path | None = None,\n    *,\n    allow_download: bool = False,\n    downloader: Callable[[str, Path], None] | None = None,\n) -> list[str]:\n    """Fetch manifest-listed files that are absent locally (a fresh clone commits the manifest but\n    git-ignores the weights). Returns the relative paths fetched; `verify_snapshot` still runs after."""\n    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR\n    manifest_path = root / MANIFEST_NAME\n    if not manifest_path.is_file():\n        raise FileNotFoundError(f"manifest not found: {manifest_path}")\n    with open(manifest_path, encoding="utf-8") as fh:\n        manifest = json.load(fh)\n    if manifest.get("modelId") != MODEL_ID or manifest.get("revision") != MODEL_REVISION:\n        raise ValueError(\n            f"manifest names {manifest.get(\'modelId\')}@{manifest.get(\'revision\')}, "\n            f"package pins {MODEL_ID}@{MODEL_REVISION}; refusing to stage"\n        )\n    missing = [entry["path"] for entry in manifest["files"] if not (root / entry["path"]).is_file()]\n    if not missing:\n        return []\n    if not allow_download:\n        raise FileNotFoundError(\n            f"snapshot at {root} is missing {missing}; "\n            f"pass allow_download=True to fetch them at {MODEL_REVISION}"\n        )\n    fetch = downloader or _hub_download\n    for relative_path in missing:\n        fetch(relative_path, root)\n    return missing\n\n\ndef _resample(waveform: np.ndarray, sample_rate: int) -> np.ndarray:\n    import torch\n    import torchaudio.functional as af\n\n    resampled = af.resample(torch.from_numpy(waveform), orig_freq=sample_rate, new_freq=SAMPLE_RATE)\n    return resampled.numpy().astype(np.float32)\n\n\nINPUT_SCHEMA: dict[str, Any] = {\n    "input": "1-D float numpy array of mono samples in [-1, 1], plus the sample_rate it was captured at",\n    "sample_rate_hz": f"any positive int; resampled to SAMPLE_RATE={SAMPLE_RATE} when it differs",\n    "duration_seconds": [MIN_AUDIO_SECONDS, MAX_INPUT_SECONDS],\n    "model_window_seconds": MAX_AUDIO_SECONDS,\n    "top_k": [1, NUM_LABELS],\n    "preprocessing": (\n        f"resample to {SAMPLE_RATE} Hz when needed, 128-bin Kaldi fbank with a 10 ms hop, "\n        f"pad or crop to {WINDOW_FRAMES} frames ({MAX_AUDIO_SECONDS} s)"\n    ),\n}\n\n\ndef _check_inputs(audio: Any, sample_rate: Any, top_k: Any) -> float:\n    """Raise TypeError/ValueError naming the first violated ceiling; return the clip duration in seconds."""\n    if not isinstance(audio, np.ndarray):\n        raise TypeError(f"audio must be a numpy.ndarray, got {type(audio).__name__}")\n    if audio.ndim != 1:\n        raise ValueError(f"audio must be 1-D mono, got shape {audio.shape}")\n    if not np.issubdtype(audio.dtype, np.floating):\n        raise TypeError(f"audio must be a float array in [-1, 1], got dtype {audio.dtype}")\n    if not isinstance(sample_rate, int) or isinstance(sample_rate, bool) or sample_rate <= 0:\n        raise TypeError("sample_rate must be a positive int (the rate the audio was captured at)")\n    if not isinstance(top_k, int) or isinstance(top_k, bool) or not 1 <= top_k <= NUM_LABELS:\n        raise ValueError(f"top_k must be an int in [1, {NUM_LABELS}]")\n    duration = audio.shape[0] / sample_rate\n    if duration < MIN_AUDIO_SECONDS:\n        raise ValueError(f"audio is {duration:.4f} s; minimum is {MIN_AUDIO_SECONDS} s")\n    if duration > MAX_INPUT_SECONDS:\n        raise ValueError(f"audio is {duration:.2f} s; ceiling is {MAX_INPUT_SECONDS} s (chunk it first)")\n    if not np.all(np.isfinite(audio)):\n        raise ValueError("audio contains NaN or inf samples")\n    if np.max(np.abs(audio)) > 1.0:\n        raise ValueError("audio samples must be in [-1, 1]")\n    return duration\n\n\ndef validate_inputs(\n    waveforms: Any,\n    sample_rate: Any,\n    *,\n    top_k: int = DEFAULT_TOP_K,\n    names: Sequence[str] | None = None,\n) -> dict[str, Any]:\n    """Validation stage: return the input manifest (schema, per-clip observations, verdict).\n\n    The checks are the ones ``predict`` applies, through the same private ``_check_inputs``, so a\n    rejection here raises exactly what ``predict`` would; a caller that wants the finding recorded\n    catches the exception and stores ``str(exc)`` under ``findings``.\n    """\n    batch = [waveforms] if isinstance(waveforms, np.ndarray) else waveforms\n    if not isinstance(batch, Sequence) or isinstance(batch, str | bytes):\n        raise TypeError("waveforms must be a 1-D numpy.ndarray or a sequence of them")\n    if len(batch) < 1:\n        raise ValueError("at least one waveform is required")\n    if names is not None and len(names) != len(batch):\n        raise ValueError("names must have one entry per waveform")\n    inputs = []\n    for index, waveform in enumerate(batch):\n        duration = _check_inputs(waveform, sample_rate, top_k)\n        inputs.append(\n            {\n                "id": names[index] if names else f"clip-{index}",\n                "samples": int(waveform.shape[0]),\n                "dtype": str(waveform.dtype),\n                "duration_seconds": round(duration, 4),\n                "peak_amplitude": round(float(np.max(np.abs(waveform))), 6),\n                "will_resample": sample_rate != SAMPLE_RATE,\n                "will_truncate": duration > MAX_AUDIO_SECONDS,\n            }\n        )\n    return {\n        "schema": dict(INPUT_SCHEMA),\n        "inputs": inputs,\n        "sample_rate": sample_rate,\n        "top_k": top_k,\n        "n_clips": len(inputs),\n        "verdict": "accepted",\n        "findings": [],\n        "model_id": MODEL_ID,\n        "model_revision": MODEL_REVISION,\n    }\n\n\ndef evaluation_report(\n    result: Mapping[str, Any],\n    targets: Sequence[Any] | None = None,\n    *,\n    sample_kind: str = "synthetic",\n) -> dict[str, Any]:\n    """Describe the unadapted AudioSet inference result without inventing ground truth."""\n    activation = result.get("activation", ACTIVATION)\n    if activation != ACTIVATION:\n        raise ValueError(\n            "evaluation_report only supports unadapted AudioSet sigmoid results; "\n            f"got activation={activation!r}"\n        )\n    predictions = result["predictions"]\n    reason = "no AudioSet-labelled ground truth exists for the evaluated clip"\n    if targets is not None:\n        reason = (\n            "targets were supplied, but they are not established as labels from the 527-class "\n            "AudioSet ontology; "\n            "score only ontology-aligned labels with an AudioSet-convention mAP implementation"\n        )\n    return {\n        "task": "multi-label audio event classification over the 527 AudioSet labels",\n        "score_semantics": (\n            f"independent {activation} score per label: the scores do not sum "\n            "to one, several labels can be high at once, none is a calibrated probability, and no "\n            "threshold is shipped"\n        ),\n        "sample_kind": sample_kind,\n        "n_clips": 1,\n        "n_scored_labels": len(predictions),\n        "metrics": [],\n        "baselines": [],\n        "verdict": "not-measurable",\n        "reason": reason,\n        "needs": (\n            f"clips labelled against the same {NUM_LABELS}-label AudioSet ontology, scored over the full "\n            "score vector (predict(..., top_k=527)) with mean average precision plus per-label precision "\n            "and recall at a threshold chosen on your own labelled clips; the \'0.4593\' in the checkpoint "\n            "name is the upstream-reported AudioSet mAP and is not measured here"\n        ),\n        "truncated": bool(result.get("truncated", False)),\n        "model_id": MODEL_ID,\n        "model_revision": MODEL_REVISION,\n    }\n\n\ndef _build_reheaded_classifier(\n    model: Any,\n    class_names: Sequence[str],\n    seed: int = 42,\n) -> tuple[Any, list[str]]:\n    """Build a re-headed classifier without mutating the live model."""\n    import copy\n\n    import torch\n    import torch.nn as nn\n\n    names = [str(name).strip() for name in class_names]\n    if len(names) < 2 or any(not name for name in names):\n        raise ValueError("class_names must contain at least 2 non-empty names")\n    if len(set(names)) != len(names):\n        raise ValueError("class_names must be unique")\n    classifier = copy.deepcopy(model.classifier)\n    previous_dense = classifier.dense\n    with torch.random.fork_rng(devices=[]):\n        torch.manual_seed(seed)\n        new_dense = nn.Linear(previous_dense.in_features, len(names), bias=True)\n        nn.init.kaiming_normal_(new_dense.weight, nonlinearity="linear")\n        nn.init.zeros_(new_dense.bias)\n    new_dense = new_dense.to(device=previous_dense.weight.device, dtype=previous_dense.weight.dtype)\n    classifier.dense = new_dense\n    return classifier, names\n\n\ndef _set_class_metadata(model: Any, names: Sequence[str]) -> None:\n    """Update model label metadata after a classifier replacement succeeds."""\n    model.num_labels = len(names)\n    model.config.num_labels = len(names)\n    model.config.id2label = dict(enumerate(names))\n    model.config.label2id = {name: i for i, name in enumerate(names)}\n    model.config.problem_type = "single_label_classification"\n\n\ndef rehead_model(\n    model: Any,\n    class_names: Sequence[str],\n    seed: int = 42,\n) -> None:\n    """Dynamically replace ASTMLPHead dense layer with a new linear classifier sized to class_names."""\n    classifier, names = _build_reheaded_classifier(model, class_names, seed=seed)\n    model.classifier = classifier\n    _set_class_metadata(model, names)\n\n\ndef freeze_backbone(model: Any) -> int:\n    """Freeze all parameters in the audio spectrogram transformer backbone, returning frozen count."""\n    frozen = 0\n    for param in model.audio_spectrogram_transformer.parameters():\n        param.requires_grad = False\n        frozen += param.numel()\n    return frozen\n\n\n@dataclass\nclass ASTAudioClassificationPipeline:\n    """Audio Spectrogram Transformer classifier for AudioSet and adapted audio tasks."""\n\n    _runner: Callable[[np.ndarray], np.ndarray]\n    labels: list[str]\n    device: str\n    model: Any | None = None\n    extractor: Any | None = None\n    activation: str = ACTIVATION\n    adaptation_config: dict[str, Any] = field(default_factory=dict)\n\n    @classmethod\n    def from_pretrained(\n        cls,\n        device: str | None = None,\n        weights_dir: str | Path | None = None,\n        allow_download: bool = False,\n    ) -> ASTAudioClassificationPipeline:\n        root = Path(weights_dir) if weights_dir is not None else DEFAULT_WEIGHTS_DIR\n        if (root / MANIFEST_NAME).is_file():\n            stage_missing_files(root, allow_download=allow_download)\n            verify_snapshot(root)\n            source, kwargs = str(root), dict(local_files_only=True)\n        elif allow_download:\n            source, kwargs = MODEL_ID, dict(revision=MODEL_REVISION)\n        else:\n            raise FileNotFoundError(f"no verified snapshot at {root} and allow_download=False")\n        # Refuse invalid snapshots before importing model libraries.\n        import torch\n        from transformers import ASTFeatureExtractor, ASTForAudioClassification\n\n        resolved_device = device or ("cuda:0" if torch.cuda.is_available() else "cpu")\n        extractor = ASTFeatureExtractor.from_pretrained(source, trust_remote_code=False, **kwargs)\n        model = ASTForAudioClassification.from_pretrained(\n            source, trust_remote_code=False, dtype=torch.float32, **kwargs\n        )\n        model = model.to(resolved_device).eval()\n        labels = [model.config.id2label[i] for i in range(model.config.num_labels)]\n\n        def runner(waveform: np.ndarray) -> np.ndarray:\n            inputs = extractor(waveform, sampling_rate=SAMPLE_RATE, return_tensors="pt")\n            with torch.inference_mode():\n                logits = model(inputs["input_values"].to(resolved_device)).logits\n            return logits[0].float().cpu().numpy()\n\n        return cls(runner, labels, resolved_device, model=model, extractor=extractor)\n\n    def rehead(self, class_names: Sequence[str], seed: int = 42) -> None:\n        """Dynamically rehead the underlying model and update labels and runner."""\n        if self.model is None or self.extractor is None:\n            raise RuntimeError("cannot rehead a pipeline instance without an underlying torch model")\n        rehead_model(self.model, class_names, seed=seed)\n        self.labels = [self.model.config.id2label[i] for i in range(self.model.config.num_labels)]\n        self.activation = "softmax"\n        self.adaptation_config = {\n            "mode": "classifier-head-gradient-adaptation",\n            "class_names": list(self.labels),\n            "rehead_seed": seed,\n        }\n        self.model.to(self.device)\n        self._refresh_runner()\n\n    def _refresh_runner(self) -> None:\n        """Bind inference to the current model, extractor, and device."""\n        if self.model is None or self.extractor is None:\n            raise RuntimeError("cannot bind a runner without an underlying model and extractor")\n        resolved_device = self.device\n        model = self.model\n        extractor = self.extractor\n\n        def runner(waveform: np.ndarray) -> np.ndarray:\n            import torch\n\n            inputs = extractor(waveform, sampling_rate=SAMPLE_RATE, return_tensors="pt")\n            with torch.inference_mode():\n                logits = model(inputs["input_values"].to(resolved_device)).logits\n            return logits[0].float().cpu().numpy()\n\n        self._runner = runner\n\n    def freeze_backbone(self) -> int:\n        """Freeze the underlying backbone parameters."""\n        if self.model is None:\n            raise RuntimeError(\n                "cannot freeze backbone on a pipeline instance without an underlying torch model"\n            )\n        frozen = freeze_backbone(self.model)\n        self.adaptation_config["frozen_backbone_parameters"] = frozen\n        return frozen\n\n    def finetune(\n        self,\n        train_records: Sequence[dict[str, Any]],\n        val_records: Sequence[dict[str, Any]] | None = None,\n        *,\n        epochs: int = 3,\n        batch_size: int = 4,\n        learning_rate: float = 1e-4,\n        seed: int = 42,\n    ) -> list[dict[str, Any]]:\n        """In-process supervised fine-tuning using AdamW optimizer over trainable classifier parameters."""\n        import random\n\n        import torch\n        import torch.nn.functional as F\n        from torch.optim import AdamW\n\n        from .metrics import evaluate_classification\n        from .samples import validate_dataset\n\n        if self.model is None or self.extractor is None:\n            raise RuntimeError("cannot fine-tune a pipeline without underlying model and extractor")\n        if isinstance(epochs, bool) or not isinstance(epochs, int) or epochs < 1:\n            raise ValueError("epochs must be a positive int")\n        if isinstance(batch_size, bool) or not isinstance(batch_size, int) or batch_size < 1:\n            raise ValueError("batch_size must be a positive int")\n        valid_learning_rate = isinstance(learning_rate, int | float) and not isinstance(learning_rate, bool)\n        if not valid_learning_rate or learning_rate <= 0:\n            raise ValueError("learning_rate must be positive")\n        validate_dataset(train_records, self.labels)\n        if val_records is not None:\n            validate_dataset(val_records, self.labels)\n        if any(param.requires_grad for param in self.model.audio_spectrogram_transformer.parameters()):\n            raise RuntimeError("backbone is trainable; call freeze_backbone() before finetune()")\n\n        torch.manual_seed(seed)\n        device = torch.device(self.device)\n        self.model.to(device)\n\n        trainable_params = [p for p in self.model.parameters() if p.requires_grad]\n        if not trainable_params:\n            raise RuntimeError("model has no trainable parameters")\n        optimizer = AdamW(trainable_params, lr=learning_rate)\n\n        def cached_features(\n            records: Sequence[dict[str, Any]],\n        ) -> tuple[torch.Tensor, torch.Tensor]:\n            feature_batches: list[torch.Tensor] = []\n            self.model.audio_spectrogram_transformer.eval()\n            with torch.inference_mode():\n                for start in range(0, len(records), batch_size):\n                    batch = records[start : start + batch_size]\n                    inputs = self.extractor(\n                        [record["waveform"] for record in batch],\n                        sampling_rate=SAMPLE_RATE,\n                        return_tensors="pt",\n                    )\n                    backbone_output = self.model.audio_spectrogram_transformer(\n                        inputs["input_values"].to(device)\n                    )\n                    feature_batches.append(backbone_output.pooler_output.detach())\n            targets = torch.tensor(\n                [record["label"] for record in records],\n                dtype=torch.long,\n                device=device,\n            )\n            return torch.cat(feature_batches), targets\n\n        train_features, train_targets = cached_features(train_records)\n        val_features: torch.Tensor | None = None\n        val_targets: torch.Tensor | None = None\n        if val_records:\n            val_features, val_targets = cached_features(val_records)\n\n        history: list[dict[str, Any]] = []\n        n_samples = len(train_records)\n\n        for epoch in range(1, epochs + 1):\n            self.model.classifier.train()\n            indices = list(range(n_samples))\n            rng = random.Random(seed + epoch * 13)\n            rng.shuffle(indices)\n\n            running_loss = 0.0\n            steps = 0\n\n            for i in range(0, n_samples, batch_size):\n                batch_indices = indices[i : i + batch_size]\n                index_tensor = torch.tensor(batch_indices, dtype=torch.long, device=device)\n\n                optimizer.zero_grad(set_to_none=True)\n                logits = self.model.classifier(train_features.index_select(0, index_tensor))\n                loss = F.cross_entropy(logits, train_targets.index_select(0, index_tensor))\n                loss.backward()\n                optimizer.step()\n\n                running_loss += float(loss.item())\n                steps += 1\n\n            epoch_train_loss = running_loss / steps if steps > 0 else 0.0\n            epoch_data: dict[str, Any] = {\n                "epoch": epoch,\n                "train_loss": round(float(epoch_train_loss), 4),\n                "steps": steps,\n            }\n\n            if val_features is not None and val_targets is not None:\n                self.model.classifier.eval()\n                with torch.inference_mode():\n                    predictions = self.model.classifier(val_features).argmax(dim=-1)\n                eval_metrics = evaluate_classification(\n                    predictions.tolist(), val_targets.tolist(), self.labels\n                )\n                epoch_data["val_accuracy"] = eval_metrics["accuracy"]\n                epoch_data["val_macro_f1"] = eval_metrics["macro_f1"]\n\n            history.append(epoch_data)\n\n        self.model.eval()\n        self.adaptation_config.update(\n            {\n                "epochs": epochs,\n                "batch_size": batch_size,\n                "learning_rate": float(learning_rate),\n                "training_seed": seed,\n                "train_records": len(train_records),\n                "validation_records": len(val_records or []),\n                "feature_cache": "frozen-backbone-pooler-output",\n                "history": history,\n            }\n        )\n        return history\n\n    def evaluate(self, records: Sequence[dict[str, Any]]) -> dict[str, Any]:\n        """Evaluate the pipeline on a sequence of labelled audio records."""\n        from .metrics import evaluate_classification\n        from .samples import validate_dataset\n\n        validate_dataset(records, self.labels)\n        if self.model is not None:\n            self.model.eval()\n        predictions: list[int] = []\n        targets: list[int] = []\n\n        for record in records:\n            waveform = record["waveform"]\n            sample_rate = record.get("sample_rate", SAMPLE_RATE)\n            result = self.predict(waveform, sample_rate=sample_rate, top_k=1)\n            pred_idx = result["predictions"][0]["index"]\n            predictions.append(pred_idx)\n            targets.append(record["label"])\n\n        return evaluate_classification(predictions, targets, self.labels)\n\n    def save_artifact(self, output_path: str | Path) -> Path:\n        """Export adapted weights, vocabulary, base model lineage, and artifact metadata."""\n        import torch\n\n        if self.model is None:\n            raise RuntimeError("cannot save artifact without an underlying torch model")\n\n        out = Path(output_path)\n        out.parent.mkdir(parents=True, exist_ok=True)\n\n        if self.activation != "softmax" or len(self.labels) == NUM_LABELS:\n            raise RuntimeError("artifact export requires an adapted single-label classifier")\n\n        payload = {\n            "format": ARTIFACT_FORMAT,\n            "format_version": ARTIFACT_FORMAT_VERSION,\n            "artifact_kind": "classifier-head-adapter",\n            "base_model_id": MODEL_ID,\n            "base_model_revision": MODEL_REVISION,\n            "class_names": list(self.labels),\n            "num_classes": len(self.labels),\n            "activation": self.activation,\n            "adaptation": dict(self.adaptation_config),\n            "classifier_state_dict": self.model.classifier.state_dict(),\n        }\n        torch.save(payload, out)\n        return out\n\n    def load_artifact(self, artifact_path: str | Path) -> None:\n        """Load an adapted artifact into the existing pipeline safely using weights_only=True."""\n        import torch\n\n        if self.model is None or self.extractor is None:\n            raise RuntimeError("cannot load artifact into a pipeline without an underlying torch model")\n\n        payload = torch.load(artifact_path, map_location=self.device, weights_only=True)\n        if not isinstance(payload, dict):\n            raise ValueError(f"expected artifact dict, got {type(payload).__name__}")\n        if payload.get("format") != ARTIFACT_FORMAT:\n            raise ValueError(f"unrecognized artifact format: {payload.get(\'format\')}")\n        if payload.get("format_version") != ARTIFACT_FORMAT_VERSION:\n            raise ValueError(f"unsupported artifact format_version: {payload.get(\'format_version\')}")\n        if payload.get("artifact_kind") != "classifier-head-adapter":\n            raise ValueError(f"unsupported artifact_kind: {payload.get(\'artifact_kind\')}")\n        if payload.get("base_model_id") != MODEL_ID:\n            raise ValueError(f"base_model_id mismatch: {payload.get(\'base_model_id\')} != {MODEL_ID}")\n        if payload.get("base_model_revision") != MODEL_REVISION:\n            raise ValueError(\n                f"base_model_revision mismatch: {payload.get(\'base_model_revision\')} != {MODEL_REVISION}"\n            )\n        if payload.get("activation") != "softmax":\n            raise ValueError(f"unsupported artifact activation: {payload.get(\'activation\')}")\n\n        class_names = payload.get("class_names")\n        if not isinstance(class_names, list) or payload.get("num_classes") != len(class_names):\n            raise ValueError("artifact class_names and num_classes are inconsistent")\n        if any(not isinstance(name, str) or not name.strip() for name in class_names):\n            raise ValueError("artifact class_names must contain non-empty strings")\n        if len(class_names) < 2 or len(set(class_names)) != len(class_names):\n            raise ValueError("artifact class_names must contain at least 2 unique names")\n        classifier_state = payload.get("classifier_state_dict")\n        if not isinstance(classifier_state, dict):\n            raise ValueError("artifact classifier_state_dict must be a dict")\n        adaptation = payload.get("adaptation", {})\n        if not isinstance(adaptation, dict):\n            raise ValueError("artifact adaptation metadata must be a dict")\n\n        # Build and validate off to the side so a rejected artifact cannot leave\n        # the live pipeline with a partially replaced classifier.\n        classifier, normalized_names = _build_reheaded_classifier(self.model, class_names)\n        classifier.load_state_dict(classifier_state, strict=True)\n\n        self.model.classifier = classifier\n        _set_class_metadata(self.model, normalized_names)\n        self.labels = normalized_names\n        self.activation = "softmax"\n        self.adaptation_config = dict(adaptation)\n        self.model.to(self.device).eval()\n        self._refresh_runner()\n\n    @classmethod\n    def from_artifact(\n        cls,\n        artifact_path: str | Path,\n        weights_dir: str | Path | None = None,\n        device: str | None = None,\n        allow_download: bool = False,\n    ) -> ASTAudioClassificationPipeline:\n        """Instantiate a base pipeline and load an adapted artifact."""\n        pipe = cls.from_pretrained(device=device, weights_dir=weights_dir, allow_download=allow_download)\n        pipe.load_artifact(artifact_path)\n        return pipe\n\n    def predict(\n        self,\n        audio: np.ndarray,\n        sample_rate: int,\n        top_k: int | None = None,\n    ) -> dict[str, Any]:\n        """Classify one clip. `audio` is a 1-D float array; `sample_rate` is the rate it was captured at."""\n        resolved_top_k = min(DEFAULT_TOP_K, len(self.labels)) if top_k is None else top_k\n        duration = _check_inputs(audio, sample_rate, resolved_top_k)\n        if resolved_top_k > len(self.labels):\n            raise ValueError(\n                f"top_k={resolved_top_k} exceeds the active label count ({len(self.labels)})"\n            )\n        waveform = audio.astype(np.float32, copy=False)\n        resampled = sample_rate != SAMPLE_RATE\n        if resampled:\n            waveform = _resample(waveform, sample_rate)\n        logits = np.asarray(self._runner(waveform), dtype=np.float32)\n        if logits.shape != (len(self.labels),):\n            raise RuntimeError(f"backend returned logits {logits.shape}, expected ({len(self.labels)},)")\n        if self.activation == "sigmoid":\n            scores = 1.0 / (1.0 + np.exp(-logits))\n        elif self.activation == "softmax":\n            shifted = logits - np.max(logits)\n            exp_scores = np.exp(shifted)\n            scores = exp_scores / np.sum(exp_scores)\n        else:\n            raise RuntimeError(f"unsupported activation: {self.activation}")\n        order = np.argsort(-scores)[:resolved_top_k]\n        return {\n            "predictions": [\n                {"label": self.labels[int(i)], "index": int(i), "score": float(scores[i])} for i in order\n            ],\n            "activation": self.activation,\n            "truncated": duration > MAX_AUDIO_SECONDS,\n            "duration_seconds": duration,\n            "window_seconds": MAX_AUDIO_SECONDS,\n            "resampled": resampled,\n            "input_sample_rate": sample_rate,\n            "model_id": MODEL_ID,\n            "model_revision": MODEL_REVISION,\n        }\n', 'tutorial_stages.py': '"""Stage runner for the standalone AST acoustic-ecology tutorial (NOTEBOOK_SPEC 2.2 §25.13 isolated-environment pattern).\n\nThe tutorial notebook carries this file verbatim (as ``tutorial_stages.py`` in its run directory, beside the carried\npackage under ``src/``) and runs every stage with the interpreter of an isolated, hash-locked environment::\n\n    python -u tutorial_stages.py --root RUN_DIR --weights WEIGHTS_DIR --stage dataset [--zip PATH]\n\nNothing is installed into the notebook kernel, so a hosted runtime\'s preloaded packages are never replaced and no\nrestart is needed. Each stage is a separate process and starts from files only: the verified snapshot under\n``--weights``, the dataset written by ``dataset``, the adapter artifact written by ``finetune`` and the JSON records of\nearlier stages. Learner-facing exports go to ``RUN_DIR/outputs``; hand-off state goes to ``RUN_DIR/state``. On failure\na stage writes ``RUN_DIR/state/<stage>.error.json`` with the exception type and message, which the notebook re-raises\nin the kernel.\n\nStages: weights → runtime → pretrained → dataset → baseline → finetune → evaluate → reload → bundle, plus the optional\n``activity``. The package API does the work; this runner only sequences it and adds the tutorial\'s own data hygiene\n(per-record variation of the generated clips, duplicate refusal, archive reading for BYOD).\n"""\n# ruff: noqa: E501  -- the printed dictionaries are the learner-facing output; they are kept on one line each\nfrom __future__ import annotations\n\nimport argparse\nimport csv\nimport hashlib\nimport io\nimport json\nimport platform\nimport re\nimport sys\nimport time\nimport traceback\nimport wave\nimport zipfile\nfrom pathlib import Path, PurePosixPath\nfrom typing import Any\n\nSTEM = "ast_audio_classification"\nSEED = 42\nN_RECORDS = 24\nVAL_FRACTION = 0.25\nTOP_K_BASE = 5\nUNSEEN_INDEX = 99\nUNSEEN_CLASS = 1  # biophony\nADAPTER = "ast-audio-adapter-v1.pt"\nLOCK = "requirements.txt"\nSNAPSHOT_KEY = "ast-audioset"\nPARITY_RTOL = 1e-5\nPARITY_ATOL = 1e-6\nZIP_MAX_BYTES = 64 * 1024**2\nZIP_MAX_EXPANDED = 256 * 1024**2\nZIP_MAX_FILES = 100\nSCORE_SEMANTICS = "uncalibrated softmax scores over the target classes; the predicted label is the argmax"\nDECISION_RULE = "argmax: the class with the highest score is the prediction; no threshold is applied and no abstention is possible"\n\n\n# --------------------------------------------------------------------------------------------------\n# run context and small helpers\n# --------------------------------------------------------------------------------------------------\n\n\nclass Run:\n    """Paths of one run: carried sources and state under ``root``, the snapshot under ``weights``."""\n\n    def __init__(self, root: Path, weights: Path, options: argparse.Namespace) -> None:\n        self.root = root\n        self.weights = weights\n        self.options = options\n        self.out = root / "outputs"\n        self.state = root / "state"\n        self.out.mkdir(parents=True, exist_ok=True)\n        self.state.mkdir(parents=True, exist_ok=True)\n\n    @property\n    def snapshot(self) -> Path:\n        return self.weights / SNAPSHOT_KEY\n\n    def write_state(self, name: str, value: Any) -> Path:\n        path = self.state / name\n        path.write_text(json.dumps(value, indent=2), encoding="utf-8")\n        return path\n\n    def read_state(self, name: str, needed_by: str) -> Any:\n        path = self.state / name\n        if not path.is_file():\n            raise RuntimeError(f"{name} is missing: run the stage that writes it before \'{needed_by}\' (run the notebook from the top)")\n        return json.loads(path.read_text(encoding="utf-8"))\n\n    def write_output(self, name: str, value: Any) -> Path:\n        path = self.out / name\n        path.write_text(json.dumps(value, indent=2, ensure_ascii=False), encoding="utf-8")\n        return path\n\n\ndef sha256_array(waveform: Any) -> str:\n    import numpy as np\n\n    return hashlib.sha256(np.ascontiguousarray(waveform, dtype=np.float32).tobytes()).hexdigest()\n\n\ndef sha256_file(path: Path) -> str:\n    return hashlib.sha256(path.read_bytes()).hexdigest()\n\n\ndef lock_versions(text: str) -> dict[str, str]:\n    return {m.group(1).lower(): m.group(2) for m in re.finditer(r"^([A-Za-z0-9._-]+)==([^\\s\\\\]+)", text, re.M)}\n\n\n# --------------------------------------------------------------------------------------------------\n# data: per-record variation, PCM WAV decoding, the BYOD archive reader, duplicate refusal, baselines\n# --------------------------------------------------------------------------------------------------\n\n\ndef vary_waveform(waveform: Any, rng: Any) -> Any:\n    """Seeded per-record variation of a generated clip, so no two records are the same waveform (AST-M2).\n\n    Speed perturbation (every frequency scaled by 0.92–1.08), a circular time shift, a gain of 0.55–0.95 of full scale and\n    a white-noise floor 30–45 dB below the signal. These are standard audio augmentations; the class character of the\n    clip (broadband noise, chirps, low drone) is unchanged.\n    """\n    import numpy as np\n\n    x = np.asarray(waveform, dtype=np.float64)\n    n = x.size\n    factor = rng.uniform(0.92, 1.08)\n    offset = rng.uniform(0, n)\n    positions = (np.arange(n) * factor + offset) % n\n    stretched = np.interp(positions, np.arange(n + 1), np.append(x, x[0]))\n    peak = float(np.max(np.abs(stretched))) or 1.0\n    signal = stretched / peak * rng.uniform(0.55, 0.95)\n    rms = float(np.sqrt(np.mean(signal**2))) or 1e-3\n    noise = rng.standard_normal(n) * rms * 10 ** (-rng.uniform(30.0, 45.0) / 20.0)\n    return np.clip(signal + noise, -1.0, 1.0).astype(np.float32)\n\n\ndef default_dataset(seed: int = SEED, n_records: int = N_RECORDS) -> list[dict[str, Any]]:\n    """The package\'s balanced generated dataset, each clip given its own seeded variation."""\n    import numpy as np\n\n    from ast_audio_classification_pipeline import synthetic_audio_dataset\n\n    records = synthetic_audio_dataset(n_samples=n_records, seed=seed)\n    for index, record in enumerate(records):\n        record["waveform"] = vary_waveform(record["waveform"], np.random.default_rng([seed, index]))\n    return records\n\n\ndef unseen_clip(seed: int = SEED) -> Any:\n    """A newly generated biophony clip that is not in the dataset, varied the same way."""\n    import numpy as np\n\n    from ast_audio_classification_pipeline import SAMPLE_RATE, generate_audio_clip\n\n    clip = generate_audio_clip(index=UNSEEN_INDEX, class_idx=UNSEEN_CLASS, duration=3.0, sample_rate=SAMPLE_RATE, seed=seed)\n    return vary_waveform(clip, np.random.default_rng([seed, UNSEEN_INDEX]))\n\n\ndef decode_pcm_wav(payload: bytes, name: str = "WAV") -> tuple[Any, int]:\n    """Decode an uncompressed PCM WAV to mono float32 in [-1, 1] and its sample rate."""\n    import numpy as np\n\n    try:\n        with wave.open(io.BytesIO(payload), "rb") as handle:\n            channels = handle.getnchannels()\n            sample_width = handle.getsampwidth()\n            rate = handle.getframerate()\n            frames = handle.getnframes()\n            compression = handle.getcomptype()\n            raw = handle.readframes(frames)\n    except (wave.Error, EOFError) as exc:\n        raise ValueError(f"{name}: not a readable PCM WAV ({exc}); export it as 16-bit PCM WAV") from exc\n    if compression != "NONE":\n        raise ValueError(f"{name}: compressed WAV ({compression}) is unsupported; export it as 16-bit PCM WAV")\n    if sample_width == 1:\n        samples = (np.frombuffer(raw, dtype=np.uint8).astype(np.float32) - 128.0) / 128.0\n    elif sample_width == 2:\n        samples = np.frombuffer(raw, dtype="<i2").astype(np.float32) / 32768.0\n    elif sample_width == 4:\n        samples = np.frombuffer(raw, dtype="<i4").astype(np.float32) / 2147483648.0\n    else:\n        raise ValueError(f"{name}: {8 * sample_width}-bit PCM WAV is unsupported; convert it to 16-bit PCM")\n    if channels < 1 or samples.size % channels:\n        raise ValueError(f"{name}: invalid WAV channel layout")\n    mono = samples.reshape(-1, channels).mean(axis=1) if channels > 1 else samples\n    return mono.astype(np.float32), rate\n\n\ndef _archive_layout(names: list[str], classes: tuple[str, ...]) -> tuple[str | None, list[str]]:\n    """The single parent folder to strip (``mydata/geophony/a.wav``), or None; and the skipped metadata entries."""\n    skipped = [n for n in names if n.startswith("__MACOSX/") or any(part.startswith(".") for part in PurePosixPath(n).parts)]\n    kept = [n for n in names if n not in skipped]\n    firsts = {PurePosixPath(n).parts[0] for n in kept if PurePosixPath(n).parts}\n    if len(firsts) == 1 and next(iter(firsts)) not in classes:\n        parent = next(iter(firsts))\n        if all(len(PurePosixPath(n).parts) >= 2 and PurePosixPath(n).parts[1] in classes for n in kept if n.rstrip("/") != parent):\n            return parent, skipped\n    return None, skipped\n\n\ndef load_zip_dataset(path: Path, classes: tuple[str, ...]) -> tuple[list[dict[str, Any]], dict[str, Any]]:\n    """Read a labelled BYOD ZIP in memory (never extracted): ``<class>/<clip>.wav`` for every class.\n\n    Accepts one enclosing folder (what "compress folder" produces) and skips ``__MACOSX/`` and dot-file metadata,\n    reporting both. Every refusal names the member and the corrective action (AST-m3).\n    """\n    from ast_audio_classification_pipeline import (\n        MAX_DATASET_AUDIO_SECONDS,\n        MIN_DATASET_AUDIO_SECONDS,\n        SAMPLE_RATE,\n    )\n\n    expected = "zip the three class folders " + ", ".join(f"{c}/" for c in classes) + " so they sit at the top level of the archive (or inside one folder)"\n    if path.suffix.lower() != ".zip":\n        raise ValueError(f"{path.name}: the dataset must be one .zip file")\n    if path.stat().st_size > ZIP_MAX_BYTES:\n        raise ValueError(f"{path.name}: the ZIP is larger than 64 MiB compressed; use fewer or shorter clips")\n    records: list[dict[str, Any]] = []\n    with zipfile.ZipFile(path) as archive:\n        infos = archive.infolist()\n        parent, skipped = _archive_layout([i.filename for i in infos], classes)\n        members = [i for i in infos if i.filename not in skipped and not i.is_dir()]\n        if not members or len(members) > ZIP_MAX_FILES:\n            raise ValueError(f"{path.name}: the ZIP must contain 1–{ZIP_MAX_FILES} WAV files, found {len(members)}")\n        if sum(i.file_size for i in members) > ZIP_MAX_EXPANDED:\n            raise ValueError(f"{path.name}: the ZIP expands beyond 256 MiB; use fewer or shorter clips")\n        for info in members:\n            name = info.filename\n            parts = PurePosixPath(name).parts\n            if PurePosixPath(name).is_absolute() or ".." in parts:\n                raise ValueError(f"{name}: unsafe member path (absolute or \'..\'); {expected}")\n            if parent is not None:\n                parts = parts[1:]\n            if len(parts) != 2:\n                raise ValueError(f"{name}: expected <class>/<clip>.wav; {expected}")\n            class_name = parts[0]\n            if class_name not in classes:\n                raise ValueError(f"{name}: unknown class folder {class_name!r}; the folders must be exactly {list(classes)}")\n            if info.flag_bits & 0x1 or not name.lower().endswith(".wav"):\n                raise ValueError(f"{name}: every member must be an unencrypted PCM .wav file")\n            waveform, rate = decode_pcm_wav(archive.read(info), name)\n            if rate != SAMPLE_RATE:\n                raise ValueError(f"{name}: sample rate {rate} Hz; resample the clip to {SAMPLE_RATE} Hz (16 kHz) PCM WAV")\n            seconds = waveform.size / rate\n            if seconds > MAX_DATASET_AUDIO_SECONDS:\n                raise ValueError(f"{name}: duration {seconds:.2f} s is longer than {MAX_DATASET_AUDIO_SECONDS} s; trim the clip to at most {MAX_DATASET_AUDIO_SECONDS} s")\n            if seconds < MIN_DATASET_AUDIO_SECONDS:\n                raise ValueError(f"{name}: duration {seconds:.3f} s is shorter than {MIN_DATASET_AUDIO_SECONDS} s; use a longer clip")\n            records.append({"id": name, "waveform": waveform, "sample_rate": rate, "label": classes.index(class_name), "class_name": class_name})\n    present = {r["class_name"] for r in records}\n    missing = [c for c in classes if c not in present]\n    if missing:\n        raise ValueError(f"{path.name}: no clips for {missing}; {expected}")\n    return records, {"archive": path.name, "archive_sha256": sha256_file(path), "stripped_parent_folder": parent, "skipped_metadata_entries": skipped}\n\n\ndef refuse_duplicates(records: list[dict[str, Any]]) -> dict[str, list[str]]:\n    """Refuse any waveform that occurs more than once: a copy could land in training and evaluation (AST-M2)."""\n    by_digest: dict[str, list[str]] = {}\n    for record in records:\n        by_digest.setdefault(sha256_array(record["waveform"]), []).append(record["id"])\n    duplicates = {d: ids for d, ids in by_digest.items() if len(ids) > 1}\n    if duplicates:\n        groups = "; ".join(" = ".join(ids) for ids in duplicates.values())\n        raise ValueError(f"identical waveforms found ({groups}); remove the copies so no clip can be both trained on and evaluated")\n    return by_digest\n\n\ndef training_majority_baseline(train_labels: list[int], eval_labels: list[int], class_names: list[str]) -> dict[str, Any]:\n    """A trivial predictor fitted on the TRAINING labels only, scored on the evaluation labels (AST-m2)."""\n    from ast_audio_classification_pipeline import majority_class_baseline\n\n    fitted = majority_class_baseline(train_labels, len(class_names))\n    predicted = fitted["majority_class_index"]\n    correct = sum(1 for label in eval_labels if label == predicted)\n    return {\n        "rule": "always predict the most frequent class in the training split",\n        "fitted_on": "train",\n        "predicted_class": class_names[predicted],\n        "predicted_class_index": predicted,\n        "accuracy": round(correct / len(eval_labels), 4),\n        "n_evaluation_clips": len(eval_labels),\n    }\n\n\ndef add_noise_at_snr(waveform: Any, snr_db: float, rng: Any) -> Any:\n    import numpy as np\n\n    rms = float(np.sqrt(np.mean(np.square(waveform, dtype=np.float64)))) or 1e-6\n    noisy = waveform + rng.standard_normal(waveform.size) * rms * 10 ** (-snr_db / 20.0)\n    return np.clip(noisy, -1.0, 1.0).astype(np.float32)\n\n\n# --------------------------------------------------------------------------------------------------\n# model factories (the CPU pre-flight test replaces these with a small randomly initialised stand-in)\n# --------------------------------------------------------------------------------------------------\n\n\ndef load_pipeline(run: Run) -> Any:\n    from ast_audio_classification_pipeline import ASTAudioClassificationPipeline\n\n    return ASTAudioClassificationPipeline.from_pretrained(weights_dir=run.snapshot)\n\n\ndef load_from_artifact(run: Run, artifact: Path) -> Any:\n    from ast_audio_classification_pipeline import ASTAudioClassificationPipeline\n\n    return ASTAudioClassificationPipeline.from_artifact(artifact, weights_dir=run.snapshot)\n\n\ndef load_data(run: Run, stage: str) -> tuple[list[dict[str, Any]], list[dict[str, Any]], dict[str, Any]]:\n    """The train and evaluation records the `dataset` stage wrote, checked against their recorded digests."""\n    import numpy as np\n\n    split = run.read_state("split.json", stage)\n    arrays = np.load(run.state / "dataset.npz")\n    by_id = {}\n    for i, rid in enumerate(split["ids"]):\n        waveform = arrays[f"w{i}"].astype(np.float32)\n        if sha256_array(waveform) != split["digests"][i]:\n            raise RuntimeError(f"the dataset changed after the dataset stage ({rid}); run the notebook from Section 6")\n        by_id[rid] = {"id": rid, "waveform": waveform, "sample_rate": split["sample_rate"], "label": split["labels"][i], "class_name": split["class_names"][split["labels"][i]]}\n    train = [by_id[rid] for rid in split["train_ids"]]\n    evaluation = [by_id[rid] for rid in split["eval_ids"]]\n    return train, evaluation, split\n\n\ndef score_rows(pipe: Any, records: list[dict[str, Any]], clip: Any) -> dict[str, Any]:\n    """Full score vectors for every evaluation clip and the unseen clip (the reload-parity reference)."""\n    from ast_audio_classification_pipeline import SAMPLE_RATE\n\n    k = len(pipe.labels)\n    rows = {r["id"]: pipe.predict(r["waveform"], sample_rate=r["sample_rate"], top_k=k)["predictions"] for r in records}\n    rows["unseen"] = pipe.predict(clip, sample_rate=SAMPLE_RATE, top_k=k)["predictions"]\n    return {rid: {p["label"]: p["score"] for p in preds} for rid, preds in rows.items()}\n\n\n# --------------------------------------------------------------------------------------------------\n# stages\n# --------------------------------------------------------------------------------------------------\n\n\ndef stage_weights(run: Run) -> None:\n    """Section 3: install the carried manifest, fetch the absent files at the pinned revision, verify every file."""\n    from ast_audio_classification_pipeline import (\n        MODEL_ID,\n        MODEL_LICENSE,\n        MODEL_REVISION,\n        stage_missing_files,\n        verify_snapshot,\n    )\n    from ast_audio_classification_pipeline.pipeline import MANIFEST_NAME\n\n    carried = run.root / "weights" / SNAPSHOT_KEY / MANIFEST_NAME\n    manifest = json.loads(carried.read_text(encoding="utf-8"))\n    if (manifest["modelId"], manifest["revision"]) != (MODEL_ID, MODEL_REVISION):\n        raise RuntimeError("the carried manifest does not name the identity in the carried pipeline module; regenerate the notebook")\n    run.snapshot.mkdir(parents=True, exist_ok=True)\n    (run.snapshot / MANIFEST_NAME).write_text(json.dumps(manifest, indent=2), encoding="utf-8")\n    print({"model_id": MODEL_ID, "revision": MODEL_REVISION, "license": MODEL_LICENSE, "files": len(manifest["files"]), "total_bytes": manifest["totalBytes"]}, flush=True)\n    fetched = stage_missing_files(run.snapshot, allow_download=True)\n    verified = verify_snapshot(run.snapshot)\n    files = verified.get("files", []) if isinstance(verified, dict) else []\n    record = {"weights_dir": str(run.snapshot), "fetched": fetched, "verified_files": len(files) if isinstance(files, list) else files, "revision": MODEL_REVISION}\n    run.write_output("weights.json", record)\n    print(record)\n\n\ndef stage_runtime(run: Run) -> None:\n    """Section 4: the isolated environment\'s versions, checked against the carried lock; device and input ceilings."""\n    import torch\n    import torchaudio\n    import transformers\n\n    from ast_audio_classification_pipeline import (\n        MAX_AUDIO_SECONDS,\n        MAX_INPUT_SECONDS,\n        MIN_AUDIO_SECONDS,\n        NUM_LABELS,\n        SAMPLE_RATE,\n    )\n\n    locked = lock_versions((run.root / LOCK).read_text(encoding="utf-8"))\n    installed = {"torch": torch.__version__, "torchaudio": torchaudio.__version__, "transformers": transformers.__version__}\n    mismatched = {name: {"locked": locked.get(name), "installed": version} for name, version in installed.items() if version.split("+", 1)[0] != locked.get(name)}\n    if mismatched:\n        raise RuntimeError(f"installed versions differ from the carried lock: {mismatched}; rebuild the isolated environment from Section 2")\n    record = {\n        "python": platform.python_version(),\n        **installed,\n        "versions_match_lock": True,\n        "device": "cuda:0" if torch.cuda.is_available() else "cpu",\n        "ceilings": {"SAMPLE_RATE": SAMPLE_RATE, "MIN_AUDIO_SECONDS": MIN_AUDIO_SECONDS, "MAX_AUDIO_SECONDS": MAX_AUDIO_SECONDS, "MAX_INPUT_SECONDS": MAX_INPUT_SECONDS, "NUM_LABELS": NUM_LABELS},\n    }\n    run.write_output("runtime.json", record)\n    print(record)\n\n\ndef stage_pretrained(run: Run) -> None:\n    """Section 5: the unchanged 527-label AudioSet head on the tutorial tone (or one BYOD WAV)."""\n    from ast_audio_classification_pipeline import SAMPLE_RATE, tutorial_tone, validate_inputs\n    from ast_audio_classification_pipeline.samples import TONE_AMPLITUDE, TONE_HZ\n\n    if run.options.wav:\n        path = Path(run.options.wav)\n        audio, sample_rate = decode_pcm_wav(path.read_bytes(), path.name)\n        clip_name, sample_kind = path.name, "BYOD"\n    else:\n        sample_rate = SAMPLE_RATE\n        audio = tutorial_tone(duration=3.0, sample_rate=sample_rate, frequency=TONE_HZ, amplitude=TONE_AMPLITUDE)\n        clip_name, sample_kind = f"synthetic_sine_{int(TONE_HZ)}hz_3s.wav", "synthetic"\n    manifest = validate_inputs(audio, sample_rate, top_k=TOP_K_BASE, names=[clip_name])\n    run.write_state("input_manifest.json", manifest)\n    pipe = load_pipeline(run)\n    result = pipe.predict(audio, sample_rate=sample_rate, top_k=TOP_K_BASE)\n    record = {"sample_kind": sample_kind, "name": clip_name, "samples": int(audio.shape[0]), "sample_rate": sample_rate, "sha256": sha256_array(audio), "device": pipe.device, "result": result}\n    run.write_output(f"{STEM}_pretrained.json", record)\n    print({"sample_kind": sample_kind, "name": clip_name, "samples": int(audio.shape[0]), "sample_rate": sample_rate, "sha256": record["sha256"][:16] + "...", "activation": result["activation"], "truncated": result["truncated"]})\n    for rank, item in enumerate(result["predictions"], start=1):\n        print(f"  {rank:>2}. index {item[\'index\']:>3}  score {item[\'score\']:.4f}  {item[\'label\']}")\n\n\ndef stage_dataset(run: Run) -> None:\n    """Sections 6–7: build or read the dataset, validate it, refuse duplicates, split it, record the over-long probe."""\n    import numpy as np\n\n    from ast_audio_classification_pipeline import (\n        ADAPT_CLASSES,\n        MAX_INPUT_SECONDS,\n        SAMPLE_RATE,\n        split_dataset,\n        validate_dataset,\n        validate_inputs,\n    )\n\n    source: dict[str, Any] = {}\n    if run.options.zip:\n        records, source = load_zip_dataset(Path(run.options.zip), ADAPT_CLASSES)\n        kind = "BYOD"\n    else:\n        records, kind = default_dataset(), "synthetic"\n        source = {"generator": "synthetic_audio_dataset(n_samples=24, seed=42) + per-record variation (vary_waveform, seed [42, index])"}\n    summary = validate_dataset(records, ADAPT_CLASSES)\n    thin = [name for name, count in summary["class_counts"].items() if count < 2]\n    if thin:\n        raise ValueError(f"every class needs at least two clips for a disjoint split; add clips for {thin}")\n    digests = refuse_duplicates(records)\n    train, evaluation = split_dataset(records, val_fraction=VAL_FRACTION, seed=SEED)\n    train_digests = {sha256_array(r["waveform"]) for r in train}\n    overlap = sorted(r["id"] for r in evaluation if sha256_array(r["waveform"]) in train_digests)\n    if overlap:\n        raise RuntimeError(f"evaluation clips identical to training clips: {overlap}")\n    np.savez(run.state / "dataset.npz", **{f"w{i}": r["waveform"] for i, r in enumerate(records)})\n    run.write_state("split.json", {\n        "dataset_kind": kind, "sample_rate": SAMPLE_RATE, "class_names": list(ADAPT_CLASSES),\n        "ids": [r["id"] for r in records], "labels": [r["label"] for r in records], "digests": [sha256_array(r["waveform"]) for r in records],\n        "train_ids": [r["id"] for r in train], "eval_ids": [r["id"] for r in evaluation],\n    })\n    manifest = run.read_state("input_manifest.json", "dataset")\n    manifest.update({"dataset_kind": kind, "dataset_source": source, "dataset_representation": summary["representation"], "dataset_records": summary["n_records"], "class_counts": summary["class_counts"], "distinct_waveforms": len(digests)})\n    try:\n        validate_inputs(np.zeros(int((MAX_INPUT_SECONDS + 1) * SAMPLE_RATE), dtype=np.float32), SAMPLE_RATE)\n    except ValueError as exc:\n        manifest["findings"].append({"input": "over-long-probe", "verdict": "rejected", "message": str(exc)})\n    run.write_output(f"{STEM}_input_manifest.json", manifest)\n    counts = {\n        "train": {c: sum(r["class_name"] == c for r in train) for c in ADAPT_CLASSES},\n        "evaluation": {c: sum(r["class_name"] == c for r in evaluation) for c in ADAPT_CLASSES},\n    }\n    split_record = {\n        "dataset_kind": kind, "records": len(records), "distinct_waveforms": len(digests), "class_counts": summary["class_counts"],\n        "split": "stratified, seeded (seed 42), 75% train / 25% held-out evaluation", "train_clips": len(train), "evaluation_clips": len(evaluation),\n        "counts": counts, "train_evaluation_duplicates": len(overlap), "eval_ids": [r["id"] for r in evaluation],\n    }\n    run.write_output(f"{STEM}_split.json", split_record)\n    print({"dataset_kind": kind, "records": len(records), "distinct_waveforms": len(digests), "class_counts": summary["class_counts"], "verdict": summary["verdict"], **({"archive": source} if kind == "BYOD" else {})})\n    print({"refusal_probe": manifest["findings"][-1] if manifest["findings"] else None})\n\n\ndef stage_baseline(run: Run) -> None:\n    """Section 8: re-head, freeze, and score the untrained head and the training-majority rule on the evaluation clips."""\n    train, evaluation, split = load_data(run, "baseline")\n    pipe = load_pipeline(run)\n    pipe.rehead(split["class_names"], seed=SEED)\n    frozen = pipe.freeze_backbone()\n    trainable = sum(p.numel() for p in pipe.model.parameters() if p.requires_grad)\n    pre = pipe.evaluate(evaluation)\n    pre.pop("baseline", None)\n    pre.pop("accuracy_delta_vs_baseline", None)\n    baseline = training_majority_baseline([r["label"] for r in train], [r["label"] for r in evaluation], split["class_names"])\n    record = {"stage": "reheaded_pre_adaptation", "classes": pipe.labels, "frozen_backbone_parameters": frozen, "trainable_parameters": trainable, "untrained_head": pre, "majority_baseline": baseline}\n    run.write_output(f"{STEM}_pre_adaptation.json", record)\n    print({"classes": pipe.labels, "frozen_backbone_parameters": frozen, "trainable_parameters": trainable})\n    print({"untrained_head_accuracy": pre["accuracy"], "untrained_head_macro_f1": pre["macro_f1"], "majority_baseline": f"always \'{baseline[\'predicted_class\']}\' (fitted on train) -> accuracy {baseline[\'accuracy\']}"})\n\n\ndef stage_finetune(run: Run) -> None:\n    """Section 9: bounded head fine-tune on cached frozen features; record the in-memory scores; export the adapter."""\n    train, evaluation, split = load_data(run, "finetune")\n    opts = run.options\n    pipe = load_pipeline(run)\n    pipe.rehead(split["class_names"], seed=SEED)\n    pipe.freeze_backbone()\n    history = pipe.finetune(train_records=train, val_records=evaluation, epochs=opts.epochs, batch_size=opts.batch_size, learning_rate=opts.learning_rate, seed=SEED)\n    for row in history:\n        row["evaluation_accuracy"] = row.pop("val_accuracy", None)\n        row["evaluation_macro_f1"] = row.pop("val_macro_f1", None)\n    run.write_state("in_memory_scores.json", score_rows(pipe, evaluation, unseen_clip()))\n    artifact = pipe.save_artifact(run.out / ADAPTER)\n    run.write_output(f"{STEM}_training_history.json", {"epochs": opts.epochs, "batch_size": opts.batch_size, "learning_rate": opts.learning_rate, "seed": SEED, "history": history})\n    run.write_state("adapter.json", {"path": str(artifact), "bytes": artifact.stat().st_size, "sha256": sha256_file(artifact)})\n    for row in history:\n        print(f"Epoch {row[\'epoch\']}/{opts.epochs}: train_loss={row[\'train_loss\']:.4f}  evaluation_accuracy={row[\'evaluation_accuracy\']:.4f}  steps={row[\'steps\']}")\n    print({"adapter": artifact.name, "bytes": artifact.stat().st_size, "sha256": sha256_file(artifact)[:16] + "...", "contains": "classifier head only (the frozen backbone is rebuilt from the pinned base revision)"})\n\n\ndef stage_evaluate(run: Run) -> None:\n    """Sections 10–11: a fresh process loads the exported adapter, scores the evaluation clips and the unseen clip."""\n    from ast_audio_classification_pipeline import ARTIFACT_FORMAT, MODEL_ID, MODEL_REVISION, SAMPLE_RATE\n\n    train, evaluation, split = load_data(run, "evaluate")\n    adapter = run.read_state("adapter.json", "evaluate")\n    pipe = load_from_artifact(run, Path(adapter["path"]))\n    report = pipe.evaluate(evaluation)\n    report.pop("baseline", None)\n    report.pop("accuracy_delta_vs_baseline", None)\n    baseline = training_majority_baseline([r["label"] for r in train], [r["label"] for r in evaluation], split["class_names"])\n    payload = {\n        "task": "multiclass acoustic ecology classification",\n        "score_semantics": SCORE_SEMANTICS,\n        "decision_rule": DECISION_RULE,\n        "calibration": "none: the scores are not calibrated probabilities; calibration and any threshold are the responsibility of a downstream user, on their own data",\n        "sample_kind": f"{split[\'dataset_kind\']}_heldout_evaluation",\n        "n_clips": len(evaluation),\n        "classes": split["class_names"],\n        "metrics": report,\n        "baseline": baseline,\n        "accuracy_delta_vs_baseline": round(report["accuracy"] - baseline["accuracy"], 4),\n        "split_assumption": "random stratified split; assumes clips are independent (no shared recording or source across clips)",\n        "verdict": "sample-sanity",\n        "reason": f"{len(evaluation)} held-out clip(s) evaluated against a training-majority baseline; not a benchmark claim",\n        "adapter_format": ARTIFACT_FORMAT,\n        "model_id": MODEL_ID,\n        "model_revision": MODEL_REVISION,\n    }\n    run.write_output(f"{STEM}_evaluation_report.json", payload)\n    clip = unseen_clip()\n    result = pipe.predict(clip, sample_rate=SAMPLE_RATE, top_k=len(pipe.labels))\n    unseen = {"clip": "unseen_synthetic_biophony_99", "true_class": split["class_names"][UNSEEN_CLASS], "sha256": sha256_array(clip), "in_dataset": sha256_array(clip) in set(split["digests"]), "score_semantics": SCORE_SEMANTICS, "decision_rule": DECISION_RULE, "result": result}\n    run.write_output(f"{STEM}_unseen_prediction.json", unseen)\n    with (run.out / f"{STEM}_top_k.csv").open("w", encoding="utf-8", newline="") as handle:\n        writer = csv.writer(handle)\n        writer.writerow(["clip", "rank", "index", "label", "score"])\n        for rank, item in enumerate(result["predictions"], start=1):\n            writer.writerow(["unseen_test_clip_99", rank, item["index"], item["label"], f"{item[\'score\']:.6f}"])\n    print({"evaluation_clips": len(evaluation), "accuracy": report["accuracy"], "macro_f1": report["macro_f1"], "majority_baseline_accuracy": baseline["accuracy"], "delta_vs_baseline": payload["accuracy_delta_vs_baseline"], "verdict": "sample-sanity"})\n    print("Per class (precision / recall / F1 / support):")\n    for name, values in report["per_class"].items():\n        print(f"  {name:<12} {values[\'precision\']:.4f} / {values[\'recall\']:.4f} / {values[\'f1\']:.4f} / {values[\'support\']}")\n    print("Confusion matrix (rows = true class, columns = predicted class):", split["class_names"])\n    for name, row in zip(split["class_names"], report["confusion_matrix"], strict=True):\n        print(f"  {name:<12} {row}")\n\n\ndef stage_reload(run: Run) -> None:\n    """Section 12: another fresh process rebuilds the pipeline from the adapter and compares every score with training."""\n    import numpy as np\n\n    _train, evaluation, _split = load_data(run, "reload")\n    reference = run.read_state("in_memory_scores.json", "reload")\n    adapter = run.read_state("adapter.json", "reload")\n    if sha256_file(Path(adapter["path"])) != adapter["sha256"]:\n        raise RuntimeError("the adapter file changed after it was exported; rerun from Section 9")\n    reloaded = load_from_artifact(run, Path(adapter["path"]))\n    scores = score_rows(reloaded, evaluation, unseen_clip())\n    labels_equal = all(max(scores[k], key=scores[k].get) == max(reference[k], key=reference[k].get) for k in reference)\n    deltas = [abs(scores[k][c] - reference[k][c]) for k in reference for c in reference[k]]\n    close = all(np.isclose(scores[k][c], reference[k][c], rtol=PARITY_RTOL, atol=PARITY_ATOL) for k in reference for c in reference[k])\n    record = {"clips_compared": len(reference), "scores_compared": len(deltas), "max_abs_score_difference": float(max(deltas)), "predicted_labels_identical": labels_equal, "rtol": PARITY_RTOL, "atol": PARITY_ATOL, "labels": reloaded.labels, "reload_verification": "PASSED" if (close and labels_equal) else "FAILED"}\n    run.write_output(f"{STEM}_reload_parity.json", record)\n    print(record)\n    if not (close and labels_equal):\n        raise RuntimeError("the reloaded adapter does not reproduce the trained model\'s scores; do not use the export")\n\n\ndef stage_bundle(run: Run) -> None:\n    """Section 13: one provenance bundle that links every record of this run."""\n    from ast_audio_classification_pipeline import MODEL_ID, MODEL_LICENSE, MODEL_REVISION\n\n    def load(name: str) -> Any:\n        path = run.out / name\n        if not path.is_file():\n            raise RuntimeError(f"{name} is missing: run the notebook from the top before the bundle")\n        return json.loads(path.read_text(encoding="utf-8"))\n\n    source = json.loads((run.root / "source.json").read_text(encoding="utf-8"))\n    payload = {\n        "evaluation_report": load(f"{STEM}_evaluation_report.json"),\n        "pre_adaptation": load(f"{STEM}_pre_adaptation.json"),\n        "input_manifest": load(f"{STEM}_input_manifest.json"),\n        "split": load(f"{STEM}_split.json"),\n        "test_prediction": load(f"{STEM}_unseen_prediction.json"),\n        "training_history": load(f"{STEM}_training_history.json"),\n        "reload_parity": load(f"{STEM}_reload_parity.json"),\n        "notebook_source": source,\n        "repository_revision": source["revision"],\n        "model_id": MODEL_ID,\n        "model_revision": MODEL_REVISION,\n        "model_license": MODEL_LICENSE,\n        "runtime": load("runtime.json"),\n        "environment": "isolated hash-locked uv environment; nothing installed into the notebook kernel",\n    }\n    payload["files"] = {p.name: sha256_file(p) for p in sorted(run.out.iterdir()) if p.is_file() and p.name != f"{STEM}_result.json"}\n    run.write_output(f"{STEM}_result.json", payload)\n    print({"outputs_directory": str(run.out), "files": sorted(p.name for p in run.out.iterdir() if p.is_file())})\n\n\ndef stage_activity(run: Run) -> None:\n    """Section 14 (optional): add seeded noise at one SNR to the evaluation clips and score them with the same adapter."""\n    import numpy as np\n\n    _train, evaluation, split = load_data(run, "activity")\n    adapter = run.read_state("adapter.json", "activity")\n    canonical = {p.name: sha256_file(p) for p in sorted(run.out.iterdir()) if p.is_file()}\n    pipe = load_from_artifact(run, Path(adapter["path"]))\n    snr = float(run.options.snr_db)\n    clean = pipe.evaluate(evaluation)\n    noisy_records = [{**r, "waveform": add_noise_at_snr(r["waveform"], snr, np.random.default_rng([SEED, 7, i]))} for i, r in enumerate(evaluation)]\n    noisy = pipe.evaluate(noisy_records)\n    changed = sum(1 for a, b in zip(clean["confusion_matrix"], noisy["confusion_matrix"], strict=True) if a != b)\n    record = {"changed_variable": "evaluation-clip noise level", "snr_db": snr, "clean": {"accuracy": clean["accuracy"], "macro_f1": clean["macro_f1"], "confusion_matrix": clean["confusion_matrix"]}, "noisy": {"accuracy": noisy["accuracy"], "macro_f1": noisy["macro_f1"], "confusion_matrix": noisy["confusion_matrix"]}, "confusion_rows_changed": changed, "classes": split["class_names"]}\n    out = run.out / "activity"\n    out.mkdir(exist_ok=True)\n    tag = f"{snr:g}".replace("-", "minus").replace(".", "p")\n    (out / f"{STEM}_activity_snr_{tag}db.json").write_text(json.dumps(record, indent=2), encoding="utf-8")\n    after = {p.name: sha256_file(p) for p in sorted(run.out.iterdir()) if p.is_file()}\n    if after != canonical:\n        raise RuntimeError("the activity changed a canonical output; it must write only under outputs/activity/")\n    print(record)\n\n\nSTAGES = {\n    "weights": stage_weights,\n    "runtime": stage_runtime,\n    "pretrained": stage_pretrained,\n    "dataset": stage_dataset,\n    "baseline": stage_baseline,\n    "finetune": stage_finetune,\n    "evaluate": stage_evaluate,\n    "reload": stage_reload,\n    "bundle": stage_bundle,\n    "activity": stage_activity,\n}\n\n\ndef parse_args(argv: list[str] | None = None) -> argparse.Namespace:\n    parser = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)\n    parser.add_argument("--root", type=Path, required=True, help="run directory holding the carried sources")\n    parser.add_argument("--weights", type=Path, required=True, help="directory holding the pinned snapshot")\n    parser.add_argument("--stage", choices=sorted(STAGES), required=True)\n    parser.add_argument("--wav", default="", help="pretrained: one BYOD PCM WAV instead of the tutorial tone")\n    parser.add_argument("--zip", default="", help="dataset: a labelled BYOD ZIP instead of the generated dataset")\n    parser.add_argument("--epochs", type=int, default=5, help="finetune: training epochs")\n    parser.add_argument("--batch-size", type=int, default=4, help="finetune: clips per optimiser step")\n    parser.add_argument("--learning-rate", type=float, default=1e-3, help="finetune: AdamW learning rate")\n    parser.add_argument("--snr-db", type=float, default=0.0, help="activity: signal-to-noise ratio of the added noise, in dB")\n    return parser.parse_args(argv)\n\n\ndef main(argv: list[str] | None = None) -> int:\n    options = parse_args(argv)\n    root = options.root.resolve()\n    carried_src = root / "src"\n    if carried_src.is_dir() and str(carried_src) not in sys.path:\n        sys.path.insert(0, str(carried_src))\n    run = Run(root, options.weights.resolve(), options)\n    error_file = run.state / f"{options.stage}.error.json"\n    error_file.unlink(missing_ok=True)\n    started = time.perf_counter()\n    try:\n        STAGES[options.stage](run)\n    except Exception as exc:  # the notebook re-raises this message in the kernel\n        traceback.print_exc()\n        message = str(exc) or repr(exc)\n        error_file.write_text(json.dumps({"stage": options.stage, "type": type(exc).__name__, "message": message}), encoding="utf-8")\n        print(f"STAGE FAILED ({options.stage}): {type(exc).__name__}: {message}", flush=True)\n        return 2\n    print({"stage": options.stage, "status": "ok", "seconds": round(time.perf_counter() - started, 1)}, flush=True)\n    return 0\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n', 'requirements.txt': '# This file was autogenerated by uv via the following command:\n#    uv pip compile pyproject.toml --python-version 3.12 --python-platform x86_64-manylinux_2_28 --generate-hashes --only-binary :all: -o tutorials/requirements-colab.lock.txt\ncertifi==2026.7.22 \\\n    --hash=sha256:62f22742b58a1a33014a2b6b706588a8d7e2a88ae7bd1a6ebe8c992928483775 \\\n    --hash=sha256:741e2c3b351ddf169a738da9f2c048608ff7f2c5cc02f1ebc6b118bb090d5d55\n    # via requests\ncharset-normalizer==3.5.2 \\\n    --hash=sha256:01077390b03f7988f11d700a2194e69b119741a86b1a638b1db88891e3eced8e \\\n    --hash=sha256:01b0c0d2262a9e28e8484a278c7e1b5d650e3ac8cf2683d2967e25899f208bdf \\\n    --hash=sha256:04851f73ae72b8413dddadb16a49dfee95263553741fd42d546f7d66907e6be5 \\\n    --hash=sha256:0521c5665880b33d603717defa76c094048900010897909952397feb3039da56 \\\n    --hash=sha256:0774bf9bf620249fee3e0b8b9fd3065de213be30f3aa94ce2494b3b638949e26 \\\n    --hash=sha256:0891b9d3903c5571c03771ca669a4b0ec5618ca722a5c957d3d29cd4e5062848 \\\n    --hash=sha256:0c951d5e6dd9c2ff60609476752bee49da4206adde960ebc247766937f72e718 \\\n    --hash=sha256:0fed1d06615f022ee3b13caf5e8b180cfea32bb2c5aded8a9d44277afc040f93 \\\n    --hash=sha256:114e4d0c92d618409ed82a99e22b5c5e768fe995f2973f78265f4524f49d4640 \\\n    --hash=sha256:11912e4bb14baae7c5d8791aa55ba0a3a03ec6729073307b0f57270abaa713d3 \\\n    --hash=sha256:11a4d68a6ecda3292cb1e50239e111543ba5d709bb62a6b4ea1afcfa729d8875 \\\n    --hash=sha256:124fbf1a8ff966d87ae05bb8bd45a71f966055ed8bba320d0c7cf450bc5f4d0e \\\n    --hash=sha256:1461ac396c4fdb983a675f20aa555624f0ee18ac83d832b9244ffff3d8055275 \\\n    --hash=sha256:1503bccbeb36d5527790c3930327704c39af22de3112f1b1666a9f3ce15ee204 \\\n    --hash=sha256:15bb4005af6320d259dc7593ca84a38d7fe06a421dbcf7b910ae23979101e787 \\\n    --hash=sha256:15c44f7edfd477b06f517a5cc317fc1707edb9de2c865f43d4b6513907473234 \\\n    --hash=sha256:16fa0eccf81304b79c5cd87f9271c3b85dd9dd99245e4422ae9c0dd45e0f99d3 \\\n    --hash=sha256:183b88127acdb4fabe59d951ab424faf1af7b63cdbb5f776186c1ea2ffcaed98 \\\n    --hash=sha256:195c26fb65950f8fce54e26349852b7bdd7c5f120aeefbcc440b8a20faaed4a3 \\\n    --hash=sha256:1afb975bd5d68d5ce9f6b6d44fdf2f7e34b895a35e95708a7a91b20a3b51d187 \\\n    --hash=sha256:1b4cbc7c3491ccb4aa17fcd8165649d01cf39f76de1696da8631b5f71b85401d \\\n    --hash=sha256:1bc0baf5ef96b6ede57d47f4b8fe4d9d84019c3bfcbeb20a41edc6a6ee341f1f \\\n    --hash=sha256:1c50fe28bbc2ced33386f298650d91218076c05420e6cbd790b913adc41659e7 \\\n    --hash=sha256:1db38f4c5496827c1a501846d64d14c3b80c7e6714e406cd7dc36a9899fa1011 \\\n    --hash=sha256:211d5a3eb6af8f513b8d4ca19a8c1b7accab1b5f0d3175f9826b03c1a920dc1f \\\n    --hash=sha256:23851fb4e1b85ed3f6c2a27b777cdfe2e19fb5b38429a8faf38c7542b7665869 \\\n    --hash=sha256:254eb48b9fa5ee9898a3c445825a1f340fe53712a098904b39b0bddba8ea3cb1 \\\n    --hash=sha256:2625388c6c754520c37abaf3b41eb34d1cc4a373f457898f08606c8e362b891d \\\n    --hash=sha256:281cb91036248400f4cc957495cccd44c275c2e0c5854f7e45ac5cf7dc193847 \\\n    --hash=sha256:28a15fdad492a99b6eccfaaed66ef3f74050680545ea61ec8b2f4c538f1f1320 \\\n    --hash=sha256:28b4f0d66fb834ff90f28209ac7bce77868c45d8c93e26f906709d9b7c2e1af9 \\\n    --hash=sha256:2a925889534b3748302dae5dead07cc13480de1dac3aea80a941b729b471ef93 \\\n    --hash=sha256:2b7b3bbfb4fe8ef40600792d762fbaa9057559f9d3fad209525b7a22b99e91fd \\\n    --hash=sha256:2c9ad19a6cfcd5ea5c0d41161d22f9df1dcc277e9bef2751391334546a314c00 \\\n    --hash=sha256:2cc961b171b3f3440f410489ab3573e86aea8736134ebbb40ea1338b7f0831bc \\\n    --hash=sha256:2ce45c6627b22c47e390bc91a41c3d13032192e699fa0bea96e9671b373d69b0 \\\n    --hash=sha256:2e06a3a98f916dd41d27f3105e02e7a40181c98c94b9158733d03a6f80506c09 \\\n    --hash=sha256:304d5463e65a35d7bb0850550e0780395395f6fcf452f04db7d5ca7cecc425ac \\\n    --hash=sha256:304d8e4d493af723536393eee0c689eb7813f4a474c8b479dee63f1fdd98f621 \\\n    --hash=sha256:30fcd120b732aa79317f08dee04d7de0847822e4cf7ee0e9f445bb958832252c \\\n    --hash=sha256:31f3930700408d211f13378ccbe1c40845d8da54bd0681fac3a9b5aae81c7aa8 \\\n    --hash=sha256:34276fd796040bf0993ab33a369aa572e6979c7aab225a88893667ad8eac8f7a \\\n    --hash=sha256:355ad8011081dec5412240c087a9a0c9d4d5039f3ed11a3f13e18c2b29b56c51 \\\n    --hash=sha256:38a873987f3be698494da8b2e3085e29da02da7b633dce73e79c699a113d7bf0 \\\n    --hash=sha256:39de2a259fc954455c57274dc94c79d5842774e1247a016aff30bc0efed0f4ef \\\n    --hash=sha256:3d14b50de6bf4d0edf857a9386836846f982b8f524e188e2e68b96d702bcf4aa \\\n    --hash=sha256:3d21b8b13c7592db2ac5e544a6d83187b995257472b0c9e8351b6d507ae37ed6 \\\n    --hash=sha256:3d31298449090ab8d47b7b1b2a555ff73cac7ed438a08b7ac160980c7ebed649 \\\n    --hash=sha256:3ddacd27458c45bdacd6bd6db644bfb730efbf9e830310186e3045c9c5be8fb2 \\\n    --hash=sha256:3df041de8887954562c9b261cba85ca0e9ded74048daf125f45edcfaa4832229 \\\n    --hash=sha256:40ab6bffa02ae10a0581e6c198be7d2d8ca5c2a0c64e4ed3465d766df457573e \\\n    --hash=sha256:4275811936e2f06feff5e598fb42a1b7ae852da8e39605211892b56b81a34efd \\\n    --hash=sha256:443eae2bf318abeaf6f15d785138f71fd6de770e99a92158b8b814265e079115 \\\n    --hash=sha256:447441e76ec720b15e64418d32e092297340387053047c7c694f579efb0ee1d9 \\\n    --hash=sha256:4495c5002a7b28557e7e222e77e0b661183e432b7d6d2e788101e3f240e05b8c \\\n    --hash=sha256:44bd4fbb29dfbeba60e7d2bd000c59e4b21ddb3cc53912b14048d37092706d7c \\\n    --hash=sha256:4685902cf26edf013ed7a3da0f426ebba7a00ebb9541386d835afbf002c11cab \\\n    --hash=sha256:498dc3188ca05a68231ac3fdbfc7f57eb67e1343c30e0fea17f8218c1599b253 \\\n    --hash=sha256:4c2b5031f63e331e3839b40aed2dd6f191e9c07edbde303e7876846ea1946995 \\\n    --hash=sha256:4d48f2d08b9de5864e2c8744d4461b862fb149a18274abc8b698c45975573438 \\\n    --hash=sha256:4f87960d57feabfb618e4e0af6e7371645fa26a277860739d6e5d6e0012c92f0 \\\n    --hash=sha256:50e3adfb96fc189eb27b1cf62d3b598b89b4bb0420d93a3d3e42e137409011be \\\n    --hash=sha256:51cf45226a9b588d0d2b4880c62d686934b63ab0bd79ca23ab0e9762eb27441b \\\n    --hash=sha256:52aa6992700996af31f375de0c6bacd402b0097fe40b53c426b9f51a90ebabc7 \\\n    --hash=sha256:55ea99acb17b9325618de155a0cd6a2e8f5d10be008113e1d433bbb58db543b2 \\\n    --hash=sha256:56bc200a365efb37383b7852e4cc5898d3b2da5987289b543956cf8cad71018a \\\n    --hash=sha256:588461c2e8384d309bd63e5826019b6977bc66d629b99ac8737bb795d7b2cb5a \\\n    --hash=sha256:58ca3755ee7ff7f59b57789ec9833c9de9ea275405cdd240eda1f193112e398a \\\n    --hash=sha256:58f361dcbab699cf8f42db3f47c8e7fd1036f138c23a5d08de9fde5f425a730c \\\n    --hash=sha256:598a11a2c7ebaa5334bf698bf29568c9c390abac6a154d8170fedecd1cea38c5 \\\n    --hash=sha256:59f63901b0031c3136cf64704dcb21de0bbae62ce2c9529bc39d27665463de37 \\\n    --hash=sha256:5cde776b7cc66e4f6c99612cea4aa7269aa65863f7a15841b2c264f103822f4e \\\n    --hash=sha256:5e2b6b57e9733d39f0c9fd3185efa6b8e29652c4cd8fe94180272cf6ed9a78c4 \\\n    --hash=sha256:5fb29fb8cd1a46c27a1bf9613ad5ec2599310d46b4025d9556404a6b6a292800 \\\n    --hash=sha256:6045373d5a89a5ec71afde535db987ca28e76dfa276c2d4c818265b375d4b055 \\\n    --hash=sha256:619799369eeef6366ed3e8755a5670f4f2f0fb6b30a0fd7264dc0fdc2357058e \\\n    --hash=sha256:62588a277bfb59def052abd940703fa35107152bf479781a878617d60faf8fb5 \\\n    --hash=sha256:62603db9a7caa0802eaa28c1c46fecd7b3a263a774069c24c3c28c302448721c \\\n    --hash=sha256:65cd72beeeca9d3aaea1201e5923859f308f952f9c71de93f06063c79f0f7a3b \\\n    --hash=sha256:68eb192d85ab8e5f6ec69c2bc6ac0179fbf04a5ac1569d12fbef74883fe102d0 \\\n    --hash=sha256:6bd128f206a7752ae1f2ab6c61bf8a24ba28913a10df8b14c2637b973ff97a80 \\\n    --hash=sha256:6be488a102b8cf28d0391d8c4ba7748938ae28b78ad901f8585520fca33ead1a \\\n    --hash=sha256:7218e8f32b0956cfcd048fd42d9d5779809745ca1d86113ca56f66e7ae1549c4 \\\n    --hash=sha256:7441d755b7ab94f8d4eb3e43ec05482d760842fd263d003a99102d742cd835e2 \\\n    --hash=sha256:749e97e1b32313717a565abbe321bc2190bc8b35f1a67e4cdbc7c56c8d8ffe58 \\\n    --hash=sha256:75a3ceed0724d625d64b86ca20aba182e4df462e04c2414fc941c0f523f06aac \\\n    --hash=sha256:780fbe7cab297b81dad9fb8dc5eb003c0468ffb0d9e5f65068c53a34661a96bc \\\n    --hash=sha256:78456a747de8dc58360ffa581f30a002baf5aa28cb262536545e91f113ed7639 \\\n    --hash=sha256:7967d08cf06dee78443b874f98c98036f624f3a4e73e11f9f64f5be4d25393cf \\\n    --hash=sha256:7a881931aa470808df94a8c380eed2bbbc76cd9dc622310f99665658c821eb6d \\\n    --hash=sha256:7dcd882da75ef9adf94903b1e3b9419e8aa8fb4c7396822b834b9ef7fb96954f \\\n    --hash=sha256:7e841fb9010836c992c9f12fcbd43a831de93a5f726fc1ccd8ca1d0268c5014c \\\n    --hash=sha256:7fdde2c9fd9e3eca40631e024664cf2584272cc8f96308cbe5fdfc930f51d8bc \\\n    --hash=sha256:8024d00c3faf3fc0c16e07a69f4405e8eac7cc0ab15f65fe6cf43827c4cf72b4 \\\n    --hash=sha256:80d02b6f04e92601a081dd97b23d3128033098bff5d35d392ddcc0476ea11253 \\\n    --hash=sha256:838dcc90063569a0448120554591a1d6c4a4ffe11babf048908793154ab86ade \\\n    --hash=sha256:849df64e889b2e17230d58410a03dba311a65b163508fd33679b2b737d4b7858 \\\n    --hash=sha256:87475fabc8d9996fd9c27debb395e642e8c838d78a00b6e932227a0e06b81e26 \\\n    --hash=sha256:87e50a3e7cb90af586b6c5faf23e302a970415ac73bd7bd90a515a04b427ef96 \\\n    --hash=sha256:89b53f3cda69831909888e0494f4fa0bcd3537e3e138dabeb620bd6ad946bae8 \\\n    --hash=sha256:8a893cc101149f80a653f82062ebc95b34525a2614382e1da5458fe7c6997249 \\\n    --hash=sha256:8b2bfab86aa71ae13aa41a6a26aab338e0db2b8bc75434b05aea89e011ff35a4 \\\n    --hash=sha256:8d86d6fc60743dc916eb79e2eb1ec4818e21e427731543af40a3021851174a13 \\\n    --hash=sha256:915563965d418f986e7e145accc592eae9e1a1be3566ff98a05d7a9ec42a76e1 \\\n    --hash=sha256:92888bb3187c5ba50500b00b3b310c9f2c651709d28036077680cb5255450a03 \\\n    --hash=sha256:93223adc95033dd47133a46ccfc316a0139176fd79085762e27202ec56018f03 \\\n    --hash=sha256:9373ad13ef0d2c0fb761e04e55bfdee5a08b52cef2c882c8fbe9935b1517152e \\\n    --hash=sha256:9409a8bf35cf78353942504b24a57de3d75b708997a1e4bd8db71ac8633ce364 \\\n    --hash=sha256:9b7f416ff0978e2f2249330527f0ad6fa02f4932e6199692d3b52da2048c19e4 \\\n    --hash=sha256:9bde855991b7e362c146535e3136a50bfaffc0487d38b33ca7e5edefc6e23849 \\\n    --hash=sha256:9cae88599c7219005d879f98e5ed53341e9a122af585e1091200358a3003d2a0 \\\n    --hash=sha256:9cf9b1a857e25c4baceeb3624e92a56df3668f398c4acba74e174d81fb4d1d3a \\\n    --hash=sha256:9f56f72050826f63dcee7a7f55b0a77168cb3bfc553fd405e7f8f9ece75a4036 \\\n    --hash=sha256:a090bb2c68df85450502e3e20d665e3a5af9c65a84d6508ed477badd49166fd3 \\\n    --hash=sha256:a192e2c40070d92c3ccf777e3a5c4ff515573cd2bb7ed0c537fdadbbec5bbf21 \\\n    --hash=sha256:a19a731138fc27d5682277d3b9df22855cea1239bce7fcec5f78f42ef2d1f3c3 \\\n    --hash=sha256:a66c3bc5ab1f0ff2164fc9965ddd611ff0802173f4b9d24554c563f6ab7e1d6e \\\n    --hash=sha256:a815775b6c38d4e0ff7bcffbeba67feded90202bb6a226b8dd35f1c855217413 \\\n    --hash=sha256:a89012d6d5476ee112d20d998570ed58df2260a852afb1758809cd6900411d21 \\\n    --hash=sha256:ae4f5fea5b8b8ccff88238cc8569303e5ee95efae67fa62922a311397a71f346 \\\n    --hash=sha256:b6856554c4f44d79fc2307d5768854310a8f0096e501c75637542c82292b0429 \\\n    --hash=sha256:b6b751274acb69d77b3323d6b7dbaa3c7fdfc1eb829b7eb61d262f32e1af9685 \\\n    --hash=sha256:b736353c0a625bbd5fcec108576e2385db3496f4f771f785ff32e108d3c3bc45 \\\n    --hash=sha256:b7fd005a73d9e657273b7a10dc71a9e03c8fb9ee6999798d6918ce095b81ac7f \\\n    --hash=sha256:b91363207bd9dc966a691e959bb47f64b30f7ac4b072be9968b366982f7db77c \\\n    --hash=sha256:ba0b1d2620edf869789c3879223f52bf2afc5d31b3cb47cc57b3a12c05e2aa9d \\\n    --hash=sha256:bbbfc8e28816f19d7c0f1816664980c0a9875d01b27cdf8eedddb639d9e108ad \\\n    --hash=sha256:bd16aabe4a02a297c23417aa17ac6299dbd8c49f673bcd645b4929b11f5a4400 \\\n    --hash=sha256:c0afc6800ba57ccc350374c5bd6150419915d95ce93cdbab2d783d75eaf30ecb \\\n    --hash=sha256:c6708715abcf3c73b99508253e961a9967f02fe536532834149574eda6de0d1c \\\n    --hash=sha256:c7c9ab723cde841fefb34efbad91e87f00a674b1fe1cd0784fde742bf2c154dc \\\n    --hash=sha256:c8f3d67aeaf55f017982b73683f0e7342ba2f6635a78f69ce89ebb26aa411e5c \\\n    --hash=sha256:c9790464842f85f437dbbb54417eda1e0e6bfc52dd8d22d6fd1c994b73b2dc74 \\\n    --hash=sha256:ca403d7e4798f525fdfc78e258820419cbbd0f0ecbab9de7840e3c017cf6b8cf \\\n    --hash=sha256:d008d90a7f2471519aef0c90dfbe73b3e6e4d5e66ac48e19154c17e89e98b604 \\\n    --hash=sha256:d19fbd981a488e22cd04883659ca6b08f50b5974f9fd7c95655ef6a043e5893f \\\n    --hash=sha256:d1befeed746d247c81127bb14de9dc3d30edb6e5976d34f83f86ed262b1d9105 \\\n    --hash=sha256:d2374b62878abb00cd8309b32af6c0b715cd02dec0ca74ef12e5069bdc64144a \\\n    --hash=sha256:d376bbd28b3a8999db1a103b3b388aee6f1ddeb3e51bc2172993efdcd86e064d \\\n    --hash=sha256:d4a7319f304a774bed22115bc891618e45f85065ab44ea6acd07d274e750519a \\\n    --hash=sha256:d6734d2ef8a50fbf8445c139477da401f50d62a0606bf00e20ec6d87773fefb1 \\\n    --hash=sha256:d760fe2a4d7c3b226cb9026d6a842868d52a7901bd98420e1baf14e80da85cf5 \\\n    --hash=sha256:d913de495d90407cd859d263bee2e5d1a4ed3eb6573c04e70d9ec619a7cbed7f \\\n    --hash=sha256:db19d07e2e0129e974a0e65d0064fc222a446cd5122c2fd4184d2af9fc734a9e \\\n    --hash=sha256:dca9ab98072a5a54ebacebdc45f53e645336b320c667410b061be1ca588ae709 \\\n    --hash=sha256:ddc7dacc8ece3a182e7f15cb862d1fd616b46d076cb1ae9dd232b2c38b655874 \\\n    --hash=sha256:ddf19c062bea7a0cc80f519243d2c01dd091be0cf952a0750d4ad576709559f5 \\\n    --hash=sha256:def79fa35ef0cef8d2accec024f4fdc7ead3012ff02f5215c783f39f03ef8cfc \\\n    --hash=sha256:df29a0a7107f7011e77f4eebdddec4c7331e24d787a0b21a46d63bdf7445da95 \\\n    --hash=sha256:e09a3942ecbdee5cce73ea9d42da82b81b72ac1bf031ce069b93b5adf4eac8cd \\\n    --hash=sha256:e242bb1c5e76e97dfa9e7f209a71e93a01d7f19ffdd5cfbb2e2d55b4f08f8ab0 \\\n    --hash=sha256:e243bd13217235fc7290c621941c3f5cc8b66e4872495be821d7436ba2fb838d \\\n    --hash=sha256:e2af3aad578aa6bd1384bcf4750fc285e5a9de53f40b7d41e5a0bf748edeb2b3 \\\n    --hash=sha256:e4e81e09c1578b8df602e3db08b0b3ea0a6947ad612f52bf8dc5ea8d47691f0c \\\n    --hash=sha256:e54da4baf05720032d527874d40b65fa4d7e5c6c6a43d0c3adbeffcaf275a2b3 \\\n    --hash=sha256:e80e6c2f55656b4824d72065abb4ddd6a525c74bd78a0aab5d9fc2cf4fb5af50 \\\n    --hash=sha256:ed2a239c0ea213acc1908150a3037257083c7c083128f1a4cec2ec4b97dca491 \\\n    --hash=sha256:ed905975ab14056a2e5eb1c376cb2e1ebc5396baf84163939c518556fccde9f5 \\\n    --hash=sha256:ee21e28f0430bd6dc9086c6e525d5e818a44a5ad19720c8a0ef766792f3eb5e5 \\\n    --hash=sha256:ee43c17b173d46a3212baa6ead3ae258eeabdae48c263a01ccf0218c366dd655 \\\n    --hash=sha256:ef4fcbf3327382cd4c9f540babd61248208af7b93eec4de397b4d5f58a09e288 \\\n    --hash=sha256:eff0ac9dbe711a4aee69bf04a83896aa9b85f19641264053a9f6d48573abb7dd \\\n    --hash=sha256:f0aa869112ef88429ae17820d99c3dd9504c9e9c671d3c246f3d7442cb051084 \\\n    --hash=sha256:f3c96f633825733f735c5a9cf21d21a257d8e1edf0b1cee0a064b9c424ca0f7d \\\n    --hash=sha256:f5833ad231be5eb6553de524a70f48d71b2c8563101750531e0b80184e175cd4 \\\n    --hash=sha256:f5ec61164adcec446f8969a3358ec3f9b26bbda3b9213e5586d219afa8df2915 \\\n    --hash=sha256:f7d486c83842422badd511868fd8a9a20e9407ace71564b6af47ce7e60a336c1 \\\n    --hash=sha256:fb9e68df06293761f9fe66ade60a9bc6d0f5e42b8acf2939a9158af86ab0e5bd \\\n    --hash=sha256:fc14a032f813bf5fe624d991960ea83e9715adc27e4c1830a2361eb1d02ac341 \\\n    --hash=sha256:fcff63213e8e6e47770541a4607175404f47cbb3ebea7b6058cc82d524a0e424 \\\n    --hash=sha256:fd1fbe0f116b6e55da77aca2c6ddcddcfac2186cbf78bdebf40fc156efca389d \\\n    --hash=sha256:fe9753dfee015c570d73df76f899f18444d41388bffcde097deba51c4fadbb9f\n    # via requests\ncuda-bindings==13.4.3 \\\n    --hash=sha256:044c03b056dcc5cecfad426a071187dd9e1e6817fbb363bc2f3a7520170b3e67 \\\n    --hash=sha256:0deff5b22462bb410859684299b1fc6a621780c43a4729342aa6ec28783485b4 \\\n    --hash=sha256:130ff1daae550db2cef559ba477f3a63d040756bd135f5deba4b5bdc4e110252 \\\n    --hash=sha256:145cc9b02dfb7bcc3288701b6f19c69c590e4451c62916ba0a8a3db9a64a91a8 \\\n    --hash=sha256:1fd7d8459b364aedc11f3e59703453ced823135f78a9111ca70feef8d56d4d21 \\\n    --hash=sha256:3d7a6506e625be59cdc119ef4423afa0fc3a4830b129dc010cbd51324b93d66d \\\n    --hash=sha256:4796864ce829bd95ef2ef0d23c6ba21bb64e08f7fab0a377302ed1affb6605c7 \\\n    --hash=sha256:52d7f3f5f7f014dddddc66cd802ed0ddf65ae99ad42b5619fae7b82e5ddd6771 \\\n    --hash=sha256:6c6bb1a4c4f7520e062669c6f3605a7016b2e65ba51b8922f92edd91d5fadf8d \\\n    --hash=sha256:7e11cfe8fec4c85ce79feda18124971c52596f0cbd642a94f5dafc257124a4b3 \\\n    --hash=sha256:81eef62bbb95cb4a705fb423b2ad1c63d716edb352ee2af0c28bf8a29cc8258c \\\n    --hash=sha256:a9ea13b9cfe711515ae83b5efc99101af4d3f8ad882f99724a839c8b5417fb7e \\\n    --hash=sha256:b89d6e738494b7b95c38e3413f86d32c24682c8e714870531a5a2b193a2fc50a \\\n    --hash=sha256:bbacde6f75665b197016b986164cfdaa33b17515e5e635a63ddb75926aaa71c3 \\\n    --hash=sha256:bc51990309b416e0780a11a921ac597ef9c0e52e1bdf5ae0b8e8c4766b66442b \\\n    --hash=sha256:bfbd3f7d4ac04dd41dc49121b9e408c8283992f47124c2290ecb79bbbadcca8e \\\n    --hash=sha256:c2af7e69d2557fdb4e5fc1169159fd08da7e861aa84fb83307a1a0396b6b0973 \\\n    --hash=sha256:d5f72bcfcdf3be23e1da3c792f68f508586f48d037bca8b10f552c4cca5971f2 \\\n    --hash=sha256:d6eb969920e28f66f8fc3b0b3afcb6e09381cc96bf8e8158d774e9488ae89980 \\\n    --hash=sha256:d7c6c9f46fca7f3fc61959ef9a2398ac656172145b43f408e0a6492360cf1c0c \\\n    --hash=sha256:df8b3767facca8acde216460df684dfc3826d519096c13adfd3030a55870dc79 \\\n    --hash=sha256:e52f66340785a51b8b77f4487329de188f3cabbf37c62e68f410a8299e8677ae \\\n    --hash=sha256:f8519603001c92bf83e7095df3b8211e3999a9c4ded096b57de0f3ff52b66368\n    # via torch\ncuda-pathfinder==1.8.3 \\\n    --hash=sha256:e29e59829c297a7a5233bd9cc71094fc5bddbd076951482670178f9eade39b1f\n    # via cuda-bindings\ncuda-toolkit==13.0.3.0 \\\n    --hash=sha256:d693caaa261214ddd7dbb60d68e71cbed884e68c2be7509778f3051da0b91c3f\n    # via torch\nfilelock==4.0.9 \\\n    --hash=sha256:635e7d67fa92654eed444e75e9ca18426d34e77ad9c469bf4373f75a932f7b22 \\\n    --hash=sha256:9287fd61b99a806e5202be29a83034c0808a1b9830e820537c2f5773981f7eeb\n    # via\n    #   huggingface-hub\n    #   torch\n    #   transformers\nfsspec==2026.9.0 \\\n    --hash=sha256:0f08147951c8cb31d844c3547d631053b127863b60be04cf06e121333ee0e2fe \\\n    --hash=sha256:8dd6e646e99ea382bd85f97a45e6b526a442d79423a7dc673f1e2756d05fcb5f\n    # via\n    #   huggingface-hub\n    #   torch\nhf-xet==1.6.0 \\\n    --hash=sha256:0e6e21fa3cdfcdcd76748564bf593870a5e013f47d97cf10aed63aa222cff5b7 \\\n    --hash=sha256:23379c2f9ec8696d952b16414a2bae72cad86a52df869b050698ba60f538c675 \\\n    --hash=sha256:2e58454a340b3556dfa4972d5451aff4fba8dd42a236600ba1a1d2b1514f0fef \\\n    --hash=sha256:35cec30d75c6f9eb9c16a77cef68e85a103b72e24d4b473714ec9ff06428bab9 \\\n    --hash=sha256:3dc3e35441ba395006af5aaacc40ef2e603c51ef46c3530b9156185f00935ea3 \\\n    --hash=sha256:4fc74352a17015bd0ee90038bc9efe38db894cde45f268b6712b04fce8cd0acb \\\n    --hash=sha256:5153e6bb103ad49d6ea9f1b2e230db5a2ea32551ad09a706d2f61d7c7c80d80e \\\n    --hash=sha256:5789835d7c6bc9436962853192082374297fb72d7eff7e7762ec25ceb7e25338 \\\n    --hash=sha256:633dc0cd71d32da58ab8c03ad38e2fac452c15c2b0a2866ebf6ededfe0a5061d \\\n    --hash=sha256:70cbb9c896901600128cb9b6f06e132954fbede1db30f31f7c6c63f84cb7c31d \\\n    --hash=sha256:75765820ce4700db3750c94acc8fe27c5fae4c9ec000a0dbac3ca082acf97765 \\\n    --hash=sha256:8fb4f71cba6129110c3374a33f919001ff130488fc23553698e34cc1c2a1198c \\\n    --hash=sha256:948f15d3a9545cfe5932f6bd8b440f6ae630aee108f14b7bd6c561f7c2dcc522 \\\n    --hash=sha256:d62671bb130879cef0ee4c9ebe47a14af6c66ec53e6d84dc15936e5ffdfac82f \\\n    --hash=sha256:f0906082d9932ae0c0057fa194041c22b4e2cdb46b2592ef3b91f020d62a081a \\\n    --hash=sha256:f2f7278c05c22fd60cb436cda1269649b3e81db65ecdc8496e5e164aa4143e7b \\\n    --hash=sha256:fb4fadde1b2b70bf4c0c14a6dccbe7194b1c28947fefd5bbe3fed9d940676c3b\n    # via huggingface-hub\nhuggingface-hub==0.36.2 \\\n    --hash=sha256:1934304d2fb224f8afa3b87007d58501acfda9215b334eed53072dd5e815ff7a \\\n    --hash=sha256:48f0c8eac16145dfce371e9d2d7772854a4f591bcb56c9cf548accf531d54270\n    # via\n    #   ast-audio-classification-pipeline (pyproject.toml)\n    #   tokenizers\n    #   transformers\nidna==3.20 \\\n    --hash=sha256:a7db850025b95ded1eae8a46181a1a6c56c92c96f0e2b005d9ff8dc0210cab44 \\\n    --hash=sha256:ab7ae7122974553370f0bdb919e1a960b2cd1bc1ef0276416d896db81c14582c\n    # via requests\njinja2==3.1.6 \\\n    --hash=sha256:0137fb05990d35f1275a587e9aee6d56da821fc83491a0fb838183be43f66d6d \\\n    --hash=sha256:85ece4451f492d0c13c5dd7c13a64681a86afae63a5f347908daf103ce6d2f67\n    # via torch\nmarkupsafe==3.0.3 \\\n    --hash=sha256:0303439a41979d9e74d18ff5e2dd8c43ed6c6001fd40e5bf2e43f7bd9bbc523f \\\n    --hash=sha256:068f375c472b3e7acbe2d5318dea141359e6900156b5b2ba06a30b169086b91a \\\n    --hash=sha256:0bf2a864d67e76e5c9a34dc26ec616a66b9888e25e7b9460e1c76d3293bd9dbf \\\n    --hash=sha256:0db14f5dafddbb6d9208827849fad01f1a2609380add406671a26386cdf15a19 \\\n    --hash=sha256:0eb9ff8191e8498cca014656ae6b8d61f39da5f95b488805da4bb029cccbfbaf \\\n    --hash=sha256:0f4b68347f8c5eab4a13419215bdfd7f8c9b19f2b25520968adfad23eb0ce60c \\\n    --hash=sha256:1085e7fbddd3be5f89cc898938f42c0b3c711fdcb37d75221de2666af647c175 \\\n    --hash=sha256:116bb52f642a37c115f517494ea5feb03889e04df47eeff5b130b1808ce7c219 \\\n    --hash=sha256:12c63dfb4a98206f045aa9563db46507995f7ef6d83b2f68eda65c307c6829eb \\\n    --hash=sha256:133a43e73a802c5562be9bbcd03d090aa5a1fe899db609c29e8c8d815c5f6de6 \\\n    --hash=sha256:1353ef0c1b138e1907ae78e2f6c63ff67501122006b0f9abad68fda5f4ffc6ab \\\n    --hash=sha256:15d939a21d546304880945ca1ecb8a039db6b4dc49b2c5a400387cdae6a62e26 \\\n    --hash=sha256:177b5253b2834fe3678cb4a5f0059808258584c559193998be2601324fdeafb1 \\\n    --hash=sha256:1872df69a4de6aead3491198eaf13810b565bdbeec3ae2dc8780f14458ec73ce \\\n    --hash=sha256:1b4b79e8ebf6b55351f0d91fe80f893b4743f104bff22e90697db1590e47a218 \\\n    --hash=sha256:1b52b4fb9df4eb9ae465f8d0c228a00624de2334f216f178a995ccdcf82c4634 \\\n    --hash=sha256:1ba88449deb3de88bd40044603fafffb7bc2b055d626a330323a9ed736661695 \\\n    --hash=sha256:1cc7ea17a6824959616c525620e387f6dd30fec8cb44f649e31712db02123dad \\\n    --hash=sha256:218551f6df4868a8d527e3062d0fb968682fe92054e89978594c28e642c43a73 \\\n    --hash=sha256:26a5784ded40c9e318cfc2bdb30fe164bdb8665ded9cd64d500a34fb42067b1c \\\n    --hash=sha256:2713baf880df847f2bece4230d4d094280f4e67b1e813eec43b4c0e144a34ffe \\\n    --hash=sha256:2a15a08b17dd94c53a1da0438822d70ebcd13f8c3a95abe3a9ef9f11a94830aa \\\n    --hash=sha256:2f981d352f04553a7171b8e44369f2af4055f888dfb147d55e42d29e29e74559 \\\n    --hash=sha256:32001d6a8fc98c8cb5c947787c5d08b0a50663d139f1305bac5885d98d9b40fa \\\n    --hash=sha256:3524b778fe5cfb3452a09d31e7b5adefeea8c5be1d43c4f810ba09f2ceb29d37 \\\n    --hash=sha256:3537e01efc9d4dccdf77221fb1cb3b8e1a38d5428920e0657ce299b20324d758 \\\n    --hash=sha256:35add3b638a5d900e807944a078b51922212fb3dedb01633a8defc4b01a3c85f \\\n    --hash=sha256:38664109c14ffc9e7437e86b4dceb442b0096dfe3541d7864d9cbe1da4cf36c8 \\\n    --hash=sha256:3a7e8ae81ae39e62a41ec302f972ba6ae23a5c5396c8e60113e9066ef893da0d \\\n    --hash=sha256:3b562dd9e9ea93f13d53989d23a7e775fdfd1066c33494ff43f5418bc8c58a5c \\\n    --hash=sha256:457a69a9577064c05a97c41f4e65148652db078a3a509039e64d3467b9e7ef97 \\\n    --hash=sha256:4bd4cd07944443f5a265608cc6aab442e4f74dff8088b0dfc8238647b8f6ae9a \\\n    --hash=sha256:4e885a3d1efa2eadc93c894a21770e4bc67899e3543680313b09f139e149ab19 \\\n    --hash=sha256:4faffd047e07c38848ce017e8725090413cd80cbc23d86e55c587bf979e579c9 \\\n    --hash=sha256:509fa21c6deb7a7a273d629cf5ec029bc209d1a51178615ddf718f5918992ab9 \\\n    --hash=sha256:5678211cb9333a6468fb8d8be0305520aa073f50d17f089b5b4b477ea6e67fdc \\\n    --hash=sha256:591ae9f2a647529ca990bc681daebdd52c8791ff06c2bfa05b65163e28102ef2 \\\n    --hash=sha256:5a7d5dc5140555cf21a6fefbdbf8723f06fcd2f63ef108f2854de715e4422cb4 \\\n    --hash=sha256:69c0b73548bc525c8cb9a251cddf1931d1db4d2258e9599c28c07ef3580ef354 \\\n    --hash=sha256:6b5420a1d9450023228968e7e6a9ce57f65d148ab56d2313fcd589eee96a7a50 \\\n    --hash=sha256:722695808f4b6457b320fdc131280796bdceb04ab50fe1795cd540799ebe1698 \\\n    --hash=sha256:729586769a26dbceff69f7a7dbbf59ab6572b99d94576a5592625d5b411576b9 \\\n    --hash=sha256:77f0643abe7495da77fb436f50f8dab76dbc6e5fd25d39589a0f1fe6548bfa2b \\\n    --hash=sha256:795e7751525cae078558e679d646ae45574b47ed6e7771863fcc079a6171a0fc \\\n    --hash=sha256:7be7b61bb172e1ed687f1754f8e7484f1c8019780f6f6b0786e76bb01c2ae115 \\\n    --hash=sha256:7c3fb7d25180895632e5d3148dbdc29ea38ccb7fd210aa27acbd1201a1902c6e \\\n    --hash=sha256:7e68f88e5b8799aa49c85cd116c932a1ac15caaa3f5db09087854d218359e485 \\\n    --hash=sha256:83891d0e9fb81a825d9a6d61e3f07550ca70a076484292a70fde82c4b807286f \\\n    --hash=sha256:8485f406a96febb5140bfeca44a73e3ce5116b2501ac54fe953e488fb1d03b12 \\\n    --hash=sha256:8709b08f4a89aa7586de0aadc8da56180242ee0ada3999749b183aa23df95025 \\\n    --hash=sha256:8f71bc33915be5186016f675cd83a1e08523649b0e33efdb898db577ef5bb009 \\\n    --hash=sha256:915c04ba3851909ce68ccc2b8e2cd691618c4dc4c4232fb7982bca3f41fd8c3d \\\n    --hash=sha256:949b8d66bc381ee8b007cd945914c721d9aba8e27f71959d750a46f7c282b20b \\\n    --hash=sha256:94c6f0bb423f739146aec64595853541634bde58b2135f27f61c1ffd1cd4d16a \\\n    --hash=sha256:9a1abfdc021a164803f4d485104931fb8f8c1efd55bc6b748d2f5774e78b62c5 \\\n    --hash=sha256:9b79b7a16f7fedff2495d684f2b59b0457c3b493778c9eed31111be64d58279f \\\n    --hash=sha256:a320721ab5a1aba0a233739394eb907f8c8da5c98c9181d1161e77a0c8e36f2d \\\n    --hash=sha256:a4afe79fb3de0b7097d81da19090f4df4f8d3a2b3adaa8764138aac2e44f3af1 \\\n    --hash=sha256:ad2cf8aa28b8c020ab2fc8287b0f823d0a7d8630784c31e9ee5edea20f406287 \\\n    --hash=sha256:b8512a91625c9b3da6f127803b166b629725e68af71f8184ae7e7d54686a56d6 \\\n    --hash=sha256:bc51efed119bc9cfdf792cdeaa4d67e8f6fcccab66ed4bfdd6bde3e59bfcbb2f \\\n    --hash=sha256:bdc919ead48f234740ad807933cdf545180bfbe9342c2bb451556db2ed958581 \\\n    --hash=sha256:bdd37121970bfd8be76c5fb069c7751683bdf373db1ed6c010162b2a130248ed \\\n    --hash=sha256:be8813b57049a7dc738189df53d69395eba14fb99345e0a5994914a3864c8a4b \\\n    --hash=sha256:c0c0b3ade1c0b13b936d7970b1d37a57acde9199dc2aecc4c336773e1d86049c \\\n    --hash=sha256:c47a551199eb8eb2121d4f0f15ae0f923d31350ab9280078d1e5f12b249e0026 \\\n    --hash=sha256:c4ffb7ebf07cfe8931028e3e4c85f0357459a3f9f9490886198848f4fa002ec8 \\\n    --hash=sha256:ccfcd093f13f0f0b7fdd0f198b90053bf7b2f02a3927a30e63f3ccc9df56b676 \\\n    --hash=sha256:d2ee202e79d8ed691ceebae8e0486bd9a2cd4794cec4824e1c99b6f5009502f6 \\\n    --hash=sha256:d53197da72cc091b024dd97249dfc7794d6a56530370992a5e1a08983ad9230e \\\n    --hash=sha256:d6dd0be5b5b189d31db7cda48b91d7e0a9795f31430b7f271219ab30f1d3ac9d \\\n    --hash=sha256:d88b440e37a16e651bda4c7c2b930eb586fd15ca7406cb39e211fcff3bf3017d \\\n    --hash=sha256:de8a88e63464af587c950061a5e6a67d3632e36df62b986892331d4620a35c01 \\\n    --hash=sha256:df2449253ef108a379b8b5d6b43f4b1a8e81a061d6537becd5582fba5f9196d7 \\\n    --hash=sha256:e1c1493fb6e50ab01d20a22826e57520f1284df32f2d8601fdd90b6304601419 \\\n    --hash=sha256:e1cf1972137e83c5d4c136c43ced9ac51d0e124706ee1c8aa8532c1287fa8795 \\\n    --hash=sha256:e2103a929dfa2fcaf9bb4e7c091983a49c9ac3b19c9061b6d5427dd7d14d81a1 \\\n    --hash=sha256:e56b7d45a839a697b5eb268c82a71bd8c7f6c94d6fd50c3d577fa39a9f1409f5 \\\n    --hash=sha256:e8afc3f2ccfa24215f8cb28dcf43f0113ac3c37c2f0f0806d8c70e4228c5cf4d \\\n    --hash=sha256:e8fc20152abba6b83724d7ff268c249fa196d8259ff481f3b1476383f8f24e42 \\\n    --hash=sha256:eaa9599de571d72e2daf60164784109f19978b327a3910d3e9de8c97b5b70cfe \\\n    --hash=sha256:ec15a59cf5af7be74194f7ab02d0f59a62bdcf1a537677ce67a2537c9b87fcda \\\n    --hash=sha256:f190daf01f13c72eac4efd5c430a8de82489d9cff23c364c3ea822545032993e \\\n    --hash=sha256:f34c41761022dd093b4b6896d4810782ffbabe30f2d443ff5f083e0cbbb8c737 \\\n    --hash=sha256:f3e98bb3798ead92273dc0e5fd0f31ade220f59a266ffd8a4f6065e0a3ce0523 \\\n    --hash=sha256:f42d0984e947b8adf7dd6dde396e720934d12c506ce84eea8476409563607591 \\\n    --hash=sha256:f71a396b3bf33ecaa1626c255855702aca4d3d9fea5e051b41ac59a9c1c41edc \\\n    --hash=sha256:f9e130248f4462aaa8e2552d547f36ddadbeaa573879158d721bbd33dfe4743a \\\n    --hash=sha256:fed51ac40f757d41b7c48425901843666a6677e3e8eb0abcff09e4ba6e664f50\n    # via jinja2\nmpmath==1.3.0 \\\n    --hash=sha256:7a28eb2a9774d00c7bc92411c19a89209d5da7c4c9a9e227be8330a23a25b91f \\\n    --hash=sha256:a0b2b9fe80bbcd81a6647ff13108738cfb482d481d826cc0e02f5b35e5c88d2c\n    # via sympy\nnetworkx==3.7 \\\n    --hash=sha256:e3fd2c13a7814cee3746340d8d7f8598a67f16a58bf47fb7f8793fab6efca1b0 \\\n    --hash=sha256:fd77a511bd90f39f3d016351345b52cf5319b813bdca01de3f755d3cca62e96a\n    # via torch\nnumpy==2.5.3 \\\n    --hash=sha256:012e66aca395d795496446e52aeeb5866312a5d4d3f27da270e5a0b43f70dc5c \\\n    --hash=sha256:09d5a423c71ad5feb5625844ad58050e35df43871004b52ac9c0ad44a56775be \\\n    --hash=sha256:09ffa5d903faeaa5c4dd05009cf81c8bab9f2cb37c548b8d39b65b4cfa7c97f7 \\\n    --hash=sha256:0a59a421a32580a009e8a1751345bf829631b990dc1794b80514ab722b435def \\\n    --hash=sha256:116f96cadd935c6122e9228d676fe7ede19e741f5c8bb1c3cddbe0c51ccebea2 \\\n    --hash=sha256:1302b90c0e52281681b2975adfe8a860cb7b12216a27b4b0b4207c44bf7bccf0 \\\n    --hash=sha256:15aa985ac73a8db02db7663381aa109510449d3819d37206caed27b33a65a8a6 \\\n    --hash=sha256:1aad64d99730d013cfc6debafed22783b4fc5a7f4b8bc744d2d8cf7dcc880551 \\\n    --hash=sha256:1c80eabb4035ecf4ca9cd49cde8a9fdd69a729e63e6474887d1523ade7aa277f \\\n    --hash=sha256:1f3ed25271581281f2fccb1adcedfcde4c07362eec69189b50baf6f90e3ae159 \\\n    --hash=sha256:1fb6f8fb9ff0b3a69f52c66ce397b0246583e9f28616231b0e32ca49259a5fa6 \\\n    --hash=sha256:214045a5bf00113a146ab9ee9730c44501af6723cdf1f6830932f7b5ef2e7af0 \\\n    --hash=sha256:26e15e4aecd8617dfbaecb37d223e365d7b39411fba20454be2670a96aa74cb5 \\\n    --hash=sha256:2c25dfa72943e4336ddb6b0ee4277b47a0c85bede0807530ec68103bf58e2c10 \\\n    --hash=sha256:2d8240cb4c16fd831074aa2b2cf9fc54664d826341d61c372245b96a74a49a9a \\\n    --hash=sha256:350ba9783ce969cf9f7ce6e6a9a58e1a6e2a19ca025b7ee448c4db727706212a \\\n    --hash=sha256:4c8a6d2ebce6305fd82fbefca827775437147052a976ee7c94b36a0c1b52ac6c \\\n    --hash=sha256:4f8929ee6c96bfbd7b4ed2032e0c03af86fe1826740ab61ddabf9072d06e57ff \\\n    --hash=sha256:536f963710a4e63934d80ac0dc4f478804a83e9a84b6828018f25d09953ada33 \\\n    --hash=sha256:54a115e5a73b8fc44f0cebef486365a1894b5c9760685d4558b72b7c3eb846e0 \\\n    --hash=sha256:595d020938c84e320bcf40ad71089e108eac0d377cd018e14a8c094f39e98d85 \\\n    --hash=sha256:66a78fe4556c60aceda5916f9eacd638b18e9e681016ec302dcb4682d6d4d034 \\\n    --hash=sha256:6b05c171afb3aa07adbd20abc00aea86fe375beb0fdb9ef780ec5b7f63bab1c0 \\\n    --hash=sha256:6cef4bb1706dfec49243c05d921eefb4e190d41e2528b30d8035ea1f36b4c24a \\\n    --hash=sha256:6f24021b9f22bc6301c37b196974a92c1c18dccedb6fef3dd252e95f2d6adbe4 \\\n    --hash=sha256:71b39d9f935b6ec0f8753e3e2afb51e3efba6f2e05b68b32a40754d24bcd4a3c \\\n    --hash=sha256:71cad2b2a7451ab79d8f5e71b453485b6775963d5cf794179144a7463fe6e8ec \\\n    --hash=sha256:76c2c1e6bfa5c84adc6434dfbf013aa92096a7985221762c8f11fedfd20fff58 \\\n    --hash=sha256:8617bbfae4486cf99c9f899966699428d19da931d06ca94ad3da986c76e15997 \\\n    --hash=sha256:86bff898a431c0fb71f7610b75726e75a54d47b37edc9d537f48de63bb3c0b90 \\\n    --hash=sha256:8e4dd766076855b5ff7ea52fa5f07ce26286726e0f8bff446b7739d02e6ea204 \\\n    --hash=sha256:92f30e89b8ee0ecf363033576c422b2f58fed6a80bed0aa48dff6d14c654663e \\\n    --hash=sha256:93e1f5447e2b1e479d7bd74701e84746b86450cff1fc368b132d195e2b8f8211 \\\n    --hash=sha256:9a37475425b431b4d060f23b4f52cd2f3aef6bc7c654bd760adf0040eec9d435 \\\n    --hash=sha256:9deb49575e5b0b94ed72c8a64ec4d033381adc27e9060ae842971f697ba96104 \\\n    --hash=sha256:a5fa86b80fd24bcd1aff83ad23be44ea323de3f787be8f8b15d4a65621e25321 \\\n    --hash=sha256:a6391fafaba97500887132cd582abc6e19452b1ac775a47caa7b24490e152058 \\\n    --hash=sha256:a72f874bc9e10e4b8f80426fb49716d5141f64442a0c8418065093ec8017fbb0 \\\n    --hash=sha256:ac7bb1c52d445bd4f8f7f97fefe6abc3a084dc4d63df50d79b17fa2b78e89297 \\\n    --hash=sha256:adc1ada2662f8a5f960b8a10d9986897e7499ef07e06d4cfe7197f8cce923c07 \\\n    --hash=sha256:b00eefbcf0f292945c4b4dec2ae845389ef5bcdcd596e6e4328051db5b5ba694 \\\n    --hash=sha256:b0521d0f4aebb6e06189451025fa17a913287b13c03d5fe05c017333b654ea5b \\\n    --hash=sha256:b5d93cf48f687479941d12b69c873ad2cc76bbd487f0091c2200636497f34034 \\\n    --hash=sha256:b7e18c623bb5c95acb3b3328861272816ba199fb531921c5d6d0b675f1fde9e3 \\\n    --hash=sha256:bd4cb9ad3c7889b9b3fe0a9a9fb5d2ed26f9879bff2608d9f01aed147a20d231 \\\n    --hash=sha256:be5a8381859b6da607c84f4f7d6847725f1cf1853ef8a2c9e115b7d58bef47dc \\\n    --hash=sha256:befa1ae5bd6030b3f512b43ff3fa5290bbed6b84411a44244b14adf835f5b89d \\\n    --hash=sha256:bf63afbe037eb5d2fe87fbcc7778e61da53ebaf21d938a4515aa73b62532a5d4 \\\n    --hash=sha256:c00abe94c1a69d75d827dcf1c025b25c8a45d230b3bcd77a9020883a1b047653 \\\n    --hash=sha256:c2381f82999704f818e2c987a865050e285ec3621262c66d40f5a96c8f899f8e \\\n    --hash=sha256:c76d5dde9f445058f83d0c02af00557a4db91de9a9a57c0df87d1535001d654b \\\n    --hash=sha256:cb189f09db39283b26bfd061ec16189e14f71c6755207f72a0f7540867afe5b9 \\\n    --hash=sha256:ccb32e0525d29e8b0572eb84c9a57af0e7a4e615726927506f55063c62414034 \\\n    --hash=sha256:ccbc4665079665c3cf3bab4db9f6b095370cd6437d66be549b6c2a1fd19e1958 \\\n    --hash=sha256:d1c89973648c85069c5046ad460f7b8a00218b29a2e42359ac8cc63e9ab94832 \\\n    --hash=sha256:df2d5874ff183595a4ba404edd04f6bd9b5505c1d7708573f6a6c17489a67563 \\\n    --hash=sha256:e01c918ac3d48e18a927cf7b14a26a3e29ff2bdf2eacb976da0aecd6a43ed034 \\\n    --hash=sha256:e6ab667ba76450084eb64013762c438ea76d9d29cc676dcd6c2e9892ba37f841 \\\n    --hash=sha256:e931e4f499e0dc7ef29d269a8e5b35dd722e5d14be07df6240166ea7c6532fae \\\n    --hash=sha256:f54660b0eb6b0b9f36e7fe1cdfdff472028dd0d14acd9b9b65098efbad059469 \\\n    --hash=sha256:f59a878c33d6b88122d80d239bb3b845d58708750b0cb06a09aebb9b18ec696c \\\n    --hash=sha256:f7fabeb6cea87d65f3b926de33d03fb016cfdc29314c90974383b5582ae72891 \\\n    --hash=sha256:f9579f383d1bf9df80081e72760e84960a7fd4f88cf0c9e535a8597c9bb646f5 \\\n    --hash=sha256:f9a2353b37a1a9e78fd82b27ad7e2a32a2d036604d18f02b05e3136c62ca3b09 \\\n    --hash=sha256:fc36dc566135b5eceec4cf89758fcb719266a019ef07dae1754ae7c9f617ef3e \\\n    --hash=sha256:ffdc76bfcae6b255dff75202c5e7feaf95b40246bc0a17944facc1fecf9f79ab\n    # via\n    #   ast-audio-classification-pipeline (pyproject.toml)\n    #   torchvision\n    #   transformers\nnvidia-cublas==13.1.1.3 \\\n    --hash=sha256:37936a16db8fe4ac1f065c2139360608a543a09275cb1a1af612e08cfa065436 \\\n    --hash=sha256:b6cdce694e47ff6aadf0a69df1cab6628d696f5ff56e8d16af50309d855fa20f \\\n    --hash=sha256:b7a210458267ac818974c53038fbec2e969d5c99f305ab15c72522fa9f001dd5\n    # via\n    #   cuda-toolkit\n    #   nvidia-cudnn-cu13\n    #   nvidia-cusolver\nnvidia-cuda-cupti==13.0.85 \\\n    --hash=sha256:4eb01c08e859bf924d222250d2e8f8b8ff6d3db4721288cf35d14252a4d933c8 \\\n    --hash=sha256:683f58d301548deeefcb8f6fac1b8d907691b9d8b18eccab417f51e362102f00 \\\n    --hash=sha256:796bd679890ee55fb14a94629b698b6db54bcfd833d391d5e94017dd9d7d3151\n    # via cuda-toolkit\nnvidia-cuda-nvrtc==13.0.88 \\\n    --hash=sha256:6bcd4e7f8e205cbe644f5a98f2f799bef9556fefc89dd786e79a16312ce49872 \\\n    --hash=sha256:ad9b6d2ead2435f11cbb6868809d2adeeee302e9bb94bcf0539c7a40d80e8575 \\\n    --hash=sha256:d27f20a0ca67a4bb34268a5e951033496c5b74870b868bacd046b1b8e0c3267b\n    # via\n    #   cuda-toolkit\n    #   nvidia-cublas\nnvidia-cuda-runtime==13.0.96 \\\n    --hash=sha256:7f82250d7782aa23b6cfe765ecc7db554bd3c2870c43f3d1821f1d18aebf0548 \\\n    --hash=sha256:ef9bcbe90493a2b9d810e43d249adb3d02e98dd30200d86607d8d02687c43f55 \\\n    --hash=sha256:f79298c8a098cec150a597c8eba58ecdab96e3bdc4b9bc4f9983635031740492\n    # via cuda-toolkit\nnvidia-cudnn-cu13==9.24.0.43 \\\n    --hash=sha256:67a7273b5cf062f9446fd76cf464351a1c0f66501e6cd78f6675c0d604d8ac87 \\\n    --hash=sha256:71f181cd810e90f9b6023b01186fe82d13d65f0ec098581ee201d39fad769e4b \\\n    --hash=sha256:a6812a554a1ff0413e9c52b84c26c050380649ab9615f9c16bded368ce9f421f\n    # via torch\nnvidia-cufft==12.0.0.61 \\\n    --hash=sha256:2708c852ef8cd89d1d2068bdbece0aa188813a0c934db3779b9b1faa8442e5f5 \\\n    --hash=sha256:2abce5b39d2f5ae12730fb7e5db6696533e36c26e2d3e8fd1750bdd2853364eb \\\n    --hash=sha256:6c44f692dce8fd5ffd3e3df134b6cdb9c2f72d99cf40b62c32dde45eea9ddad3\n    # via cuda-toolkit\nnvidia-cufile==1.15.1.6 \\\n    --hash=sha256:08a3ecefae5a01c7f5117351c64f17c7c62efa5fffdbe24fc7d298da19cd0b44 \\\n    --hash=sha256:bdc0deedc61f548bddf7733bdc216456c2fdb101d020e1ab4b88d232d5e2f6d1\n    # via cuda-toolkit\nnvidia-curand==10.4.0.35 \\\n    --hash=sha256:133df5a7509c3e292aaa2b477afd0194f06ce4ea24d714d616ff36439cee349a \\\n    --hash=sha256:1aee33a5da6e1db083fe2b90082def8915f30f3248d5896bcec36a579d941bfc \\\n    --hash=sha256:65b1710aa6961d326b411e314b374290904c5ddf41dc3f766ebc3f1d7d4ca69f\n    # via cuda-toolkit\nnvidia-cusolver==12.0.4.66 \\\n    --hash=sha256:02c2457eaa9e39de20f880f4bd8820e6a1cfb9f9a34f820eb12a155aa5bc92d2 \\\n    --hash=sha256:0a759da5dea5c0ea10fd307de75cdeb59e7ea4fcb8add0924859b944babf1112 \\\n    --hash=sha256:16515bd33a8e76bb54d024cfa068fa68d30e80fc34b9e1090813ea9362e0cb65\n    # via cuda-toolkit\nnvidia-cusparse==12.6.3.3 \\\n    --hash=sha256:2b3c89c88d01ee0e477cb7f82ef60a11a4bcd57b6b87c33f789350b59759360b \\\n    --hash=sha256:80bcc4662f23f1054ee334a15c72b8940402975e0eab63178fc7e670aa59472c \\\n    --hash=sha256:cbcf42feb737bd7ec15b4c0a63e62351886bd3f975027b8815d7f720a2b5ea79\n    # via\n    #   cuda-toolkit\n    #   nvidia-cusolver\nnvidia-cusparselt-cu13==0.8.1 \\\n    --hash=sha256:4dca476c50bf4780d46cd0bfbd82e2bc10a08e4fef7950917ce8d7578d22a23f \\\n    --hash=sha256:786ce87568c303fadb5afcc7102d454cd3040d75f6f8626f5db460d1871f4dd0 \\\n    --hash=sha256:dccbd362f91a7b9024d1f55ee9f548ac065027ff15d8c8b0db889ab3a8f31215\n    # via torch\nnvidia-nccl-cu13==2.30.7 \\\n    --hash=sha256:ca786ffa5a647c75d4d1f5cc72a6c4f537947e2ba8823d7c8aaf768e7a7b9f77 \\\n    --hash=sha256:cefa7fdb9710efd0f39c5f1be1d61ff6fc9a996c451265bd7fbdcf9455ed4b50\n    # via torch\nnvidia-nvjitlink==13.4.92 \\\n    --hash=sha256:25f74fad0d654271c921ac4dca614bd6258bc21791242fc7b2289dad7ae9c099 \\\n    --hash=sha256:9e4a7ff4f0cafa8c624917055b863dc11f5c2912ead23c462889e166f3b0e57d \\\n    --hash=sha256:b286f3a4f227a9363efdec263c7b91788cef1478d2b8a5fa8bab7f3e82ff82fd \\\n    --hash=sha256:e0391f24ed94ec879b84e3da4d4ec320c879aff681f2c7a638462f7199284323\n    # via\n    #   cuda-toolkit\n    #   nvidia-cufft\n    #   nvidia-cusolver\n    #   nvidia-cusparse\nnvidia-nvshmem-cu13==3.4.5 \\\n    --hash=sha256:290f0a2ee94c9f3687a02502f3b9299a9f9fe826e6d0287ee18482e78d495b80 \\\n    --hash=sha256:6dc2a197f38e5d0376ad52cd1a2a3617d3cdc150fd5966f4aee9bcebb1d68fe9\n    # via torch\nnvidia-nvtx==13.0.85 \\\n    --hash=sha256:4936d1d6780fbe68db454f5e72a42ff64d1fd6397df9f363ae786930fd5c1cd4 \\\n    --hash=sha256:cb7780edb6b14107373c835bf8b72e7a178bac7367e23da7acb108f973f157a6 \\\n    --hash=sha256:d66ea44254dd3c6eacc300047af6e1288d2269dd072b417e0adffbf479e18519\n    # via cuda-toolkit\npackaging==26.3 \\\n    --hash=sha256:94edc256424af38762eb31306eed28beb9f0efc50a8837492c9d6fd6004aed79 \\\n    --hash=sha256:d7193f7c8e4e93f444fde0262bf90af30e16fa0ad0ad44cb553c87339b23cd1c\n    # via\n    #   huggingface-hub\n    #   transformers\npillow==12.3.0 \\\n    --hash=sha256:00808c5e14ef63ac5161091d242999076604ff74b883423a11e5d7bbb38bf756 \\\n    --hash=sha256:04f01d28a6aaff387bf842a13be313df23ba0597a44f1a976c9feb3c6ff4711a \\\n    --hash=sha256:06ff022112bc9cbf83b60f8e028d94ad87b60621706487e65f673de61610ab59 \\\n    --hash=sha256:0740a512dc522224c77d9aa5a8d70d8b7d73fb91f2c21125d8d025d3b8990e45 \\\n    --hash=sha256:0847a763afefb695bc912d7c131e7e0632d4edc1d8698f58ddabec8e46b8b6d3 \\\n    --hash=sha256:0dd2064cbc55aaec028ef5fbb60fa47bb6c3e7918e07ff17935284b227a9d2df \\\n    --hash=sha256:0feb2e9d6ad6c9e3c06effe9d00f3f1e618a6643273576b016f591e9315a7139 \\\n    --hash=sha256:10e41f0fbf1eec8cfd234b8fe17a4caac7c9d0db4c204d3c173a8f9f6ef3232b \\\n    --hash=sha256:1182d52bc2d5e5d7d0949503aa7e36d12f42205dc287e4883f407b1988820d39 \\\n    --hash=sha256:164b31cd1a0490ab6efae01aa5df49da7061be0af1b30e035b6e9a1bfe34ee6e \\\n    --hash=sha256:1657923d2d45afb66526e5b933e5b3052e6bdea196c90d3abb2424e18c77dae8 \\\n    --hash=sha256:186941b6aef820ad110fb01fb06eb925374dc3a21b17e37ec9a53b250c6fe2d1 \\\n    --hash=sha256:1cca606cd25738df4ed873d5ad46bbdb3d83b5cbca291f6b4ff13a4df6b0bbe8 \\\n    --hash=sha256:21900ce7ba264168cd50defae43cd75d25c833ad4ad6e73ffc5596d12e25ac89 \\\n    --hash=sha256:236ff70b9312fb68943c703aa842ca6a758abfa45ac187a5e7c1452e96ef72b5 \\\n    --hash=sha256:23aceaa007d6172b02c277f0cd359c79492bbb14f7072b4ede9fbcaf20648130 \\\n    --hash=sha256:23d27a3e0307ec2244cc51e7287b919aa68d097504ebe19df4e76a98a3eea5bd \\\n    --hash=sha256:24870b09b224f7ae3c39ed07d10e819d06f8720bc551847b1d623832b5b0e28d \\\n    --hash=sha256:251bf95b67017e27b13d82f5b326234ca62d70f9cf4c2b9032de2358a3b12c7b \\\n    --hash=sha256:25b9b82bb22e6e2b3cd07b39c68b7b862001226cb3dff7130d1cb914121b39ed \\\n    --hash=sha256:28ce87c5ab450a9dd970b52e5aca5fe63ed432d18a2eaddd1979a00a1ba24ace \\\n    --hash=sha256:300557495eb45ebb8aec96c2da9c4be642fbf7cd937278b4013ba894ea8eb0eb \\\n    --hash=sha256:30f2aa603c41533cc25c05acd0da21636e84a315768feb631c937177db558931 \\\n    --hash=sha256:331b624368d4f1d069149002f25f44bc61c8919ce8ddb3c45bdad8f6e2d89510 \\\n    --hash=sha256:37d6d0a00072fd2948eb22bce7e1475f34569d90c87c59f7a2ec59541b77f7a6 \\\n    --hash=sha256:37dc8f7bbb66efe481bb60defacef820c950c24713fb44962ed6aa2a50966de1 \\\n    --hash=sha256:3b8182a766685eaa002637e28b4ec8d6b18819a0c71f579bf0dbaa5830297cce \\\n    --hash=sha256:3edce1d53195db527e0191f84b71d02022de0540bf43a16ed734ed7537b07385 \\\n    --hash=sha256:446c34dcc4324b084a53b705127dc15717b22c5e140ae0a3c38349d4efec071e \\\n    --hash=sha256:4998562bf62a445225f22e07c896bb04b35b1b1f2eb6d760584c9c51d7a5f78c \\\n    --hash=sha256:4b0a7fe987b14c31ebda6083f74f22b561fd3739bc0ac51e019622e3d72668c7 \\\n    --hash=sha256:4e8c2a84d977f50b9daed6eeaf3baef67d00d5d74d932288f02cb94518ee3ace \\\n    --hash=sha256:4f883547d4b7f0495ebe7056b0cc2aea76094e7a4abc8e933540f3271df27d9c \\\n    --hash=sha256:514435a37670e3e5e08f3945b68718b6ed329bb84367777e16f9f4dfe1e61a0f \\\n    --hash=sha256:53aa02d20d10c3d814d536aa4e5ac9b84ca0ff5a88377963b085ad6822f93e64 \\\n    --hash=sha256:5594fc43d548a7ed94949d139aa1341b270f1863f11cfd37f5a6c8b778a6b67f \\\n    --hash=sha256:571b9fcb07b97ef3a492028fb3d2dc0993ca23a06138b0315286566d29ef718a \\\n    --hash=sha256:57b3d78c95ba9059768b10e28b813002261d3f3dfc55cc48b0c988f625175827 \\\n    --hash=sha256:5afb51d599ea772b8365ae807ae557f18bccfe46ab261fd1c2a9ed700fc6eb17 \\\n    --hash=sha256:6b02afb9b97f65fbca5f31db6a2a3ba21aa93030225f150fa3f249717e938fb4 \\\n    --hash=sha256:6c0016e7b354317c4e9e525b937ac8596c38d2d232b419529b9cd7a1cd46e39a \\\n    --hash=sha256:71d6097b330eea8fd15097780c8e89cb1a8ce7838669f48c5bacd6f663dd4701 \\\n    --hash=sha256:756c768d0c9c2955feb7a56c37ea24aea2e369f8d36a88da270b6a9f19e62b5e \\\n    --hash=sha256:78cb2c6865a35ab8ff8b75fd122f6033b92a62c82801110e48ddd6c936a45d91 \\\n    --hash=sha256:7a743ff716f746fc19a9557f60dab1600d4613255f8a7aeb3cdde4db7eb15a66 \\\n    --hash=sha256:85f998ea1848bc6757289e739cfbdda3a04adfd58b02fc018ce54d754a5ce468 \\\n    --hash=sha256:8728f216dcdb6e6d555cf971cb34076139ad74b31fc2c14da4fafc741c5f6217 \\\n    --hash=sha256:877c3f311ff35410f690861c4409e7ccbf0cd2f878e50628a28e5a0bb689e658 \\\n    --hash=sha256:8cd2f7bdda092d99c9fc2fb7391354f306d01443d22785d0cbfafa2e2c8bb418 \\\n    --hash=sha256:8e95e1385e4998ae9694eeaa4730ba5457ff61185b3a55e2e7bea0880aef452a \\\n    --hash=sha256:962864dc93511324d51ddbb5b9f8731bf71675b93ca612a07441896f4688fb8c \\\n    --hash=sha256:9cf95fe4d0f84c82d282745d9bb08ad9f926efa00be4697e767b814ce40d4330 \\\n    --hash=sha256:9e881fca225083806662a5c43d627d215f258ff43c890f831966c7d7ba9c7402 \\\n    --hash=sha256:a2b55dd6b2a4c4b7d87ffa56bdb33fdc5fdb9a462173861a7bc097f17d91cb09 \\\n    --hash=sha256:a45650e8ce7fafffd731db8550230db6b0d306d181a90b67d3e6bca2f1990930 \\\n    --hash=sha256:a876864214e136f0eb367788dbd7df045f4806801518e2cfe9e13229cfe06d8f \\\n    --hash=sha256:ae26d61dfa7a47befdc7572b521024e8745f3d809bd95ca9505a7bba9ef849ec \\\n    --hash=sha256:af8d94b0db561cf68b88a267c5c44b49e134f525d0dc2cb7ed413a66bc23559a \\\n    --hash=sha256:b343699e8308bdc51978310e1c959c584e7869cc8c40780058c87da7781a1e94 \\\n    --hash=sha256:b3c777e849237620b022f7f297dd67705f9f5cf1685f09f02e46f93e92725468 \\\n    --hash=sha256:b629de27fda84b42cde7edef0d85f13b958b47f6e9bbcbba9b673c562a89bd8b \\\n    --hash=sha256:ba09209fbe443b4acccebe845d8a138b89a8f4fbaeedd44953490b5315d5e965 \\\n    --hash=sha256:ba54cfebe86920a559a7c4d6b9050791c20513650a1952ebe3368c7dc70306f8 \\\n    --hash=sha256:bcb46e2f9feff8d06323983bd83ed00c201fdcab3d74973e7072a889b3979fcd \\\n    --hash=sha256:bcc33feacfaefce60c12fd500a277533bdc02b10a19f7f6d348763d8140bbba7 \\\n    --hash=sha256:bf16ba1b4d0b6b7c8e534936632270cf70eb00dbe09005bc345b2677b726855c \\\n    --hash=sha256:cf1845d02ad822a369a49f2bb9345b1614744267682e7a03527dc3bf6eea1777 \\\n    --hash=sha256:d69141514cc30b774ceea5e3ed3a6635c8d8a96edf664689b890f4089111fb35 \\\n    --hash=sha256:d9c7f76c0673154f044e9d78c8655fb4213f6ca31a836df48b40fe5d187717b9 \\\n    --hash=sha256:dbce0b29841537a2fa4a214c2bbf14de3587c9680caa9b4e217568472490b28f \\\n    --hash=sha256:dc624f6bc473dacdf7ef7eb8678d0d08edf15cd94fad6ae5c7d6cc67a4e4902f \\\n    --hash=sha256:e158cb00350dc278f3b91551101aa7d12415a66ebf2c91d8d5ac14e56ddd3ad0 \\\n    --hash=sha256:e491916b378fba47242221bb9ead245211b70d504f495d105d17b14a24b4907c \\\n    --hash=sha256:e795b7eb908249c4e43c7c99fac7c2c75dab0c43566e37db472a355f63693d71 \\\n    --hash=sha256:e7e480451b9fa137494bccd3a7d69adbe8ac65a87d97be61e11f1b1050a5bac3 \\\n    --hash=sha256:e91206ee562682b51b98ef4b26a6ef48fd84e15fd4c4bc5ec768eb641d206838 \\\n    --hash=sha256:e9871b1ffbfa9656b60aeee92ed5136a5742696006fa322b29ea3d8da0ecc9cf \\\n    --hash=sha256:e9aeb04d6aef139de265b29683e119b638208f88cf73cdd1658aa07221165321 \\\n    --hash=sha256:ebaea975e03d3141d9d3a507df75c9b3ec90fa9d2ffd07567b3a978d9d790b26 \\\n    --hash=sha256:f0606c8bf2cdefea14a43530f7657cbbb7ecf1c4222512492ef4a4434a9501ec \\\n    --hash=sha256:f13c32a3abd6079a66d9526e18dad9b6d280384d49d7c54040cd57b6424041d9 \\\n    --hash=sha256:f7401aebd7f581d7f83a439d87d474999317ee099218e5ad25d125290990ba65 \\\n    --hash=sha256:fa4ecea169a355be7a3ade2c783e2ed12f0e40d2c5621cda8b3297faf7fbb9f5 \\\n    --hash=sha256:fbd139c8447d25dd750ab79ee274cc5e1fe80fc56340ab10b18a195e1b6eca3e \\\n    --hash=sha256:fdafc9cce40277e0f7a0feabce0ee50dd2fa1800f3b38015e51296b5e814048d \\\n    --hash=sha256:fe3cca2e4e8a592be0f269a1ca4835c25199d9f3ce815c8491048f785b0a0198 \\\n    --hash=sha256:ffd0c5368496f41b0944be820fcb7a838aa6e623d250b01acf2643939c3f99d7\n    # via torchvision\npyyaml==6.0.3 \\\n    --hash=sha256:00c4bdeba853cc34e7dd471f16b4114f4162dc03e6b7afcc2128711f0eca823c \\\n    --hash=sha256:0150219816b6a1fa26fb4699fb7daa9caf09eb1999f3b70fb6e786805e80375a \\\n    --hash=sha256:02893d100e99e03eda1c8fd5c441d8c60103fd175728e23e431db1b589cf5ab3 \\\n    --hash=sha256:02ea2dfa234451bbb8772601d7b8e426c2bfa197136796224e50e35a78777956 \\\n    --hash=sha256:0f29edc409a6392443abf94b9cf89ce99889a1dd5376d94316ae5145dfedd5d6 \\\n    --hash=sha256:10892704fc220243f5305762e276552a0395f7beb4dbf9b14ec8fd43b57f126c \\\n    --hash=sha256:16249ee61e95f858e83976573de0f5b2893b3677ba71c9dd36b9cf8be9ac6d65 \\\n    --hash=sha256:1d37d57ad971609cf3c53ba6a7e365e40660e3be0e5175fa9f2365a379d6095a \\\n    --hash=sha256:1ebe39cb5fc479422b83de611d14e2c0d3bb2a18bbcb01f229ab3cfbd8fee7a0 \\\n    --hash=sha256:214ed4befebe12df36bcc8bc2b64b396ca31be9304b8f59e25c11cf94a4c033b \\\n    --hash=sha256:2283a07e2c21a2aa78d9c4442724ec1eb15f5e42a723b99cb3d822d48f5f7ad1 \\\n    --hash=sha256:22ba7cfcad58ef3ecddc7ed1db3409af68d023b7f940da23c6c2a1890976eda6 \\\n    --hash=sha256:27c0abcb4a5dac13684a37f76e701e054692a9b2d3064b70f5e4eb54810553d7 \\\n    --hash=sha256:28c8d926f98f432f88adc23edf2e6d4921ac26fb084b028c733d01868d19007e \\\n    --hash=sha256:2e71d11abed7344e42a8849600193d15b6def118602c4c176f748e4583246007 \\\n    --hash=sha256:34d5fcd24b8445fadc33f9cf348c1047101756fd760b4dacb5c3e99755703310 \\\n    --hash=sha256:37503bfbfc9d2c40b344d06b2199cf0e96e97957ab1c1b546fd4f87e53e5d3e4 \\\n    --hash=sha256:3c5677e12444c15717b902a5798264fa7909e41153cdf9ef7ad571b704a63dd9 \\\n    --hash=sha256:3ff07ec89bae51176c0549bc4c63aa6202991da2d9a6129d7aef7f1407d3f295 \\\n    --hash=sha256:41715c910c881bc081f1e8872880d3c650acf13dfa8214bad49ed4cede7c34ea \\\n    --hash=sha256:418cf3f2111bc80e0933b2cd8cd04f286338bb88bdc7bc8e6dd775ebde60b5e0 \\\n    --hash=sha256:44edc647873928551a01e7a563d7452ccdebee747728c1080d881d68af7b997e \\\n    --hash=sha256:4a2e8cebe2ff6ab7d1050ecd59c25d4c8bd7e6f400f5f82b96557ac0abafd0ac \\\n    --hash=sha256:4ad1906908f2f5ae4e5a8ddfce73c320c2a1429ec52eafd27138b7f1cbe341c9 \\\n    --hash=sha256:501a031947e3a9025ed4405a168e6ef5ae3126c59f90ce0cd6f2bfc477be31b7 \\\n    --hash=sha256:5190d403f121660ce8d1d2c1bb2ef1bd05b5f68533fc5c2ea899bd15f4399b35 \\\n    --hash=sha256:5498cd1645aa724a7c71c8f378eb29ebe23da2fc0d7a08071d89469bf1d2defb \\\n    --hash=sha256:5cf4e27da7e3fbed4d6c3d8e797387aaad68102272f8f9752883bc32d61cb87b \\\n    --hash=sha256:5e0b74767e5f8c593e8c9b5912019159ed0533c70051e9cce3e8b6aa699fcd69 \\\n    --hash=sha256:5ed875a24292240029e4483f9d4a4b8a1ae08843b9c54f43fcc11e404532a8a5 \\\n    --hash=sha256:5fcd34e47f6e0b794d17de1b4ff496c00986e1c83f7ab2fb8fcfe9616ff7477b \\\n    --hash=sha256:5fdec68f91a0c6739b380c83b951e2c72ac0197ace422360e6d5a959d8d97b2c \\\n    --hash=sha256:6344df0d5755a2c9a276d4473ae6b90647e216ab4757f8426893b5dd2ac3f369 \\\n    --hash=sha256:64386e5e707d03a7e172c0701abfb7e10f0fb753ee1d773128192742712a98fd \\\n    --hash=sha256:652cb6edd41e718550aad172851962662ff2681490a8a711af6a4d288dd96824 \\\n    --hash=sha256:66291b10affd76d76f54fad28e22e51719ef9ba22b29e1d7d03d6777a9174198 \\\n    --hash=sha256:66e1674c3ef6f541c35191caae2d429b967b99e02040f5ba928632d9a7f0f065 \\\n    --hash=sha256:6adc77889b628398debc7b65c073bcb99c4a0237b248cacaf3fe8a557563ef6c \\\n    --hash=sha256:79005a0d97d5ddabfeeea4cf676af11e647e41d81c9a7722a193022accdb6b7c \\\n    --hash=sha256:7c6610def4f163542a622a73fb39f534f8c101d690126992300bf3207eab9764 \\\n    --hash=sha256:7f047e29dcae44602496db43be01ad42fc6f1cc0d8cd6c83d342306c32270196 \\\n    --hash=sha256:8098f252adfa6c80ab48096053f512f2321f0b998f98150cea9bd23d83e1467b \\\n    --hash=sha256:850774a7879607d3a6f50d36d04f00ee69e7fc816450e5f7e58d7f17f1ae5c00 \\\n    --hash=sha256:8d1fab6bb153a416f9aeb4b8763bc0f22a5586065f86f7664fc23339fc1c1fac \\\n    --hash=sha256:8da9669d359f02c0b91ccc01cac4a67f16afec0dac22c2ad09f46bee0697eba8 \\\n    --hash=sha256:8dc52c23056b9ddd46818a57b78404882310fb473d63f17b07d5c40421e47f8e \\\n    --hash=sha256:9149cad251584d5fb4981be1ecde53a1ca46c891a79788c0df828d2f166bda28 \\\n    --hash=sha256:93dda82c9c22deb0a405ea4dc5f2d0cda384168e466364dec6255b293923b2f3 \\\n    --hash=sha256:96b533f0e99f6579b3d4d4995707cf36df9100d67e0c8303a0c55b27b5f99bc5 \\\n    --hash=sha256:9c57bb8c96f6d1808c030b1687b9b5fb476abaa47f0db9c0101f5e9f394e97f4 \\\n    --hash=sha256:9c7708761fccb9397fe64bbc0395abcae8c4bf7b0eac081e12b809bf47700d0b \\\n    --hash=sha256:9f3bfb4965eb874431221a3ff3fdcddc7e74e3b07799e0e84ca4a0f867d449bf \\\n    --hash=sha256:a33284e20b78bd4a18c8c2282d549d10bc8408a2a7ff57653c0cf0b9be0afce5 \\\n    --hash=sha256:a80cb027f6b349846a3bf6d73b5e95e782175e52f22108cfa17876aaeff93702 \\\n    --hash=sha256:b30236e45cf30d2b8e7b3e85881719e98507abed1011bf463a8fa23e9c3e98a8 \\\n    --hash=sha256:b3bc83488de33889877a0f2543ade9f70c67d66d9ebb4ac959502e12de895788 \\\n    --hash=sha256:b865addae83924361678b652338317d1bd7e79b1f4596f96b96c77a5a34b34da \\\n    --hash=sha256:b8bb0864c5a28024fac8a632c443c87c5aa6f215c0b126c449ae1a150412f31d \\\n    --hash=sha256:ba1cc08a7ccde2d2ec775841541641e4548226580ab850948cbfda66a1befcdc \\\n    --hash=sha256:bdb2c67c6c1390b63c6ff89f210c8fd09d9a1217a465701eac7316313c915e4c \\\n    --hash=sha256:c1ff362665ae507275af2853520967820d9124984e0f7466736aea23d8611fba \\\n    --hash=sha256:c2514fceb77bc5e7a2f7adfaa1feb2fb311607c9cb518dbc378688ec73d8292f \\\n    --hash=sha256:c3355370a2c156cffb25e876646f149d5d68f5e0a3ce86a5084dd0b64a994917 \\\n    --hash=sha256:c458b6d084f9b935061bc36216e8a69a7e293a2f1e68bf956dcd9e6cbcd143f5 \\\n    --hash=sha256:d0eae10f8159e8fdad514efdc92d74fd8d682c933a6dd088030f3834bc8e6b26 \\\n    --hash=sha256:d76623373421df22fb4cf8817020cbb7ef15c725b9d5e45f17e189bfc384190f \\\n    --hash=sha256:ebc55a14a21cb14062aa4162f906cd962b28e2e9ea38f9b4391244cd8de4ae0b \\\n    --hash=sha256:eda16858a3cab07b80edaf74336ece1f986ba330fdb8ee0d6c0d68fe82bc96be \\\n    --hash=sha256:ee2922902c45ae8ccada2c5b501ab86c36525b883eff4255313a253a3160861c \\\n    --hash=sha256:efd7b85f94a6f21e4932043973a7ba2613b059c4a000551892ac9f1d11f5baf3 \\\n    --hash=sha256:f7057c9a337546edc7973c0d3ba84ddcdf0daa14533c2065749c9075001090e6 \\\n    --hash=sha256:fa160448684b4e94d80416c0fa4aac48967a969efe22931448d853ada8baf926 \\\n    --hash=sha256:fc09d0aa354569bc501d4e787133afc08552722d3ab34836a80547331bb5d4a0\n    # via\n    #   huggingface-hub\n    #   transformers\nregex==2026.9.29 \\\n    --hash=sha256:01000ddf0e3ffef97f2413ceb514f6313040106b6d18a03ee00a4fe35c1eb1db \\\n    --hash=sha256:0166844493626c5015c6088ee15c9ca2fd060ca15b7641d1657da6a58432ae33 \\\n    --hash=sha256:044265d77d94f5e3cb2fd72c76723807c429cb8c533e9d4672d0334a6f14f588 \\\n    --hash=sha256:0476e5bcbe6e1ba3d1c4cc7bbb1c3ba78e3b979b5c8a88d0a6a8cdd4992b8c84 \\\n    --hash=sha256:066d0e3dbfdd739bce2bf8c2a41dd16f73e3d8adc2eb06dd803a36a307f56075 \\\n    --hash=sha256:0b65c72739f981377c9c22e0c5c3cd7f42da7bd8a3c9209330fac772c7d893ed \\\n    --hash=sha256:0c992c19cd45058a4b92f68f139c93db168b48fb1f322c9a7cd620806afb6b51 \\\n    --hash=sha256:0cc63b5e47c12a48d90c7e9d7de6a035dd14f62868aaedbb4e0ff8ba2b8bfe7b \\\n    --hash=sha256:0dd8af32e9f7b56b7f95cc1fd79b23054c3bdc172392ae560acc24d57b7ffe71 \\\n    --hash=sha256:0def9fb6abac55492d6d51cddb7225d07d6f279e774e0adc08569a54a5fc8d46 \\\n    --hash=sha256:0fd2c901cc307a745ad4bc87f20060d7a0825a3371d1e93488af22e7a387f78f \\\n    --hash=sha256:1043aedf5917caa861bcb25a9c11460049656bdf0017a90a309fa8f255467725 \\\n    --hash=sha256:121a76a0985db80ceae9e171c337f8c927868e37d01b54e3ce87bc87f9c6a208 \\\n    --hash=sha256:143533cc4b6fbc5b95aca0a5b8d541088d374831593def000ec89322c220221d \\\n    --hash=sha256:14e953ff3607c92d7675bf79c4d4509ef6782aa8c08509f179f9b3d6d0679e86 \\\n    --hash=sha256:18ae8eed4526e35bdb754d61562b90bf5c00a67fdcf3cc1380dd59597486631b \\\n    --hash=sha256:19959129885356df0e97556856f77eb2888380dac18bed075a7c05c5128c618d \\\n    --hash=sha256:1ba8c6a416569ce0d37e83e28a254a61dc99a419084dfb6476cea02d997f74fa \\\n    --hash=sha256:1c2a0026062abcc321a53db4a185ceba0b59a66b5d37b0808917a88b55a5257f \\\n    --hash=sha256:1d9fe8091b2e89d470df68a9331111ed008ae8aae6bf1e8e1fba4086a495c84e \\\n    --hash=sha256:2089fe39c406784d90101c726755ffa1497bb74638fd434300d2b88006186de8 \\\n    --hash=sha256:23ae6fdad9e63e54038f5ef78aba2933faca61e24d432786589e737bc5522ebb \\\n    --hash=sha256:26ec4ccce55aa533fbd603d08911b01101a8fcfec987845ac3ae2c7087b2bde3 \\\n    --hash=sha256:2f7f7aa47b229f2b39a2ae2596d2ad5625d77b5eb9856fac2dab3eb506cdd0a0 \\\n    --hash=sha256:31b003f9a070335e2a8233ee9b14a3ca8e6d792012ae011f741bf0aaf11744c5 \\\n    --hash=sha256:32ab11df9677ca80bcbb5fe4eb1da9109a5019239a054836efc6fa1c64e683cf \\\n    --hash=sha256:33026515aebc0e70d1c89978e53e8d695d35d9e472f8d5b34465ba3c74028650 \\\n    --hash=sha256:34b6925af9853bf461950e6508910f179fd6e9b1a7ec8548e069606b7e51a26b \\\n    --hash=sha256:352cf115a810b357caa35193ab656ecf5ef41056855e82f292c99e8514f8d954 \\\n    --hash=sha256:39ab5894d971f9ac68baa6eca5c50387db579cfcacf36ae8df3feceb1815e6d0 \\\n    --hash=sha256:3a21a9509d0ee88e7a70e1ad228cd2f0e0fd1e187458db132e8a8d18c97daf9d \\\n    --hash=sha256:3c5c2ef13797466aa64170cbb66ad98a32351dd4127694cea7199f80f213750d \\\n    --hash=sha256:3e778bfccd63075167709136afbc251c1f683758d5bf49c803c60ac3f894ce6b \\\n    --hash=sha256:3f1e6cb402a89457582cd696f982559217d13484a193202c394015297968c86d \\\n    --hash=sha256:42e82e578c904445d4c8a35b8f28052cf567593215fa5db06266fbc6f77aaa2e \\\n    --hash=sha256:4408b2b27a95ca8cc48b7411945753773353b5c93b307754781086c99d3a576f \\\n    --hash=sha256:446654b29bfaa30500d80947eda42cef1449dc8a87f4e3cf061cc8485d3a1f0b \\\n    --hash=sha256:45010bcfe66df41522d56c9b6114e87ecc597a08970ff6a2ced24415c141ae5f \\\n    --hash=sha256:49ee178ca31c94621294bf9b8b676a92a2e6bba8af0529591753719e57edb621 \\\n    --hash=sha256:4d7d93613b01b0199961330e49cfc52d479b3d5776c56c691db31130c0a07d91 \\\n    --hash=sha256:4fb41211d2333eb930a51e0546a65999761cf1f572a4da56ef9b8a62966c06f2 \\\n    --hash=sha256:4fe97894d1b306c919b4e50def1e6f6c522f4d03a7283811f4d108f1ce5d3ac2 \\\n    --hash=sha256:554bffadcbcb6d5f4e5fb10a61cc52084b9a63d1dab5f10bcd2c4343972e8e2c \\\n    --hash=sha256:59b49507f47479e299a9e1bc41b5cb83a7afda0540625f1dbae886615978acbf \\\n    --hash=sha256:5eeb8edc6110d9194a4d0d54610f64c37a31c605b5dbb7e407fc6ec7fa34a4a1 \\\n    --hash=sha256:612b709381c0355b70d89cdb51b7f670591ed5cbbc0e3b5337488019dc667b65 \\\n    --hash=sha256:61956f074ecd123f55adca68ee3eab46e6a07ad3f8e64e6db95dfacb444f55c4 \\\n    --hash=sha256:6398d5145689503412cc1748895242598d8846b8967b851133b20dc2ed1e21e8 \\\n    --hash=sha256:65b408d8fcb273e3499e7ef2ce796810da1becd208c7fb4373692a242d79d461 \\\n    --hash=sha256:686ac5350fceae63830bb98805fcb8039325bf4c06d9f6f048ff65229d5bffa5 \\\n    --hash=sha256:6a1a824fbed817e0a891103886b68f063b1e83cc51bc97192a90a60195a9291f \\\n    --hash=sha256:6abb75ab16bc3281714a5b99548a2225db70dba1f995f6d7f7419b76eb5a8fbe \\\n    --hash=sha256:6f7121a8914ed13fcfe2099f895341bfb789f004d4c5a0bdece8fa667da10849 \\\n    --hash=sha256:7020ed44df30b3aa492c00ee3b52d0548c1f30c2c6c5bb13ae897680900d3413 \\\n    --hash=sha256:720537c7ea6f80dc61913184edb0ce2497a306b39ef19f28505b322553d52bdb \\\n    --hash=sha256:724184b4aafed865e4f13ca313fdcb43024300c028ec67319cfa16847d84685e \\\n    --hash=sha256:7c03031610e3e6ed1768a2b7a8fc84637c1257b50c5eacaf094c6e17a84fc563 \\\n    --hash=sha256:80a5ea3b4fd9d6a5b9a44f7976a9acaaab35aa3c1f6b29e5bd857dfabaded223 \\\n    --hash=sha256:80c7cadd3fd2bfde5df8aa0787e315812cad0c313a753095d02f4c2b6c01677b \\\n    --hash=sha256:80ea96f5c1a30bf09007d48466521d9c294bebe197c708c3359096e3e3691632 \\\n    --hash=sha256:864e9b87ac33c3fb9fb4ad48166d4fdb579c351d5c77deb0d34bccb36a775cd9 \\\n    --hash=sha256:87fb80cbe3557e27e7b28b995c2b2eedf689b8886f941ab93e0e288f0976518a \\\n    --hash=sha256:8873c4a11c50b9989168881aeb3f08859f469d809941866aa1feefd8be5431f6 \\\n    --hash=sha256:888d60953908dcf761aa320c3e390ab8556efbdb551ace63921de90f6ae0848d \\\n    --hash=sha256:8b5fcc4771732191b2b7d1dd68d8f0353f47f8d90b6150f6dce58bf1112442cb \\\n    --hash=sha256:8f39588af4731c8923c26810eb3b33f76f17633985e40f59c3cd45a33805a895 \\\n    --hash=sha256:9173db3be74a35cb6731701094b98120f7ee4876a287882a59cdea1fa7da342f \\\n    --hash=sha256:92f05c9c42bde5785dc48770bc2194d9f7442544156f951e19cd31b096cec562 \\\n    --hash=sha256:951733b1bbdb71e377cec567b409f1a7881b47cfcad84121aa74cb575fa425ea \\\n    --hash=sha256:957bb708e8057ab1649ba566456429d691ec9b90d1c9ad1af1ba7ffbbeaf05f2 \\\n    --hash=sha256:9916fda742cd4eede63b286f58c06718324265d727ce0856eb1aac86d0d150d6 \\\n    --hash=sha256:9e1d3a4cb7993b708f0ada8d0c84590efd853f169e7147d2202c9da503180242 \\\n    --hash=sha256:9e4482589065c8ecd761cff522dcd85f2d39e62f551e37e025d1c7d54772def3 \\\n    --hash=sha256:a5300757f8a68f5b6cc33f57338d72a0e3589c5cc9ad5f8504ea06f028be582a \\\n    --hash=sha256:a540abfab208e1b7ef2df231c40ef3b6cbb30a0aad6204e9b6a81c10a6794628 \\\n    --hash=sha256:a5758353650079898dc1b2b0e95aa51fa23a30d020e06f62c430dd08ee56cdd8 \\\n    --hash=sha256:a64b85a4760337cfefdb27d42da6ed8b58e8cde3f2d57b6ef43e76ef6ea9ef47 \\\n    --hash=sha256:a655d34b2a6943af32401f3d94f72e9d731f6ad16285815550bf2b4ee69d420a \\\n    --hash=sha256:a714befaacbd10092ffe4cea0d3c5f008fb9efe9bc322c715bcdfdee414b9a3d \\\n    --hash=sha256:a760da040b47767b4b873adfb7c3b691e9ba2fc60f113f9d0b88f1a62f323e85 \\\n    --hash=sha256:addd736a0547d553283adaf4e05d7104e7f2c7b0b092e9b4d28756825f14531f \\\n    --hash=sha256:ae4613d7d9dda60fcba95f846cc6f808017f1843f392cf9daad14a6534493d71 \\\n    --hash=sha256:b11b589e00095ec69cf79841a76360f9b079e95b0368a25b5ebb951ab0c157ff \\\n    --hash=sha256:b3e445b66c80b4eb4234e855ce94d9adc183eedbd632816228d89930b91b2c5b \\\n    --hash=sha256:b7b893976e7fe42053da64f2aa27239c24252fd2ec6df471e1be197c0addc3b1 \\\n    --hash=sha256:b84f186a7f0536fe4ff9a9fa12d06d007b9b71d4b5352ddcc41f59ad6522a312 \\\n    --hash=sha256:b89efc38431793d28b7cd91227e2f952ad7c48df19132b17f43a5fec3c14143b \\\n    --hash=sha256:b97a38fb4c732b6832db6bf108963adbcd82ef1268ba2025dce390f45af75efa \\\n    --hash=sha256:b9d74e4eee9ddb64c2e92d5d61472c59c21684c059eb7b68767be9628e977859 \\\n    --hash=sha256:bb90e7177944b6684738c1fc36aabd2dd00d1de3be7dbe09f91e196f1bc0dc81 \\\n    --hash=sha256:bec37990e3d6121f29ecfb594bd8f1bf009e9f7926daba2e50e3b27d3892a783 \\\n    --hash=sha256:bf3c49863c23a1ad6da9c30351aed6cff8d5ddbeb63c5c8420ae54e98c7d0138 \\\n    --hash=sha256:bf48516e35cf848390ea68850aba53e7c333720d2945b4d2c25b69fc5171723f \\\n    --hash=sha256:bfc71e6d970419c1309b3640305298643e2a734cad3f7cfb6d2ddee4175ab53d \\\n    --hash=sha256:c0094897d7d01f184b2d7fe8c56c66d64efe01b31f4b7d34205b391387df1111 \\\n    --hash=sha256:c03c6eb6ece86dfdcbb34799efaa339b093132e1aceed491ba5e08fe06cdf699 \\\n    --hash=sha256:c1a9a6651197fbed6f0212591418b9def774fc3f8324f78d1bf0e6a63e5f8aa1 \\\n    --hash=sha256:c3589f40749acce747510bf5d589d54e376cb0930ea58b35effac97e5312b0c1 \\\n    --hash=sha256:c4e38dd8f39c43a91d2410ad2b85610701b0979342c3df1d69eaf8e838c757d8 \\\n    --hash=sha256:c6c8fabf1dafc1f1ddcbb67896d3f93efb092e8c4b6322d7389b944e76a484e5 \\\n    --hash=sha256:c90fcf7804ea0a54b896ce0f2b9565350220b8d4890fd0db461a476a4c687963 \\\n    --hash=sha256:c9b602fae1e00b7c035d661ce85575365719192a7b46784bd71cf64c68053aa0 \\\n    --hash=sha256:ccb64d887a9db1cd76dbc0f92051a1a478a2a67e7f56c62d915cb881d7734704 \\\n    --hash=sha256:d06fcdecc10fc7954d7c8f27a03c96055fe525274dc84a7b0dbdc3d6b9e03dab \\\n    --hash=sha256:d0c3082bf79bcd6a614d55916590ad4b8f93200e10b97f463ea5d9d07c9b5f23 \\\n    --hash=sha256:d49c18f1ea294cf4adde2e5ac256e98c82ea9d708462ce4bf799dffa7cfe8a2c \\\n    --hash=sha256:d60030baaa7bfbb02d650c126cdcddcb6e33dbff14d819434c8fa2fdcaeeeba5 \\\n    --hash=sha256:d7cab119d0df0b9413f106b4d7fc34f2872d3574ed3806fb48959c830b1537da \\\n    --hash=sha256:d9b77b25b4f395f92de6099ab08e8ae2bc7e51dfe157f22900902243a5cc90c7 \\\n    --hash=sha256:dabee8f4935e731fb46b2a3091bdda0d3d94b3bbfb907d2b4f12eefce4009619 \\\n    --hash=sha256:db5e82ba15c142425b8406690032df89e39cca4a2e8afbbb9a3d84edc2373ac3 \\\n    --hash=sha256:dc79d36d0618752265f0d575915bdc5c5130ecb9c9f6b3bcefeae32e4bdfafcf \\\n    --hash=sha256:ddfa987262763c3c22a8367d2a49c244b018a74c3a8e3ab1a864119ad45c5633 \\\n    --hash=sha256:e1172147d28d8fbcf8cb8d26c41506169f5ad8fe9ec969cb116835a19d4d8eca \\\n    --hash=sha256:e11edba5bc344a32b029a7af9d4b3173982dd79eeafa0b9dbd787364414b0509 \\\n    --hash=sha256:e2c89e9b762c57f59d5e99ee8b20202adb892e35f8d3485741340999ca55058e \\\n    --hash=sha256:e31f72490b7c12f7790e1e25c3afffd20503ee1bfb43461d7838b871ff244b19 \\\n    --hash=sha256:e8c65ef3862a8ad6e86492b6ed9327805dd66904c012bd3649dc67d822ed6c34 \\\n    --hash=sha256:ebb8912f565b8cdbbf27debfe00df04202c20e2f651b9e32767930c5eace3621 \\\n    --hash=sha256:ed511a0708e2297e1d6431e7fb217e3402791e491e02da800658ace4973df1bb \\\n    --hash=sha256:edf06545875f3efa31560d94121e95c7fd70d98b1dfedc0157097d79b13b52ea \\\n    --hash=sha256:f0fe9834e5aeccaf19a0d8feb296d66a24be1a7c9922002f842a682cd5abb787 \\\n    --hash=sha256:f1a0d5117230dd46b399a30a38afa44f79c99f3168988fdc4f425c3f928b39df \\\n    --hash=sha256:f37964e4a5e993d2fd45147741e9dff7f34a2d8c00ab94c4ea0514a4677f959e \\\n    --hash=sha256:f57dc6b8fef170f105d2cf5cdce254f47b137d7755086cf7050f47e16582abba \\\n    --hash=sha256:f93bc1c3486ef3747e07c9d7c1d0a147b8fbaab975f80e348aed6f71309dfaca \\\n    --hash=sha256:fb00027a09a8f9f08028b40dce4c933cf73e4833240ed356583fdc9cfa721566 \\\n    --hash=sha256:fb99cc9d45f48895d9d67f6a0b8a57f08d39c174d9f25ad97a313e0470267b1c \\\n    --hash=sha256:fdd88ed5e20b1bcdd234421e454962c971aa44b653bdb7f1ea9ef683e90fb649 \\\n    --hash=sha256:fe3fa1dd453ed5c7f5ea23a26218329790ed7197a99b90e94330e313959a7f52\n    # via transformers\nrequests==2.34.2 \\\n    --hash=sha256:2a0d60c172f83ac6ab31e4554906c0f3b3588d37b5cb939b1c061f4907e278e0 \\\n    --hash=sha256:f288924cae4e29463698d6d60bc6a4da69c89185ad1e0bcc4104f584e960b9ed\n    # via\n    #   huggingface-hub\n    #   transformers\nsafetensors==0.8.0 \\\n    --hash=sha256:040070828e36dc8e122178bbbd5830ff9e97920affb84cbe0f46442497bed358 \\\n    --hash=sha256:096ec1a98435df7beb08853bb5aa9081a84f23d0adc67ed1a0a10550f608373f \\\n    --hash=sha256:2ddf52eac562eda224f99acfa7889d02968c1fd59a5b011ae7d8137c37e9c02d \\\n    --hash=sha256:3ae091f16662658bdc019a4ff6cb4c085bb7d725eb5978b183ffd265863b6d2d \\\n    --hash=sha256:4124502b78f03534117c848f87a39b8f31e577b15eff423bf8bfb95f2a8c30d0 \\\n    --hash=sha256:4a95ae2b05d7726d751da4ebf626a2ca782b706e101bd894c95bc2450b1cffcc \\\n    --hash=sha256:7a46e5ff292c356d6991e60942ba7f79817682d3a2cef0702136448cb9c4d235 \\\n    --hash=sha256:7bc0a787ba8a35be368ee3574edfa2b1ad389eebd0a72e482ae275490e3f6c98 \\\n    --hash=sha256:87eec7ffed2b809f05a398a8becb7d013f19f7837cd15d9748580d6cf30dbaf4 \\\n    --hash=sha256:8e080062fcde23be189565e1c3305d16751a218ecf9412c8601e64204eb6f846 \\\n    --hash=sha256:8e9f537aa183a38ace122d27303dcd986b26bd2a7591f9181d7f0c396f4677ca \\\n    --hash=sha256:c554f85858e05226d3c2828e32395e677434685d6d94594a41643361c5e837f0 \\\n    --hash=sha256:c80201d22cbf405b80647a60ada77bba06c8fba2da2743ba1e89cdcc39a81f25 \\\n    --hash=sha256:f7838e5135a406ad3e02efdcb8cf2e5397d368b0154537c4fec682dbc544d452 \\\n    --hash=sha256:fabaf3e0f18a6618d9b36560682562157f77c2b71fcffc7b432be2baed9d753d \\\n    --hash=sha256:fcdd41ec4628fee5799f807c73c353629130fbd942aa23d83c623dd6c9d52d78 \\\n    --hash=sha256:fd6f3f93c9a0a7cc2788ee63fb763353d4bd2e89b0751bc78fcf7dda00bea774\n    # via\n    #   ast-audio-classification-pipeline (pyproject.toml)\n    #   transformers\nsetuptools==84.0.0 \\\n    --hash=sha256:51a52592b3b99e102b609654876bd65f19f999935166d1352678931132b0c670 \\\n    --hash=sha256:f4695c21257f0d9b537ec2692c941d02ee143b7cc1276941349a546573b2ef73\n    # via torch\nsympy==1.14.0 \\\n    --hash=sha256:d3d3fe8df1e5a0b42f0e7bdf50541697dbe7d23746e894990c030e2b05e72517 \\\n    --hash=sha256:e091cc3e99d2141a0ba2847328f5479b05d94a6635cb96148ccb3f34671bd8f5\n    # via torch\ntokenizers==0.22.2 \\\n    --hash=sha256:143b999bdc46d10febb15cbffb4207ddd1f410e2c755857b5a0797961bbdc113 \\\n    --hash=sha256:1a62ba2c5faa2dd175aaeed7b15abf18d20266189fb3406c5d0550dd34dd5f37 \\\n    --hash=sha256:1c774b1276f71e1ef716e5486f21e76333464f47bece56bbd554485982a9e03e \\\n    --hash=sha256:1e418a55456beedca4621dbab65a318981467a2b188e982a23e117f115ce5001 \\\n    --hash=sha256:1e50f8554d504f617d9e9d6e4c2c2884a12b388a97c5c77f0bc6cf4cd032feee \\\n    --hash=sha256:2249487018adec45d6e3554c71d46eb39fa8ea67156c640f7513eb26f318cec7 \\\n    --hash=sha256:25b85325d0815e86e0bac263506dd114578953b7b53d7de09a6485e4a160a7dd \\\n    --hash=sha256:29c30b83d8dcd061078b05ae0cb94d3c710555fbb44861139f9f83dcca3dc3e4 \\\n    --hash=sha256:319f659ee992222f04e58f84cbf407cfa66a65fe3a8de44e8ad2bc53e7d99012 \\\n    --hash=sha256:369cc9fc8cc10cb24143873a0d95438bb8ee257bb80c71989e3ee290e8d72c67 \\\n    --hash=sha256:37ae80a28c1d3265bb1f22464c856bd23c02a05bb211e56d0c5301a435be6c1a \\\n    --hash=sha256:38337540fbbddff8e999d59970f3c6f35a82de10053206a7562f1ea02d046fa5 \\\n    --hash=sha256:473b83b915e547aa366d1eee11806deaf419e17be16310ac0a14077f1e28f917 \\\n    --hash=sha256:544dd704ae7238755d790de45ba8da072e9af3eea688f698b137915ae959281c \\\n    --hash=sha256:64d94e84f6660764e64e7e0b22baa72f6cd942279fdbb21d46abd70d179f0195 \\\n    --hash=sha256:753d47ebd4542742ef9261d9da92cd545b2cacbb48349a1225466745bb866ec4 \\\n    --hash=sha256:791135ee325f2336f498590eb2f11dc5c295232f288e75c99a36c5dbce63088a \\\n    --hash=sha256:9ce725d22864a1e965217204946f830c37876eee3b2ba6fc6255e8e903d5fcbc \\\n    --hash=sha256:a6bf3f88c554a2b653af81f3204491c818ae2ac6fbc09e76ef4773351292bc92 \\\n    --hash=sha256:bfb88f22a209ff7b40a576d5324bf8286b519d7358663db21d6246fb17eea2d5 \\\n    --hash=sha256:c9ea31edff2968b44a88f97d784c2f16dc0729b8b143ed004699ebca91f05c48 \\\n    --hash=sha256:df6c4265b289083bf710dff49bc51ef252f9d5be33a45ee2bed151114a56207b \\\n    --hash=sha256:e10bf9113d209be7cd046d40fbabbaf3278ff6d18eb4da4c500443185dc1896c \\\n    --hash=sha256:f01a9c019878532f98927d2bacb79bbb404b43d3437455522a00a30718cdedb5\n    # via transformers\ntorch==2.14.0 \\\n    --hash=sha256:0e7cf18cb0d8bd666b6120932e29c7aef3502b61a08b44da4839580c539a7cdb \\\n    --hash=sha256:2fb30099be1fec9d163a6dd05ace176305ce8cf914b4d7a5f4f6d7a326aa4b60 \\\n    --hash=sha256:44b044b9f6f633d982839422a57433d6a1da520037fd88e0c8a47efde589b3b8 \\\n    --hash=sha256:4dbea8a10d80b10dff2be1ed467d2617b704fe6bf76bbfad4194774c8f6d886b \\\n    --hash=sha256:553aec938d37d77b783bcf801e638cad068870e7643c47eadac21eb180f551ed \\\n    --hash=sha256:6981872df75eb5409c439050d36b67cb57d88b4e11080516e2aa7410f6730705 \\\n    --hash=sha256:731784e3914843c6bcc7aba3987ff7610ac57dbbc816a5d6b9b62e04c240a641 \\\n    --hash=sha256:731b9ebdea402b8b1996d4c2ae613b16660e559b19e47bdc45d970568bc91c53 \\\n    --hash=sha256:84bf384779a10c02fc3c6bdbab71a9cb66b0dd93c652d1ed5d6dfc0cb37e5962 \\\n    --hash=sha256:860423e970f2ce02c4476e8e2d1350131b1c5c5a5e4912180e78b50b53241efa \\\n    --hash=sha256:8d9e232b6376c62f3090237889fb1cb6887c0ece9f73e6c1052e80fc1481f999 \\\n    --hash=sha256:96383c62423c3f2767023c4a94774b6435e2c006691daf396e9ddb55d412fbc6 \\\n    --hash=sha256:9d4b1022a5d9b71282ec67ad0d9e7235870096b8a246dc1c32d6ea1fc83dc998 \\\n    --hash=sha256:ada340e62591d06a2bcc2d68170f20f45f0b0665d372dc510a8ed7eb3b1d609a \\\n    --hash=sha256:ae530ddd3f3b94248b77f1fd3313c4f3405bd0d17283d505b81574816767133a \\\n    --hash=sha256:b2cd92bce63d40bf6fc2e5d840fc2f0063bd180a242daa761cb6088cc2f46e27 \\\n    --hash=sha256:b985c7defeb8d28691b7513ebaecc64946c5f68ec770e9f4ddba39688864f46a \\\n    --hash=sha256:c1f844f1c750e87df4b68bc3afbc0e2b0c7ef19d7b8f666e48bdcf6a0c4f0056 \\\n    --hash=sha256:cad84f41bbdf3dcf333ce394aeeaf25237c4d94fd6b659ba5eb813c829978823 \\\n    --hash=sha256:cd8cd8f714d511ccdca907282d1da3d9be8322d4e1520b9c3bce39a5c1318a4b \\\n    --hash=sha256:d2526f71e6638133b97b3cd2881ece3df521460a4727030e8ae2a72a7d3ae31d \\\n    --hash=sha256:d30207cb89e713dfd6afb5d592b30d9ef252acf80407c5543157ea7b07c4d9a0 \\\n    --hash=sha256:d70c2b41bde81efde59fcefb147b9247fd062497e778677661b7de1bee5c8c99 \\\n    --hash=sha256:fecffb58f51fd643d213acd68da21cc3fc19bea05a3bc64b4ee55128f47a4963\n    # via\n    #   ast-audio-classification-pipeline (pyproject.toml)\n    #   torchvision\ntorchaudio==2.11.0 \\\n    --hash=sha256:00e9f71ab9c656f0abdb40c515bd65d4658ab0ad380dee27a2efd7d51dabd3d6 \\\n    --hash=sha256:13cff988697ccbad539987599f9dc672f40c417bed67570b365e4e5002bbd096 \\\n    --hash=sha256:1424638adb8bb40087bc7b6eb103e8e4fe398210f09076f33b7b5e61501b5d66 \\\n    --hash=sha256:1a07ec72fd6f26a588c39b5f029e0130d16bb40bc4221635580bf8fb18fcbc80 \\\n    --hash=sha256:1be3767064364ae82705bdf2b15c1e8b41fea82c4cd04d47428a8684b634b6ed \\\n    --hash=sha256:1c1101c1243ef0e4063ec63298977e2d3655c15cf88d9eb0a1bd4fe2db9f47ea \\\n    --hash=sha256:478110f981e5d40a8d82221732c57a56c85a1d5895fb8fe646e86ee15eded3bd \\\n    --hash=sha256:492dd64645e9d0bb843e94f1d9a4d1e31426262ffc594fafecc1697df9df5eb9 \\\n    --hash=sha256:5847fe2022b17c6580aeb39c8797a443411cc09edfd9183cd50ac1a3b8ccf97c \\\n    --hash=sha256:6503c0bdb29daf2e6281bb70ea2dfe2c3553b782b619eb5d73bdadd8a3f7cecf \\\n    --hash=sha256:67f6edac29ed004652c11db5c19d9debb5d835695930574f564efc8bdd061bba \\\n    --hash=sha256:6ebb59c694909eccb5d61b7cc199d297692012c43286e36d92983aa7bad7586d \\\n    --hash=sha256:73dab4841f94d888bc7c2aed7b5547c643edc974306919fe1adfb65d57cccf4b \\\n    --hash=sha256:79fb3cb99169fd41bd9719647261402a164da0d105a4d81f42a3260844ec5e79 \\\n    --hash=sha256:7e2da1df4f6fe885c46db350a0dc90a0dff4b54541dff8846faa904d255e2bfe \\\n    --hash=sha256:88fb5e29f670a33d9bac6aabb1d2734460cf6e461bde5cdc352826035851b16d \\\n    --hash=sha256:986f4df5ed17b003dc52489468601720090e65f964f8bebccf90eb45bba75744 \\\n    --hash=sha256:9fe3083c62e035646483a14e180d33561bdc2eed436c9ab1259c137fb7120b4a \\\n    --hash=sha256:a1cf1acc883bee9cb906a933572fed6a8a933f86ef34e9ea7d803f72317e8c1b \\\n    --hash=sha256:b034d7672f1c415434f48ef17807f2cce47f29e8795338c751d4e596c9fbe8b5 \\\n    --hash=sha256:bb59ba4452bbbe95d75ad3ef18df9824955625f36698ce9a5998a4a9f3c1ba1d \\\n    --hash=sha256:bc653defca1c16154398517a1adc98d0fb7f1dd08e58ced217558d213c2c6e29 \\\n    --hash=sha256:bda09ea630ae7207384fb0f28c35e4f8c0d82dd6eba020b6b335ad0caa9fed49 \\\n    --hash=sha256:be7ad472acb16d16e98c005f0219b0db06a47dfe8f7b4d177062e1638f871e3b \\\n    --hash=sha256:cc09cd1f6015b8549e7fe255fb1be5346b57e7fee06541d3f3dbb012d8c4715f \\\n    --hash=sha256:da2725e250866da42a12934c9a6552f65a18b7187fd7a6221387f0e605fb3b96 \\\n    --hash=sha256:e3f9696a9ef1d49acc452159b052370c636406d072e9d8f10895fda87b591ea9 \\\n    --hash=sha256:ed404c4399ad7f172c86a47c1b25293d322d1d58e26b10b0456a86cf67d37d84\n    # via ast-audio-classification-pipeline (pyproject.toml)\ntorchvision==0.29.0 \\\n    --hash=sha256:01a029fa1b2eacac27e1e4f9e0dba24d112bcbd523261b294e4b4a1c7f8330bc \\\n    --hash=sha256:0e3b99c62f7f095153887330c2c60c9ebc2da84dfbe08eea564a41ec361629cf \\\n    --hash=sha256:0e631a2f8b24732672d35224d2574ff89a78d56bef5364ef6094a625fa5f1771 \\\n    --hash=sha256:2428a13706a354ee3901fd63e3f1ef8c2a59f6e20d24cce13210b4e61be08f7a \\\n    --hash=sha256:28a3964a9e6db34354d4ca440d2c2038deeef24e7219e8d6f49d9d174f44830c \\\n    --hash=sha256:710df2c4f03ab6da161520c10d9467f4c0201ca314fd1432499a48c9d7a5818f \\\n    --hash=sha256:784c8b8de9e81e02dea093373bc35040791378df2daec0017e657985183bf862 \\\n    --hash=sha256:80789f9b50f5277302e7020c8128344e0269b12f6e3b5855034e56bb7081b874 \\\n    --hash=sha256:83db0299170502b038640444214c99d7b54bad72db447f0554deb8d2c4f02f94 \\\n    --hash=sha256:85fa54bec1f7d9227b5e4e201ed4bac92247ee10fb19148d92f9836e3e91d5c3 \\\n    --hash=sha256:942a3b3fb4e981e1abb7846679ee3882dfc733533a30b68df0dadf6c06f238a9 \\\n    --hash=sha256:994687b818cac0e6d34cb407a41458e2c3262cb7db927465841c9522ba7ebb92 \\\n    --hash=sha256:9b99a7385da8d706f2dc14cacffb8cbf653d48fc37013a38c41a36659acc3731 \\\n    --hash=sha256:a292ca7044236a2702ec3d4664de135e01fc9fb3ae9422db3fecc819546e831c \\\n    --hash=sha256:add88479361b2bf08790338fce4ba11a11fdc1ae508e80a3a013daf1c0d2893d \\\n    --hash=sha256:b7a736eaa6b2e22476c95ceb62986d195f3e600cdd7d196d7329aa2dfc951994 \\\n    --hash=sha256:bb74d08d65785b51c61d4d4ec167bfa50a5ea644b57555a33ba2f4435b0d05dd \\\n    --hash=sha256:bbe15455b59a6c9d822584fd3faaaac50ee972101d7bb7e2887da456e148a3ea \\\n    --hash=sha256:bd12e152640d1024fca15deaf6f39c53702e12eb398db4b2b038a5a97ab85f7f \\\n    --hash=sha256:ce3b58631f0b7c8828e8675301bff5cb0623aaed704e3ac91079e89607753205 \\\n    --hash=sha256:dc5717feef9a0b430b052895db71f1d695939575b74344da981120e9f4da7620 \\\n    --hash=sha256:e8fdf234d76dca6fc47bc5f4af86ecf1ef57fab6285a7b21681bdcbb494e4c1b \\\n    --hash=sha256:ebf54f6744556ebd8d7b5f9b6515e2060e5d3156723970dc257bf093f63b2170 \\\n    --hash=sha256:f9e9627d5036cac8e6de76aeb2f1acc67841ff5ff4b63c4cf78ff954efeb073e\n    # via ast-audio-classification-pipeline (pyproject.toml)\ntqdm==4.70.1 \\\n    --hash=sha256:c293e525e6fef9c20e8728fd4612df02a0aa31bb5fe91ecd93e123b1b7bffa73 \\\n    --hash=sha256:cefd0eca11b2a37a3aee776544d4f4ae913f02688135b5556b8788dfa474afc4\n    # via\n    #   huggingface-hub\n    #   transformers\ntransformers==4.57.6 \\\n    --hash=sha256:4c9e9de11333ddfe5114bc872c9f370509198acf0b87a832a0ab9458e2bd0550 \\\n    --hash=sha256:55e44126ece9dc0a291521b7e5492b572e6ef2766338a610b9ab5afbb70689d3\n    # via ast-audio-classification-pipeline (pyproject.toml)\ntriton==3.8.0 \\\n    --hash=sha256:1b84e7d512490ba529111260fa6f7cad8b254a6bb5fbdf41d5ef9a5e57f52d0a \\\n    --hash=sha256:1f0497218e26b7d79773ad9c2a3fa3b539ee69f587a13fac2e552b1d322a8015 \\\n    --hash=sha256:1f6b48d0591929a3867973acac3dccd4e058585f91bfb41022de496c9ffab304 \\\n    --hash=sha256:372285307d4c44ee74cee32de0b4f04bd157e071427e38c6f4ee3e3beb2194f4 \\\n    --hash=sha256:387dae4cb0089a7b6ba1a428ae0782b65c4c58f57d94617cb22ca8593d8ccbca \\\n    --hash=sha256:398f4b009c7ab08ed9aeb1d1282dee822945b53f399a5e12a4fecb643cf2007d \\\n    --hash=sha256:68988ac85d5e7086baeda0ddc175af9667db7529b3c5e11a5c0601b8bef2200a \\\n    --hash=sha256:6d914c52f89dc942b1819db959e7c07a5999f678ec760f677625c706b7e0d743 \\\n    --hash=sha256:74217bb56ed8692759227758e4c4b3bd2d608a209c1a7a081bf361fb4c2c1bf9 \\\n    --hash=sha256:a9c404c69ed4a39e8ec632eaf6b9fe058a060bf98979c177f6ef666f06bb8d50 \\\n    --hash=sha256:b7004666652f500ed854a86988e4b3d69d247188b5d2092b5df1e44f4a954099 \\\n    --hash=sha256:e91ffa46d095b252248297292dd22bcbacd53a125a0c2eefbbbf74925a320bc3\n    # via torch\ntyping-extensions==4.16.0 \\\n    --hash=sha256:481caa481374e813c1b176ada14e97f1f67a4539ce9cfeb3f350d78d6370c2e8 \\\n    --hash=sha256:dc983d19a509c94dba722ee6abd33940f7c05a89e243c47e907eb4db6f1a43e5\n    # via\n    #   huggingface-hub\n    #   torch\nurllib3==2.8.0 \\\n    --hash=sha256:0cf3cae568d36aa9576b28dfb35f11328f1cb974ca7647d9475ebb86c75ac6e3 \\\n    --hash=sha256:63bf2ead4c879426ebf22ef2a781eeb4aa3b4ae798a0435506f8687fd5bb9b63\n    # via requests\n', 'weights/ast-audioset/dimer-base-manifest.json': '{\n  "format": "dimer_hf_snapshot",\n  "formatVersion": 1,\n  "modelKey": "ast-audioset",\n  "modelId": "MIT/ast-finetuned-audioset-10-10-0.4593",\n  "revision": "f826b80d28226b62986cc218e5cec390b1096902",\n  "files": [\n    {\n      "path": "README.md",\n      "bytes": 1165,\n      "sha256": "dbc8ce1fc5abd1635d073640d8cb032901bfe264a5e06507f363f10a6112cfc8"\n    },\n    {\n      "path": "config.json",\n      "bytes": 26763,\n      "sha256": "a93d525511d77e8ecc933d09674b85099815bbbb417c228a4edd655e252fb9ff"\n    },\n    {\n      "path": "model.safetensors",\n      "bytes": 346404948,\n      "sha256": "ae0c1e2ad4e1381d851fa9bf298ba13ebc9c5a914cdee2dbe427a6583869924d"\n    },\n    {\n      "path": "preprocessor_config.json",\n      "bytes": 297,\n      "sha256": "8d04ba5a9c6fca5d39d0de2b1fd05ecf79deb589fbba279728bbebac39934231"\n    }\n  ],\n  "totalBytes": 346433173\n}', 'LICENSE': '                                 Apache License\n                           Version 2.0, January 2004\n                        http://www.apache.org/licenses/\n\n   TERMS AND CONDITIONS FOR USE, REPRODUCTION, AND DISTRIBUTION\n\n   1. Definitions.\n\n      "License" shall mean the terms and conditions for use, reproduction,\n      and distribution as defined by Sections 1 through 9 of this document.\n\n      "Licensor" shall mean the copyright owner or entity authorized by\n      the copyright owner that is granting the License.\n\n      "Legal Entity" shall mean the union of the acting entity and all\n      other entities that control, are controlled by, or are under common\n      control with that entity. For the purposes of this definition,\n      "control" means (i) the power, direct or indirect, to cause the\n      direction or management of such entity, whether by contract or\n      otherwise, or (ii) ownership of fifty percent (50%) or more of the\n      outstanding shares, or (iii) beneficial ownership of such entity.\n\n      "You" (or "Your") shall mean an individual or Legal Entity\n      exercising permissions granted by this License.\n\n      "Source" form shall mean the preferred form for making modifications,\n      including but not limited to software source code, documentation\n      source, and configuration files.\n\n      "Object" form shall mean any form resulting from mechanical\n      transformation or translation of a Source form, including but\n      not limited to compiled object code, generated documentation,\n      and conversions to other media types.\n\n      "Work" shall mean the work of authorship, whether in Source or\n      Object form, made available under the License, as indicated by a\n      copyright notice that is included in or attached to the work\n      (an example is provided in the Appendix below).\n\n      "Derivative Works" shall mean any work, whether in Source or Object\n      form, that is based on (or derived from) the Work and for which the\n      editorial revisions, annotations, elaborations, or other modifications\n      represent, as a whole, an original work of authorship. For the purposes\n      of this License, Derivative Works shall not include works that remain\n      separable from, or merely link (or bind by name) to the interfaces of,\n      the Work and Derivative Works thereof.\n\n      "Contribution" shall mean any work of authorship, including\n      the original version of the Work and any modifications or additions\n      to that Work or Derivative Works thereof, that is intentionally\n      submitted to Licensor for inclusion in the Work by the copyright owner\n      or by an individual or Legal Entity authorized to submit on behalf of\n      the copyright owner. For the purposes of this definition, "submitted"\n      means any form of electronic, verbal, or written communication sent\n      to the Licensor or its representatives, including but not limited to\n      communication on electronic mailing lists, source code control systems,\n      and issue tracking systems that are managed by, or on behalf of, the\n      Licensor for the purpose of discussing and improving the Work, but\n      excluding communication that is conspicuously marked or otherwise\n      designated in writing by the copyright owner as "Not a Contribution."\n\n      "Contributor" shall mean Licensor and any individual or Legal Entity\n      on behalf of whom a Contribution has been received by Licensor and\n      subsequently incorporated within the Work.\n\n   2. Grant of Copyright License. Subject to the terms and conditions of\n      this License, each Contributor hereby grants to You a perpetual,\n      worldwide, non-exclusive, no-charge, royalty-free, irrevocable\n      copyright license to reproduce, prepare Derivative Works of,\n      publicly display, publicly perform, sublicense, and distribute the\n      Work and such Derivative Works in Source or Object form.\n\n   3. Grant of Patent License. Subject to the terms and conditions of\n      this License, each Contributor hereby grants to You a perpetual,\n      worldwide, non-exclusive, no-charge, royalty-free, irrevocable\n      (except as stated in this section) patent license to make, have made,\n      use, offer to sell, sell, import, and otherwise transfer the Work,\n      where such license applies only to those patent claims licensable\n      by such Contributor that are necessarily infringed by their\n      Contribution(s) alone or by combination of their Contribution(s)\n      with the Work to which such Contribution(s) was submitted. If You\n      institute patent litigation against any entity (including a\n      cross-claim or counterclaim in a lawsuit) alleging that the Work\n      or a Contribution incorporated within the Work constitutes direct\n      or contributory patent infringement, then any patent licenses\n      granted to You under this License for that Work shall terminate\n      as of the date such litigation is filed.\n\n   4. Redistribution. You may reproduce and distribute copies of the\n      Work or Derivative Works thereof in any medium, with or without\n      modifications, and in Source or Object form, provided that You\n      meet the following conditions:\n\n      (a) You must give any other recipients of the Work or Derivative\n          Works a copy of this License; and\n\n      (b) You must cause any modified files to carry prominent notices\n          stating that You changed the files; and\n\n      (c) You must retain, in the Source form of any Derivative Works\n          that You distribute, all copyright, patent, trademark, and\n          attribution notices from the Source form of the Work,\n          excluding those notices that do not pertain to any part of\n          the Derivative Works; and\n\n      (d) If the Work includes a "NOTICE" text file as part of its\n          distribution, then any Derivative Works that You distribute must\n          include a readable copy of the attribution notices contained\n          within such NOTICE file, excluding those notices that do not\n          pertain to any part of the Derivative Works, in at least one\n          of the following places: within a NOTICE text file distributed\n          as part of the Derivative Works; within the Source form or\n          documentation, if provided along with the Derivative Works; or,\n          within a display generated by the Derivative Works, if and\n          wherever such third-party notices normally appear. The contents\n          of the NOTICE file are for informational purposes only and\n          do not modify the License. You may add Your own attribution\n          notices within Derivative Works that You distribute, alongside\n          or as an addendum to the NOTICE text from the Work, provided\n          that such additional attribution notices cannot be construed\n          as modifying the License.\n\n      You may add Your own copyright statement to Your modifications and\n      may provide additional or different license terms and conditions\n      for use, reproduction, or distribution of Your modifications, or\n      for any such Derivative Works as a whole, provided Your use,\n      reproduction, and distribution of the Work otherwise complies with\n      the conditions stated in this License.\n\n   5. Submission of Contributions. Unless You explicitly state otherwise,\n      any Contribution intentionally submitted for inclusion in the Work\n      by You to the Licensor shall be under the terms and conditions of\n      this License, without any additional terms or conditions.\n      Notwithstanding the above, nothing herein shall supersede or modify\n      the terms of any separate license agreement you may have executed\n      with Licensor regarding such Contributions.\n\n   6. Trademarks. This License does not grant permission to use the trade\n      names, trademarks, service marks, or product names of the Licensor,\n      except as required for reasonable and customary use in describing the\n      origin of the Work and reproducing the content of the NOTICE file.\n\n   7. Disclaimer of Warranty. Unless required by applicable law or\n      agreed to in writing, Licensor provides the Work (and each\n      Contributor provides its Contributions) on an "AS IS" BASIS,\n      WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or\n      implied, including, without limitation, any warranties or conditions\n      of TITLE, NON-INFRINGEMENT, MERCHANTABILITY, or FITNESS FOR A\n      PARTICULAR PURPOSE. You are solely responsible for determining the\n      appropriateness of using or redistributing the Work and assume any\n      risks associated with Your exercise of permissions under this License.\n\n   8. Limitation of Liability. In no event and under no legal theory,\n      whether in tort (including negligence), contract, or otherwise,\n      unless required by applicable law (such as deliberate and grossly\n      negligent acts) or agreed to in writing, shall any Contributor be\n      liable to You for damages, including any direct, indirect, special,\n      incidental, or consequential damages of any character arising as a\n      result of this License or out of the use or inability to use the\n      Work (including but not limited to damages for loss of goodwill,\n      work stoppage, computer failure or malfunction, or any and all\n      other commercial damages or losses), even if such Contributor\n      has been advised of the possibility of such damages.\n\n   9. Accepting Warranty or Additional Liability. While redistributing\n      the Work or Derivative Works thereof, You may choose to offer,\n      and charge a fee for, acceptance of support, warranty, indemnity,\n      or other liability obligations and/or rights consistent with this\n      License. However, in accepting such obligations, You may act only\n      on Your own behalf and on Your sole responsibility, not on behalf\n      of any other Contributor, and only if You agree to indemnify,\n      defend, and hold each Contributor harmless for any liability\n      incurred by, or claims asserted against, such Contributor by reason\n      of your accepting any such warranty or additional liability.\n\n   END OF TERMS AND CONDITIONS\n\n   APPENDIX: How to apply the Apache License to your work.\n\n      To apply the Apache License to your work, attach the following\n      boilerplate notice, with the fields enclosed by brackets "[]"\n      replaced with your own identifying information. (Don\'t include\n      the brackets!)  The text should be enclosed in the appropriate\n      comment syntax for the file format. We also recommend that a\n      file or class name and description of purpose be included on the\n      same "printed page" as the copyright notice for easier\n      identification within third-party archives.\n\n   Copyright 2026 Kurt Valcorza\n\n   Licensed under the Apache License, Version 2.0 (the "License");\n   you may not use this file except in compliance with the License.\n   You may obtain a copy of the License at\n\n       http://www.apache.org/licenses/LICENSE-2.0\n\n   Unless required by applicable law or agreed to in writing, software\n   distributed under the License is distributed on an "AS IS" BASIS,\n   WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.\n   See the License for the specific language governing permissions and\n   limitations under the License.\n', 'source.json': '{\n  "repository": "kurtvalcorza/ast-audio-classification-pipeline",\n  "revision": "d20a47c1d65e88b5890fe608fe6e0d309e37c087",\n  "generator": "build_notebook.py/3.0",\n  "notebook_spec": "2.2",\n  "files": {\n    "src/ast_audio_classification_pipeline/__init__.py": "1472ec08cfdaaabfb1dc14a14c526d89d52416a927836332300eb234a54ddae2",\n    "src/ast_audio_classification_pipeline/metrics.py": "07c293514f0f5078b9da8b5a26804c7c13d8b36dbb98f0cd1e3be98dd3147e6a",\n    "src/ast_audio_classification_pipeline/samples.py": "23157c35d2682af659be15cdf4f94cfba57b8918e9aafc25a105a266ae7557dd",\n    "src/ast_audio_classification_pipeline/pipeline.py": "35099702eda941543b5f7d66db72cdc2f4663853e9e435df6eddde7141d2c54b",\n    "tutorial_stages.py": "44364114a51f2235366eaee5544f833cedb99df6f06c2d34ce8e0d75225cc2f2",\n    "requirements.txt": "8d7e3bd82486e3b7569f86d2ac4d9dd1550060b67c55e58ccf74b33b35cdf468",\n    "weights/ast-audioset/dimer-base-manifest.json": "13c8248aa95f5c6cee3eccb512248e96797d31c2a54d0cf477cbb0bbf49b91a9",\n    "LICENSE": "f9da6567cb5a1828ccec41c7b72b70997fe6d9d63811b3f3177a7b01da52d138"\n  },\n  "sources": {\n    "src/ast_audio_classification_pipeline/__init__.py": "src/ast_audio_classification_pipeline/__init__.py",\n    "src/ast_audio_classification_pipeline/metrics.py": "src/ast_audio_classification_pipeline/metrics.py",\n    "src/ast_audio_classification_pipeline/samples.py": "src/ast_audio_classification_pipeline/samples.py",\n    "src/ast_audio_classification_pipeline/pipeline.py": "src/ast_audio_classification_pipeline/pipeline.py",\n    "tutorial_stages.py": "tools/tutorial_stages.py",\n    "requirements.txt": "tutorials/requirements-colab.lock.txt",\n    "weights/ast-audioset/dimer-base-manifest.json": "weights/ast-audioset/dimer-base-manifest.json",\n    "LICENSE": "LICENSE"\n  }\n}\n'}
CARRIED_HASHES = {'src/ast_audio_classification_pipeline/__init__.py': '1472ec08cfdaaabfb1dc14a14c526d89d52416a927836332300eb234a54ddae2', 'src/ast_audio_classification_pipeline/metrics.py': '07c293514f0f5078b9da8b5a26804c7c13d8b36dbb98f0cd1e3be98dd3147e6a', 'src/ast_audio_classification_pipeline/samples.py': '23157c35d2682af659be15cdf4f94cfba57b8918e9aafc25a105a266ae7557dd', 'src/ast_audio_classification_pipeline/pipeline.py': '35099702eda941543b5f7d66db72cdc2f4663853e9e435df6eddde7141d2c54b', 'tutorial_stages.py': '44364114a51f2235366eaee5544f833cedb99df6f06c2d34ce8e0d75225cc2f2', 'requirements.txt': '8d7e3bd82486e3b7569f86d2ac4d9dd1550060b67c55e58ccf74b33b35cdf468', 'weights/ast-audioset/dimer-base-manifest.json': '13c8248aa95f5c6cee3eccb512248e96797d31c2a54d0cf477cbb0bbf49b91a9', 'LICENSE': 'f9da6567cb5a1828ccec41c7b72b70997fe6d9d63811b3f3177a7b01da52d138', 'source.json': '2cdc03352422c670ccbe77ec539b70e206fd0ea814f2c05602aedfa30e1715eb'}
for name, text in CARRIED_FILES.items():
    path = ROOT / name
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text, encoding='utf-8', newline='\n')
    if hashlib.sha256(path.read_bytes()).hexdigest() != CARRIED_HASHES[name]:
        raise RuntimeError('Carried file integrity failure: ' + name + '. Do not edit this cell; regenerate the notebook from the repository.')
NOTEBOOK_SOURCE = json.loads((ROOT / 'source.json').read_text(encoding='utf-8'))
print({'carried_files': len(CARRIED_FILES), 'verified': True, 'repository': NOTEBOOK_SOURCE['repository'], 'revision': NOTEBOOK_SOURCE['revision'], 'generator': NOTEBOOK_SOURCE['generator']})

{'carried_files': 9, 'verified': True, 'repository': 'kurtvalcorza/ast-audio-classification-pipeline', 'revision': 'd20a47c1d65e88b5890fe608fe6e0d309e37c087', 'generator': 'build_notebook.py/3.0'}


**Expected result:** `carried_files`, `verified: True`, and the repository revision the notebook was generated from.

The next cell installs nothing into this notebook's kernel. It downloads one pinned file — the `uv` installer wheel, refused unless its size and SHA-256 match — creates a separate virtual environment with its own CPython 3.12.12, and installs `requirements.txt` into it with `--require-hashes --only-binary :all:`: every package must be the locked version, a prebuilt wheel, and match a locked digest. The hosted runtime's own packages are never replaced, which is why no restart is needed. The cell also defines `run_stage`, `load_record` and `obtain_upload`, the helpers the learner cells use.

**Infrastructure: the isolated environment.** Installation messages from `uv` are normal and can take a few minutes. A failed download or a hash mismatch stops the cell; never remove a pin or a hash to get past one.

In [ ]:
# @title Infrastructure: install the locked runtime into an isolated environment and define the stage runner
import io
import urllib.error
import urllib.request
import zipfile

UV_URL = 'https://files.pythonhosted.org/packages/1e/fd/432451d732917c49152a291de3ef171aa6b0f1a22d39780fb2c1f085ca4c/uv-0.12.15-py3-none-manylinux_2_17_x86_64.manylinux2014_x86_64.whl'
UV_BYTES = 20081404
UV_SHA256 = 'aee9802f46bae436bd91751bb33ddeb379ef1596b5c19df193219d545d244b60'
for attempt in range(3):
    try:
        with urllib.request.urlopen(UV_URL, timeout=90) as response:
            wheel = response.read(UV_BYTES + 1)
        break
    except (urllib.error.URLError, TimeoutError, ConnectionError):
        if attempt == 2:
            raise
        time.sleep(2 ** attempt)
if len(wheel) != UV_BYTES or hashlib.sha256(wheel).hexdigest() != UV_SHA256:
    raise RuntimeError('uv 0.12.15 wheel size/hash mismatch: refusing to run it')
ENV_ROOT.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(io.BytesIO(wheel)) as archive:
    member = next(n for n in archive.namelist() if n.endswith('.data/scripts/uv'))
    UV = ENV_ROOT / 'uv'
    UV.write_bytes(archive.read(member))
UV.chmod(0o700)
# The stage processes get no Hugging Face token (every download is public) and no kernel Python path. The kernel may
# export an inline matplotlib backend that the isolated environment cannot import; stages write figures to files.
ENV = dict(os.environ, HF_HUB_DISABLE_IMPLICIT_TOKEN='1', HF_HUB_DISABLE_TELEMETRY='1', DO_NOT_TRACK='1', UV_CACHE_DIR=str(ENV_ROOT / 'cache'), MPLBACKEND='Agg')
for name in ('HF_TOKEN', 'HUGGING_FACE_HUB_TOKEN', 'PYTHONPATH', 'PYTHONHOME'):
    ENV.pop(name, None)
subprocess.run([str(UV), 'venv', '--managed-python', '--python', '3.12.12', str(ENV_ROOT / 'venv')], env=ENV, check=True)
PYTHON = ENV_ROOT / 'venv' / 'bin' / 'python'
subprocess.run([str(UV), 'pip', 'install', '--python', str(PYTHON), '--require-hashes', '--only-binary', ':all:', '--index-url', 'https://pypi.org/simple', '-r', str(ROOT / 'requirements.txt')], env=ENV, check=True)
probe = subprocess.run([str(PYTHON), '-c', "import json, platform, torch, torchaudio, transformers; print(json.dumps({'python': platform.python_version(), 'torch': torch.__version__, 'torchaudio': torchaudio.__version__, 'transformers': transformers.__version__, 'cuda': torch.cuda.is_available()}))"], env=ENV, check=True, capture_output=True, text=True)
RUNTIME = json.loads(probe.stdout.strip().splitlines()[-1])
print({'notebook_source': NOTEBOOK_SOURCE['revision'], **RUNTIME, 'locked_packages': 47, 'environment': str(ENV_ROOT / 'venv'), 'setup_seconds': round(time.perf_counter() - SESSION_START)})


def run_stage(stage, *options):
    """Run one stage of the carried runner in its own process with the isolated interpreter; stream its output."""
    log = ROOT / 'logs' / (stage + '.log')
    log.parent.mkdir(exist_ok=True)
    error_file = ROOT / 'state' / (stage + '.error.json')
    error_file.unlink(missing_ok=True)
    command = [str(PYTHON), '-u', str(ROOT / 'tutorial_stages.py'), '--root', str(ROOT), '--weights', str(WEIGHTS), '--stage', stage, *map(str, options)]
    print('Running stage', repr(stage), 'in the isolated environment; log:', log, flush=True)
    with log.open('w', encoding='utf-8') as output:
        process = subprocess.Popen(command, env=ENV, cwd=str(ROOT), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for line in process.stdout:
            output.write(line)
            output.flush()
            if line.strip() and len(line) < 4000:
                print(line.rstrip(), flush=True)
        process.wait()
    if process.returncode:
        detail = 'see the log above'
        if error_file.is_file():
            error = json.loads(error_file.read_text(encoding='utf-8'))
            detail = error['type'] + ': ' + error['message']
        raise RuntimeError(f'Stage {stage!r} failed (exit {process.returncode}): {detail}')


def load_record(name):
    """A JSON record a stage wrote to ROOT/outputs."""
    return json.loads((ROOT / 'outputs' / name).read_text(encoding='utf-8'))


def obtain_upload(path_text, suffix, field):
    """A BYOD file: the path in a form field, or (field empty, Google Colab only) one file from the upload dialog."""
    if path_text.strip():
        path = Path(path_text).expanduser()
        if not path.is_file():
            raise FileNotFoundError(f'{field} = {path_text!r} is not a file on this machine; fix the path or leave it empty to upload in Colab')
        return path.resolve()
    try:
        from google.colab import files
    except ImportError as exc:
        raise RuntimeError(f'The upload dialog needs Google Colab. Elsewhere, put the file on this machine and set {field} to its path.') from exc
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError(f'upload exactly one {suffix} file')
    name, payload = next(iter(uploaded.items()))
    if not name.lower().endswith(suffix):
        raise ValueError(f'the uploaded file must end in {suffix}, got {name!r}')
    path = ROOT / 'byod' / Path(name).name
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_bytes(payload)
    return path

{'notebook_source': 'd20a47c1d65e88b5890fe608fe6e0d309e37c087', 'python': '3.12.12', 'torch': '2.14.0+cu130', 'torchaudio': '2.11.0+cu130', 'transformers': '4.57.6', 'cuda': True, 'locked_packages': 47, 'environment': '/tmp/ast_audio_classification_env_322b15d2d44d/venv', 'setup_seconds': 67}


**Expected result:** one dictionary with the generating revision, the isolated environment's Python (3.12.12), the `torch`, `torchaudio` and `transformers` versions, `cuda` (`True` on a GPU runtime, `False` on CPU — both are supported), the number of locked packages and the setup time.

## 3. Pin, stage and verify the model · [Engineering]

> **Infrastructure.** The next code cell is collapsed. It downloads about 346 MB of pinned weights and checks every file's size and SHA-256; you may run it without studying its implementation.

The model identity is carried twice — `MODEL_ID`/`MODEL_REVISION` in the carried `pipeline.py` and the snapshot manifest (paths, byte sizes, SHA-256) — and the `weights` stage first checks that they agree. It installs the carried manifest into `weights/`, fetches exactly the files that are absent from the Hugging Face Hub **at the pinned revision** of `MIT/ast-finetuned-audioset-10-10-0.4593` (never `main`), and re-hashes every file, raising on the first size or digest mismatch. There is no fallback to a different download, and no remote model code is executed. Every later stage verifies the snapshot again before loading it.

In [ ]:
# @title Infrastructure: stage and digest-verify the pinned snapshot
run_stage('weights')

Running stage 'weights' in the isolated environment; log: /content/outputs/ast_audio_classification/322b15d2d44d/logs/weights.log


{'model_id': 'MIT/ast-finetuned-audioset-10-10-0.4593', 'revision': 'f826b80d28226b62986cc218e5cec390b1096902', 'license': 'bsd-3-clause', 'files': 4, 'total_bytes': 346433173}


{'weights_dir': '/content/weights/ast-audioset', 'fetched': ['README.md', 'config.json', 'model.safetensors', 'preprocessor_config.json'], 'verified_files': 4, 'revision': 'f826b80d28226b62986cc218e5cec390b1096902'}


{'stage': 'weights', 'status': 'ok', 'seconds': 5.8}


**What to notice:** the model id, revision, licence and file count; a `fetched` list of the files downloaded on this run (empty on a rerun, because staging only fetches files that are absent); and a count of verified files. A size or SHA-256 mismatch stops the cell with an error naming the file — see **Troubleshooting**, and never edit a manifest to get past one.

## 4. Confirm the isolated runtime · [Engineering]

From here on, every code cell runs one stage of the carried runner with `run_stage` (or reads a record a stage wrote). This cell runs the `runtime` stage. It compares the `torch`, `torchaudio` and `transformers` versions installed in the isolated environment with the versions in the carried lock and **stops if any differs**; it then prints the device the stages will use and the input ceilings the pipeline enforces.

**Expected result:** `versions_match_lock: True`, the device (`cuda:0` on a GPU runtime, `cpu` otherwise) and the ceilings: 16 kHz input, at least 0.025 s, a 10.24 s model window, at most 120 s of input, and 527 AudioSet labels.

In [ ]:
run_stage('runtime')

Running stage 'runtime' in the isolated environment; log: /content/outputs/ast_audio_classification/322b15d2d44d/logs/runtime.log


{'python': '3.12.12', 'torch': '2.14.0+cu130', 'torchaudio': '2.11.0+cu130', 'transformers': '4.57.6', 'versions_match_lock': True, 'device': 'cuda:0', 'ceilings': {'SAMPLE_RATE': 16000, 'MIN_AUDIO_SECONDS': 0.025, 'MAX_AUDIO_SECONDS': 10.24, 'MAX_INPUT_SECONDS': 120.0, 'NUM_LABELS': 527}}


{'stage': 'runtime', 'status': 'ok', 'seconds': 3.2}


## 5. The pretrained AudioSet head · [Concept]

Before adapting anything, the `pretrained` stage runs the unchanged model on a 3-second 440 Hz sine tone (or, with `USE_BYOD = True`, on your own PCM WAV). `predict` turns the clip into a spectrogram, runs the backbone and the original 527-class AudioSet head, and applies an independent **sigmoid** to every label. Each score is **not a calibrated probability**, the 527 scores do not sum to one, and no decision threshold is shipped, so the list is a ranking, not a verdict.

**BYOD:** set `USE_BYOD = True` and either put a WAV path in `BYOD_WAV_PATH` (any runtime) or leave it empty to get the Colab upload dialog. A rate other than 16 kHz is resampled; clips longer than 10.24 s are cropped to the model window and reported as `truncated`.

**Predict before running:** a pure 440 Hz tone is the note A. Which kinds of AudioSet labels do you expect near the top — speech, music, environmental sounds? Write one guess down.

In [ ]:
USE_BYOD = False  # @param {type:"boolean"}
BYOD_WAV_PATH = ''  # @param {type:"string"}

pretrained_options = []
if USE_BYOD:
    pretrained_options = ['--wav', obtain_upload(BYOD_WAV_PATH, '.wav', 'BYOD_WAV_PATH')]
run_stage('pretrained', *pretrained_options)

Running stage 'pretrained' in the isolated environment; log: /content/outputs/ast_audio_classification/322b15d2d44d/logs/pretrained.log


{'sample_kind': 'synthetic', 'name': 'synthetic_sine_440hz_3s.wav', 'samples': 48000, 'sample_rate': 16000, 'sha256': 'be7c8d33e0606949...', 'activation': 'sigmoid', 'truncated': False}


   1. index 501  score 0.8288  Sine wave


   2. index 393  score 0.0290  Dial tone


   3. index 394  score 0.0204  Busy signal


   4. index 481  score 0.0172  Beep, bleep


   5. index 503  score 0.0171  Chirp tone


{'stage': 'pretrained', 'status': 'ok', 'seconds': 9.2}


**What to notice:** `sample_kind`, the clip's sample count and digest, `activation: sigmoid`, and five ranked labels with their scores. Several unrelated labels can all have moderate scores at once, because each label is scored on its own.

**Checkpoint:** the top label for the tone has a score of, say, 0.4. Does that mean the model is 40% sure? Could two labels both score above 0.5?

<details>
<summary>Check your reasoning (open after answering)</summary>

No. A sigmoid score is a per-label output of the training objective, not a calibrated probability: nothing here measured how often a score of 0.4 is right. And yes — labels are scored independently, so several can be high at once (AudioSet clips often contain several sounds), and the scores do not sum to one. A tone usually draws musical or tonal labels (sine wave, music, tuning fork and similar), but the list is a ranking for inspection; this notebook makes no claim that the base model recognises any particular sound.

</details>

## 6. Build and validate the acoustic-ecology dataset · [Evaluation practice]

The adaptation task is the Krause/Pijanowski acoustic ecology split of a soundscape into three sources: **geophony** (non-biological natural sound — rain, wind, water), **biophony** (animal sound — bird song, insects) and **anthrophony** (human-made sound — engines, machinery). The `dataset` stage builds 24 balanced 3-second 16 kHz clips with the package's generator (`synthetic_audio_dataset`): filtered noise for geophony, frequency-modulated chirps for biophony and a harmonic drone for anthrophony. The generator alone repeats itself — anthrophony would be one waveform eight times — so the stage gives **every clip its own seeded variation** (`vary_waveform`): a speed change that shifts every frequency by up to ±8%, a time shift, a gain and a quiet noise floor. It then validates every record (`validate_dataset`: sample rate, mono float32, finite samples in [-1, 1], duration, labels), and **refuses any waveform that occurs twice**, naming the copies, because a copy could otherwise end up both in training and in evaluation. The records follow the owner-namespaced `io.github.kurtvalcorza.dataset.audio.waveform-classification.v1` representation.

**BYOD:** set `USE_BYOD_DATASET = True` and put a ZIP path in `BYOD_ZIP_PATH`, or leave the path empty for the Colab upload dialog. The ZIP must hold the three class folders, at the top level or inside one enclosing folder (the folder name is stripped and reported); `__MACOSX/` and dot-files are skipped and listed. Every refusal names the file and what to do: resample to 16 kHz, trim to 10.24 s, remove a duplicate, add clips to a class.

**Expected result:** `records: 24`, `distinct_waveforms: 24`, eight clips per class, `verdict: accepted`, and a refusal probe showing that a 121-second input is rejected before any model runs. The input manifest is written to `outputs/ast_audio_classification_input_manifest.json` in the run directory.

In [ ]:
USE_BYOD_DATASET = False  # @param {type:"boolean"}
BYOD_ZIP_PATH = ''  # @param {type:"string"}

dataset_options = []
if USE_BYOD_DATASET:
    dataset_options = ['--zip', obtain_upload(BYOD_ZIP_PATH, '.zip', 'BYOD_ZIP_PATH')]
run_stage('dataset', *dataset_options)

Running stage 'dataset' in the isolated environment; log: /content/outputs/ast_audio_classification/322b15d2d44d/logs/dataset.log


{'dataset_kind': 'synthetic', 'records': 24, 'distinct_waveforms': 24, 'class_counts': {'geophony': 8, 'biophony': 8, 'anthrophony': 8}, 'verdict': 'accepted'}


{'refusal_probe': {'input': 'over-long-probe', 'verdict': 'rejected', 'message': 'audio is 121.00 s; ceiling is 120.0 s (chunk it first)'}}


{'stage': 'dataset', 'status': 'ok', 'seconds': 0.7}


**What to notice:** `distinct_waveforms` equals `records`. The generated clips are synthetic stand-ins with clear acoustic signatures, so they test the workflow, not field performance.

**Checkpoint:** why check for identical waveforms *before* splitting, rather than trusting that 24 records with 24 different ids are 24 different clips?

<details>
<summary>Check your reasoning (open after answering)</summary>

Ids are labels on records, not evidence about their content. Before this check was added, the generator produced 24 ids but only 13 distinct waveforms (8 geophony, 4 biophony, 1 anthrophony), and two of the six evaluation clips were byte-identical to training clips. A model can score perfectly on such a clip by remembering it, so part of the held-out result would have measured memorisation. Checking digests before the split guarantees that no clip can be on both sides; the next section confirms it.

</details>

## 7. The held-out evaluation split · [Evaluation practice]

The `dataset` stage also split the records with `split_dataset(..., val_fraction=0.25, seed=42)`: per class, a seeded shuffle puts 25% of the clips into a **held-out evaluation set** and the rest into training — 18 training clips (6 per class) and 6 evaluation clips (2 per class). Stratifying keeps the class balance identical on both sides. The evaluation clips are used only to measure the result; nothing is tuned or selected on them, and no separate validation set exists. A random split like this **assumes the clips are independent** — no two clips cut from the same recording or source. That holds for the generated data; for your own recordings, put clips from one recording on the same side.

**Expected result:** 18 / 6 clips, 6 / 2 per class, and `train_evaluation_duplicates: 0`.

In [ ]:
split = load_record('ast_audio_classification_split.json')
print({key: split[key] for key in ('split', 'train_clips', 'evaluation_clips', 'counts', 'train_evaluation_duplicates')})
print('evaluation clips:', split['eval_ids'])

{'split': 'stratified, seeded (seed 42), 75% train / 25% held-out evaluation', 'train_clips': 18, 'evaluation_clips': 6, 'counts': {'train': {'geophony': 6, 'biophony': 6, 'anthrophony': 6}, 'evaluation': {'geophony': 2, 'biophony': 2, 'anthrophony': 2}}, 'train_evaluation_duplicates': 0}
evaluation clips: ['clip-011', 'clip-017', 'clip-012', 'clip-010', 'clip-009', 'clip-022']


**Checkpoint:** with only two evaluation clips per class, how much does one wrong prediction move the accuracy?

<details>
<summary>Check your reasoning (open after answering)</summary>

One clip out of six is about 0.17 of accuracy, and it halves that class's recall. Six clips are enough to check that the workflow behaves sensibly — a *sample-sanity* check — but far too few to estimate how well the model would do on new recordings, which is why the reports below say `sample-sanity`, not a benchmark result.

</details>

## 8. Re-head, freeze, and two baselines · [Concept]

`pipe.rehead(ADAPT_CLASSES, seed=42)` replaces the 527-class AudioSet head with a new, randomly initialised 3-class head (`Linear(768, 3)`, seed 42), and `pipe.freeze_backbone()` freezes the 85.5 M backbone parameters, so only the head — 3,843 parameters — can change. Before any training, the `baseline` stage scores two trivial references on the evaluation clips: the **untrained head** (random weights) and the **majority baseline**, a rule fitted on the *training* labels that always predicts the training split's most common class. It does not look at the evaluation labels, so it is a predictor you could actually build; with balanced classes it scores one third.

**Predict before running:** roughly what accuracy do you expect from the untrained head on three balanced classes?

In [ ]:
run_stage('baseline')

Running stage 'baseline' in the isolated environment; log: /content/outputs/ast_audio_classification/322b15d2d44d/logs/baseline.log


{'classes': ['geophony', 'biophony', 'anthrophony'], 'frozen_backbone_parameters': 86187264, 'trainable_parameters': 3843}


{'untrained_head_accuracy': 0.0, 'untrained_head_macro_f1': 0.0, 'majority_baseline': "always 'biophony' (fitted on train) -> accuracy 0.3333"}


{'stage': 'baseline', 'status': 'ok', 'seconds': 6.4}


**What to notice:** the frozen and trainable parameter counts, the untrained head's accuracy and macro-F1, and the majority rule's class and accuracy. These are the numbers the fine-tuned head has to beat.

<details>
<summary>Check your reasoning (open after answering)</summary>

Around one third — chance level for three balanced classes — though with six clips it can easily land at 0, 0.17, 0.33 or 0.5. A random head often predicts one class for most clips, so its macro-F1 can be far lower than its accuracy. The majority rule scores exactly 2/6 = 0.3333 here because every class has two evaluation clips.

</details>

## 9. Fine-tune the head and export the adapter · [Concept]

The `finetune` stage re-heads and freezes a fresh model exactly as in Section 8, runs every training and evaluation clip through the frozen backbone **once** and caches the 768-number features, then trains only the head with `torch.optim.AdamW` and cross-entropy for `EPOCHS = 5` epochs at `BATCH_SIZE = 4` and `LEARNING_RATE = 1e-3` (5 optimizer steps per epoch, 25 in total). After each epoch it prints the training loss and the accuracy on the evaluation clips, for information only: nothing is chosen from those numbers. It then records the trained model's scores for later comparison and saves the head as `outputs/ast-audio-adapter-v1.pt`: the trained weights, the class names, the training settings and the exact base-model identity. The 85 M-parameter backbone is not duplicated in this file; it is rebuilt from the pinned base revision when the adapter is loaded.

**Predict before running:** will the training loss fall every epoch? How many epochs will the head need to classify all six evaluation clips correctly?

In [ ]:
EPOCHS = 5
BATCH_SIZE = 4
LEARNING_RATE = 1e-3
run_stage('finetune', '--epochs', EPOCHS, '--batch-size', BATCH_SIZE, '--learning-rate', LEARNING_RATE)

Running stage 'finetune' in the isolated environment; log: /content/outputs/ast_audio_classification/322b15d2d44d/logs/finetune.log


Epoch 1/5: train_loss=0.9809  evaluation_accuracy=1.0000  steps=5


Epoch 2/5: train_loss=0.1480  evaluation_accuracy=1.0000  steps=5


Epoch 3/5: train_loss=0.0334  evaluation_accuracy=1.0000  steps=5


Epoch 4/5: train_loss=0.0119  evaluation_accuracy=1.0000  steps=5


Epoch 5/5: train_loss=0.0059  evaluation_accuracy=1.0000  steps=5


{'adapter': 'ast-audio-adapter-v1.pt', 'bytes': 19103, 'sha256': '135c01e4aec82202...', 'contains': 'classifier head only (the frozen backbone is rebuilt from the pinned base revision)'}


{'stage': 'finetune', 'status': 'ok', 'seconds': 9.1}


**What to notice:** five epoch lines with the training loss, the evaluation accuracy and `steps=5`, then the adapter's size (about 19 KB) and digest. The notebook does not promise a monotonic loss or any quality threshold on this sample.

<details>
<summary>Check your reasoning (open after answering)</summary>

The loss usually falls, but with mini-batches of four it can rise for an epoch. On these clearly separable synthetic classes the head often classifies all evaluation clips correctly within one or two epochs — which is exactly why the evaluation accuracy is printed for information only: if it were used to pick an epoch, it would no longer be a held-out measurement.

</details>

## 10. Held-out evaluation in a fresh process · [Evaluation practice]

The `evaluate` stage starts a new process, rebuilds the base model from the verified snapshot, loads the adapter with `torch.load(..., weights_only=True)`, and scores the six evaluation clips. For each clip the adapted head produces three **softmax scores**: positive numbers that sum to one. The **decision rule is argmax** — the class with the highest score is the prediction; there is no threshold and no "unsure" answer. These scores are **uncalibrated**: six clips cannot show how often a score of 0.9 is right, so do not read them as probabilities. Calibration and any threshold are the responsibility of whoever uses the model, on their own data.

How to read the report:

- **Accuracy** is the share of evaluation clips predicted correctly.
- **Per class**, *precision* is the share of clips predicted as that class that really are it, *recall* the share of that class's clips found, *F1* their harmonic mean, and *support* the number of clips of that class.
- **Macro-F1** averages the three F1 scores with equal weight, so a class the model ignores pulls it down even when accuracy looks good.
- In the **confusion matrix**, rows are the true class and columns the predicted class, in the order printed; the diagonal counts correct predictions and every off-diagonal count is a specific mistake.
- The **baseline** is the training-majority rule from Section 8; `delta_vs_baseline` is the accuracy gain over it.

The machine-readable report is written to `outputs/ast_audio_classification_evaluation_report.json` with verdict `sample-sanity`.

**Predict before running:** will the adapted head beat the majority baseline? On which class, if any, do you expect mistakes?

In [ ]:
run_stage('evaluate')

Running stage 'evaluate' in the isolated environment; log: /content/outputs/ast_audio_classification/322b15d2d44d/logs/evaluate.log


{'evaluation_clips': 6, 'accuracy': 1.0, 'macro_f1': 1.0, 'majority_baseline_accuracy': 0.3333, 'delta_vs_baseline': 0.6667, 'verdict': 'sample-sanity'}


Per class (precision / recall / F1 / support):


  geophony     1.0000 / 1.0000 / 1.0000 / 2


  biophony     1.0000 / 1.0000 / 1.0000 / 2


...

Confusion matrix (rows = true class, columns = predicted class): ['geophony', 'biophony', 'anthrophony']


  geophony     [2, 0, 0]


  biophony     [0, 2, 0]


  anthrophony  [0, 0, 2]


{'stage': 'evaluate', 'status': 'ok', 'seconds': 7.1}


**What to notice:** accuracy and macro-F1 against the baseline's 0.3333, the per-class lines, and where any off-diagonal counts fall in the confusion matrix.

**Question tested:** on clips that played no part in training, does the adapted head separate the three sources better than a rule that ignores the audio?

<details>
<summary>Check your reasoning (open after answering)</summary>

A perfect or near-perfect score is the expected outcome here: the generated classes differ in obvious ways (broadband noise, chirps, a low drone), and the evaluation clips are distinct from the training clips. That shows the adaptation workflow works end to end and that the head learned *these* signatures. It does not show that the model would separate real field recordings, where sources overlap, vary and are recorded on different equipment.

</details>

## 11. An unseen clip · [Evaluation practice]

The `evaluate` stage also scored one newly generated biophony clip (`unseen_synthetic_biophony_99`) that is not in the dataset — its digest is checked against all 24 records. The three uncalibrated softmax scores are exported, rank-ordered, to `outputs/ast_audio_classification_top_k.csv`. The prediction is the argmax; the top score is a ranking margin, not a confidence.

**Expected result:** `in_dataset: False`, the true class `biophony`, and three scores that sum to one.

In [ ]:
unseen = load_record('ast_audio_classification_unseen_prediction.json')
print({'clip': unseen['clip'], 'true_class': unseen['true_class'], 'in_dataset': unseen['in_dataset'], 'decision_rule': 'argmax'})
for rank, item in enumerate(unseen['result']['predictions'], start=1):
    print(f"  {rank}. {item['label']:<12} uncalibrated softmax score {item['score']:.4f}")

{'clip': 'unseen_synthetic_biophony_99', 'true_class': 'biophony', 'in_dataset': False, 'decision_rule': 'argmax'}
  1. biophony     uncalibrated softmax score 0.9718
  2. geophony     uncalibrated softmax score 0.0148
  3. anthrophony  uncalibrated softmax score 0.0134


**Checkpoint:** the top score is close to 1. Would you report "the model is 99% confident this is biophony"?

<details>
<summary>Check your reasoning (open after answering)</summary>

No. Softmax always produces scores that sum to one, and a head trained to separate three easy synthetic classes tends to push one score close to 1 whatever the input — even for a sound that belongs to none of the classes, because argmax over a closed set has no "other" answer. The honest statement is "the adapted head ranks biophony first"; how often such a ranking is right would have to be measured on real, labelled data.

</details>

## 12. Fresh reload and numeric parity · [Engineering]

Section 10 already loaded the adapter in a new process. The `reload` stage checks that this is faithful: it checks the adapter file's digest, rebuilds the pipeline from it in another fresh process, scores every evaluation clip and the unseen clip again, and compares all 21 scores with the ones the trained model produced in Section 9 (`rtol=1e-5`, `atol=1e-6`), together with the predicted labels. A mismatch stops the notebook.

**Expected result:** `reload_verification: PASSED`, `predicted_labels_identical: True` and a maximum absolute score difference at or near 0.

In [ ]:
run_stage('reload')

Running stage 'reload' in the isolated environment; log: /content/outputs/ast_audio_classification/322b15d2d44d/logs/reload.log


{'clips_compared': 7, 'scores_compared': 21, 'max_abs_score_difference': 0.0, 'predicted_labels_identical': True, 'rtol': 1e-05, 'atol': 1e-06, 'labels': ['geophony', 'biophony', 'anthrophony'], 'reload_verification': 'PASSED'}


{'stage': 'reload', 'status': 'ok', 'seconds': 6.5}


**What to notice:** parity shows that the export is complete and loads safely — not that the predictions are right. A saved model reproduces its mistakes just as faithfully as its successes.

## 13. Export the provenance bundle · [Engineering]

The `bundle` stage writes `outputs/ast_audio_classification_result.json`, which links every record of this run — input manifest, split, untrained-head and baseline results, training history, evaluation report, unseen prediction and reload parity — with the notebook's source revision, the model id, revision and licence, the runtime versions, and the SHA-256 of every output file.

**Expected result:** a list of the run's output files, including `ast_audio_classification_input_manifest.json`, `ast_audio_classification_evaluation_report.json`, `ast_audio_classification_result.json`, `ast_audio_classification_top_k.csv` and `ast-audio-adapter-v1.pt`.

In [ ]:
run_stage('bundle')

Running stage 'bundle' in the isolated environment; log: /content/outputs/ast_audio_classification/322b15d2d44d/logs/bundle.log


{'outputs_directory': '/content/outputs/ast_audio_classification/322b15d2d44d/outputs', 'files': ['ast-audio-adapter-v1.pt', 'ast_audio_classification_evaluation_report.json', 'ast_audio_classification_input_manifest.json', 'ast_audio_classification_pre_adaptation.json', 'ast_audio_classification_pretrained.json', 'ast_audio_classification_reload_parity.json', 'ast_audio_classification_result.json', 'ast_audio_classification_split.json', 'ast_audio_classification_top_k.csv', 'ast_audio_classification_training_history.json', 'ast_audio_classification_unseen_prediction.json', 'runtime.json', 'weights.json']}


{'stage': 'bundle', 'status': 'ok', 'seconds': 0.1}


## 14. Optional activity: change one thing — evaluation noise · [Concept]

**Predict → Change one thing → Run → Observe → Explain.** This activity is off by default and changes nothing the canonical path produced: with `RUN_ACTIVITY = False` the next cell only prints how to switch it on. When on, the `activity` stage loads the exported adapter, adds seeded white noise to each of the six evaluation clips at `ACTIVITY_SNR_DB`, and scores clean and noisy clips with the same model. It writes only under `outputs/activity/` and stops if a canonical output changed.

**Change one thing:** the noise level. 0 dB means the noise is as loud as the sound; 20 dB means it is ten times quieter (in amplitude); -10 dB means it is louder. The adapter, the clips and the noise seed stay fixed. Set `RUN_ACTIVITY = True` and run the cell with the default `ACTIVITY_SNR_DB = -10.0`; then change the value to 0 and to 20 and run the cell again for each.

**Predict before running:** at -10 dB, which classes do you expect to be misclassified, and as what? Will 0 dB and 20 dB differ from it? Write it down.

In [ ]:
RUN_ACTIVITY = False  # @param {type:"boolean"}
ACTIVITY_SNR_DB = -10.0  # @param {type:"number"}

if RUN_ACTIVITY:
    run_stage('activity', '--snr-db', ACTIVITY_SNR_DB)
else:
    print({'activity': 'skipped (optional)', 'to_run': 'set RUN_ACTIVITY = True and an ACTIVITY_SNR_DB value, then run this cell'})

{'activity': 'skipped (optional)', 'to_run': 'set RUN_ACTIVITY = True and an ACTIVITY_SNR_DB value, then run this cell'}


**Observe:** clean and noisy accuracy, macro-F1 and confusion matrices, and how many confusion-matrix rows changed.

**Explain:** did the result match your prediction? Which true class moved off the diagonal, and into which predicted class?

**Question tested:** how robust is a head trained on clean synthetic clips when the recording conditions change?

<details>
<summary>Check your reasoning (open after answering)</summary>

White noise is broadband, which is what the geophony clips are made of. Once the noise is louder than the sound, noisy biophony and anthrophony clips are typically pulled to geophony: in the local CPU pre-flight of this notebook, -10 dB moved all four biophony and anthrophony clips to geophony (accuracy 0.33), while 0 dB and 20 dB changed nothing. Your runtime may differ slightly, and six clips at one noise seed are a tiny sample: a result against that direction is an observation about this run, not a failed activity. Because only the noise level changed, any difference is caused by it. The lesson carries over to real data: a head trained and evaluated on clean recordings says little about noisy ones.

</details>

## Troubleshooting · [Engineering]

| Observation | Action |
|---|---|
| Section 1 stops: not Linux x86_64 | Use Google Colab, Kaggle or a Linux Jupyter server; the locked environment is built for manylinux x86_64. |
| Section 1 stops: not enough disk | Start a fresh runtime, or delete earlier `outputs/ast_audio_classification/` run directories. |
| Section 2: download, `uv` or hash failure | Retry once on a stable connection. Never remove a pin or a hash; a hash mismatch means the file is not the locked one. |
| Section 3: Hugging Face download fails or a digest mismatches | Retry; delete `weights/ast-audioset/` and rerun Section 3 if a partial file remains. Never edit the manifest. |
| A stage fails | The cell repeats the stage's error message; the full log is in the run directory under `logs/<stage>.log`. Fix the cause and rerun from that section. |
| "… is missing: run the stage that writes it" | A later cell ran before an earlier one. Run the notebook from the top. |
| `The upload dialog needs Google Colab` | Outside Colab, put the file on the machine and set `BYOD_WAV_PATH` or `BYOD_ZIP_PATH` to its path. |
| BYOD ZIP refused | The message names the file and the fix: resample to 16 kHz, trim to 10.24 s, keep exactly the three class folders, remove duplicate clips, or add a second clip to a class. |
| Reload parity fails | Do not use the export. Rerun from Section 9; if it persists, keep the logs and report it. |

Nothing is installed into the kernel, so no runtime restart is ever needed.

## Interpretation and limits

The adapted classifier produces uncalibrated softmax scores over `ADAPT_CLASSES` (`geophony`, `biophony`, `anthrophony`) and predicts by argmax. The baseline comparison measures performance relative to a trivial majority rule fitted on the training labels, on six held-out synthetic clips. Training is bounded to the classifier head with frozen backbone parameters; it runs on CPU or GPU. The generated clips are distinct and the split has no duplicates, but they remain simple synthetic signatures, and six evaluation clips are a sample-sanity check, not an estimate of field performance.

Successful execution proves that the recorded repository revision's package and stage runner, carried in this standalone notebook, can build a hash-locked environment without touching the kernel, acquire and digest-verify the pinned model, validate the demonstrated dataset contract, execute supervised head adaptation, evaluate it against a training-majority baseline, reload the adapter with numeric parity, and emit the shown machine-readable artifacts — without the repository being reachable. It does **not** establish benchmark superiority, calibrated confidence, production fitness, or performance on real field recordings.

## Conclude with evidence · [Evaluation practice]

Complete this paragraph with the numbers your run printed:

> On [number] held-out synthetic clips that are distinct from the training clips, the adapted head reached accuracy [value] and macro-F1 [value], against [value] for a majority rule fitted on the training labels and [value] for the untrained head. The reloaded adapter reproduced the trained model's scores within [max difference]. With noise at [SNR] dB (optional activity), accuracy was [value]. These results show [what they do show] and do not show [one thing they cannot show].

**Transfer:** switch on dataset BYOD (Section 6) with your own short recordings of the three sources — at least two per class, clips from one recording kept together, nothing confidential on a hosted runtime — and compare your result with the synthetic one. Which assumption from Section 7 is hardest to keep with real recordings?

**AI Assistance Disclosure:** this notebook's code and explanations were developed with generative AI assistance under maintainer direction. The maintainer remains responsible for reviewing implementation, validating results and making release decisions.

## References

- Repository README: https://github.com/kurtvalcorza/ast-audio-classification-pipeline/blob/main/README.md
- Repository model card: https://github.com/kurtvalcorza/ast-audio-classification-pipeline/blob/main/MODEL_CARD.md
- Weight provenance: https://github.com/kurtvalcorza/ast-audio-classification-pipeline/blob/main/docs/WEIGHTS.md
- Upstream model: https://huggingface.co/MIT/ast-finetuned-audioset-10-10-0.4593
- Upstream code: https://github.com/YuanGongND/ast
- AST: Audio Spectrogram Transformer (Gong, Chung, Glass, 2021): https://arxiv.org/abs/2104.01778
- Acoustic ecology (Krause, 2008; Pijanowski et al., 2011)